# BM25 / Ruri / BGE-M3 / Qwen 組み合わせ RAG ベースライン

**目的**: 日本語 Wikipedia のチャンク（約100万件）から根拠を検索し、短答式の質問301問に答える。この notebook は配布済みインデックスを使い、正解を含まない質問ファイルで推論して提出ファイルを作る。採点は行わない。

| 項目 | 内容 |
|---|---|
| 質問数 | 301問 |
| チャンク | 1,003,139件（日本語 Wikipedia 49,043記事を約500文字ごとに分割） |
| 実行環境 | Google Colab（GPU: T4、ランタイムのバージョン 2026.07） |

**回答の形式**: 生成モデルは、回答と、その根拠にしたチャンクの ID を1つだけ出力する。

```json
{"answer": "1803年9月28日", "chunk_id": 500276}
```

**提出形式**: `submission.json` に、`q_id`・`answer`・`chunk` の3項目だけを持つ JSON 配列を保存する。評価は別の運営用スクリプトで行う。

**ベースラインの構成**: Hugging Face のデータセットから取得したインデックスをそのまま使うので、インデックスの作成は不要である。検索は BM25・Ruri・BGE-M3・Qwen から任意の組み合わせを選べる。デフォルトは Ruri 単体・リランカーなし。

| 工程 | 設定 |
|---|---|
| BM25 | SudachiPy（SplitMode C、辞書形）で分かち書きし、Okapi BM25（k1 = 1.2、b = 0.75）で上位100件 |
| 埋め込み検索 | Ruri / BGE-M3 / Qwen のうち選択した各モデルで上位100件 |
| 統合 | 検索を2つ以上選んだ場合だけ、各検索の順位を等重みの RRF（k = 60）で統合して上位100件。1つなら順位をそのまま使う |
| リランク | 使う場合（`USE_RERANKER = True`）は、`RERANKER_NAME` で選んだリランカー（Ruri / BGE / Qwen。デフォルトは Ruri）で統合結果の上位50件を並べ替え、上位10件を使う。 |
| 回答生成 | `Qwen/Qwen3.5-0.8B` を vLLM で実行。サンプリングはモデルカードの推奨値、thinking 無効、最大1024トークン、出力は JSON スキーマで制約 |

**事前に済ませておくこと**:
- Hugging Face のアカウントを作り、データセット [`bumbuboon/llm-course-final-2026`](https://huggingface.co/datasets/bumbuboon/llm-course-final-2026) のページで利用条件に同意する
- Hugging Face の[トークン設定ページ](https://huggingface.co/settings/tokens)で Read 権限のトークンを作り、Colab の左の鍵アイコン（シークレット）に `HF_READ_TOKEN` という名前で追加して「ノートブックからのアクセス」を有効にする。データとインデックスの取得に使う。書き込み権限は不要
- Colab のメニュー「ランタイム」→「ランタイムのタイプを変更」で、ハードウェア アクセラレータに **T4 GPU** を選び、ライブラリの版を固定するためにランタイムのバージョンに **2026.07** を選ぶ。最新版を選ぶと、Colab の更新でライブラリが変わり動かなくなることがある

## データの置き場所

質問は、マイドライブの `LLM講座最終コンペ_work/questions.json`（設定セルの `WORK_DIR` の下）から読み込む。JSON 配列の各要素は `{"q_id": "q001", "question": "質問文"}` の2項目だけで、正解や根拠 ID は含まない。

Drive にこのファイルがない場合だけ、Hugging Face の `data/questions.json` を取得して同じ場所に保存する。ファイルがすでにあればそのまま使い、上書きしない。

チャンクとインデックスは、Hugging Face から Colab のローカルディスク（`/content/rag_data`）に取得する。

実行結果は `LLM講座最終コンペ_work/<RUN_ID>/` に保存する。提出ファイル `submission.json` のほか、検索設定の `retrieval_config.json` と生成設定の `generation_config.json` を置く。

## 利用ルール

### 使用できる深層学習モデル

エンコーダ・リランカー・LLM のいずれも、以下のリストにあるモデルだけを使用できる。

| 用途 | モデルID |
|---|---|
| エンコーダ | `cl-nagoya/ruri-v3-310m` |
| エンコーダ | `BAAI/bge-m3` |
| エンコーダ | `Qwen/Qwen3-Embedding-0.6B` |
| リランカー | `cl-nagoya/ruri-v3-reranker-310m` |
| リランカー | `BAAI/bge-reranker-v2-m3` |
| リランカー | `Qwen/Qwen3-Reranker-0.6B` |
| LLM | `Qwen/Qwen3.5-0.8B` |

### 禁止事項

- 外部 API の使用は一切禁止する。
- 配布チャンクの本文や分割を変更することは禁止する。
- 配布外データ（Web 検索、別のコーパス、出典の Wikipedia 記事など）を参照することは禁止する。
- リストにない深層学習モデルを使用することは禁止する。エンコーダ・リランカー・LLM のすべてが対象である。
- 指定されたもの以外のエンコーダで埋め込みを作成することは禁止する。
  - ただし、指定されたエンコーダを学習した派生モデルは使用でき、それで埋め込みを作成してよい。
- `Qwen/Qwen3.5-0.8B` 以外の LLM で、質問文やチャンクの加工・アノテーションを行うことは禁止する。
- `Qwen/Qwen3.5-0.8B` 以外の LLM による推論・生成は禁止する。
- 深層学習以外の機械学習モデルも、Colab 以外で実行することは禁止する。

### 許可事項

以下は、上記のモデル・データ・実行環境の制約の範囲で許可する。

- 検索パラメータの変更。
- 指定された範囲内でのエンコーダの切り替え。
- ハイブリッド検索（複数の検索方式の組み合わせ）。
- 指定されたエンコーダの学習。
- 指定されたリランカーの学習。
- マルチクエリ。
- エージェンティックサーチ。
- 推論パラメータの変更。
- プロンプトの調整。
- 深層学習以外の機械学習モデルを Colab 上で実行すること。

マルチクエリやエージェンティックサーチで LLM を使う場合も、使用できる LLM は `Qwen/Qwen3.5-0.8B` だけである。

## Step 0: 環境セットアップ

vLLM（LLM 推論）と SudachiPy（分かち書き）を、版を固定して入れる。

- vLLM は PyPI 版が CUDA 13 用で、このランタイムの torch（CUDA 12 系）と組み合わせられないため、GitHub から入れる。
- vLLM が torch を入れ替えることを防ぐため、torch を固定する。
- torchcodec が vLLM の読み込みを妨げるため外す。
- 依存関係の警告（numba、protobuf など）が出るが、Colab に最初から入っている別のパッケージに関するもので、この notebook の処理には影響しない。

In [ ]:

!nvidia-smi

Sat Nov 21 03:50:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# [V0 GPU確認] 一番上に置く。GPU がなければ止める（第7部で CPU のまま動いた事故の対策）
import subprocess
r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
if r.returncode != 0 or "GPU" not in r.stdout:
    raise SystemExit("GPU がつながっていない。ランタイムのタイプで T4 を選び、つなぎ直してから最初からやり直す")
print("V0 OK: GPU あり")

GPU 0: Tesla T4 (UUID: GPU-96770676-00c3-ed51-d3c3-d140cb0fd4b2)
V0 OK: GPU あり


In [2]:
# [step0.1 setup-install]  ←  このセルの名前（以降のセルも1行目に名前を記載）
VLLM_WHEEL = "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
!printf "torch==2.11.0\ntorchvision==0.26.0\n" > /tmp/constraints.txt
!pip install -q "$VLLM_WHEEL" "sudachipy==0.6.11" "sudachidict-core==20260428" -c /tmp/constraints.txt > /tmp/pip_install.log 2>&1
print("exit code:", _exit_code)   # 直前の pip install の終了コード（0 なら成功）
!pip uninstall -y -q torchcodec
!tail -3 /tmp/pip_install.log

exit code: 0
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


**⚠ 上のセルの実行後、「ランタイム」→「セッションを再起動する」を実施し、下の [step1.1 config] 以降を実行する。**

vLLM は、インストール前に読み込まれた torch などのモジュールが残っていると正しく初期化できないため。

## Step 1: BM25 / Ruri / BGE-M3 / Qwen を自由に組み合わせる

設定セルの `RETRIEVERS` に、使いたい検索だけを並べる。空でない15通りの組み合わせに対応する。

```python
RETRIEVERS = ["qwen"]                        # 単体
RETRIEVERS = ["ruri", "bge"]                # 埋め込み同士
RETRIEVERS = ["bm25", "bge", "qwen"]        # 3種類
RETRIEVERS = ["bm25", "ruri", "bge", "qwen"] # 全部
```

各検索の候補数は `BM25_TOP_K`・`RURI_TOP_K`・`BGE_TOP_K`・`QWEN_TOP_K` で個別に指定する。検索が2つ以上なら等重みの RRF で統合し、1つなら順位をそのまま使う。統合後の候補数は `FUSED_TOP_K` である。リランカーを使うかどうかは `USE_RERANKER` で、使うリランカーは `RERANKER_NAME`（`"ruri"`・`"bge"`・`"qwen"`）で選ぶ。デフォルトは `RETRIEVERS = ["ruri"]`、`USE_RERANKER = False`（Ruri 単体・リランカーなし）である。

インデックスは、Hugging Face のデータセットの `index/` の下にある次のフォルダから取得する。BM25 は `bm25-sudachi-c/`、Ruri は `ruri-v3-310m/`、BGE-M3 は `bge-m3/`、Qwen は `qwen3-embedding-0.6b/` である。

質問の埋め込み設定は `EMBED_CONFIGS` で指定する。デフォルト値は配布インデックスの `index_config.json` と同じである。model・revision・pooling・normalize・trust_remote_code は文書ベクトルを作ったときと揃える必要がある。異なる場合は検索精度が下がる。文書ベクトルを別の設定で作り直した場合は、`EMB_INDICES` をそのインデックスに変え、`EMBED_CONFIGS` も合わせる。`query_prefix`・`max_length` は質問側だけの設定なので、変更可能である。

追加学習したモデルなどで独自の埋め込み検索を加える場合は、新しい名前を `RETRIEVERS` に入れ、同じ名前で `EMB_INDICES`・`EMBED_CONFIGS`・`RETRIEVER_TOP_K` に設定を加える。`RETRIEVERS` が空の場合、同じ名前が重複している場合、設定が足りない場合は、設定セルがエラーで止まる。

出力フォルダ名は `RUN_ID` で指定する。条件を変えて結果を残す場合は `RUN_ID` も変える。設定を変えたときは、ランタイムを再起動して設定セルから実行し直す。

In [22]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# # 【キャッシュ作成セル】1回だけ。データとモデルを、Drive に1つずつの tar ファイルとして保存する
# # 前提：このセッションで、検索（bm25・ruri・bge）まで実行して、/content/rag_data と ~/.cache/huggingface がそろっていること
# import os, subprocess, time
# CACHE = "/content/drive/MyDrive/LLM講座最終コンペ_cache"
# os.makedirs(CACHE, exist_ok=True)
# assert os.path.isfile("/content/rag_data/index/chunk_offsets.npy"), "/content/rag_data がそろっていない。先に [step1.3 storage] まで実行する"
# for name, parent, folder, tar in [
#     ("rag_data", "/content", "rag_data", CACHE + "/rag_data.tar"),
#     ("モデル", "/root/.cache", "huggingface", CACHE + "/hf_cache.tar"),
# ]:
#     t0 = time.time()
#     tmp = tar + ".tmp"
#     subprocess.run(["tar", "-cf", tmp, "-C", parent, folder], check=True)
#     os.replace(tmp, tar)
#     print("%s: %.1f GB / %.0f 秒" % (name, os.path.getsize(tar) / 1e9, time.time() - t0))

In [ ]:
# import time
# CACHE = "/content/drive/MyDrive/LLM講座最終コンペ_cache"
# t0 = time.time()
# !rsync -a --info=progress2 "$CACHE/rag_data/" /content/rag_data/
# print("rag_data 復元: %.0f 秒" % (time.time() - t0))

In [2]:
# [step0.9 cache-restore] Drive のキャッシュから、データとモデルを Colab のローカルディスクに戻す
# tar があれば tar から（速い）、なければ元のフォルダから rsync で戻す。復元済みなら飛ばす
#from google.colab import drive
#drive.mount("/content/drive")
import os, subprocess, time
CACHE = "/content/drive/MyDrive/LLM講座最終コンペ_cache"
JOBS = [
    ("rag_data", CACHE + "/rag_data.tar", "/content", CACHE + "/rag_data/", "/content/rag_data/", "/content/.restored_rag_data"),
    ("モデル", CACHE + "/hf_cache.tar", "/root/.cache", CACHE + "/hf_cache/", "/root/.cache/huggingface/", "/content/.restored_hf_cache"),
]
t0 = time.time()
procs = []
for name, tar, tar_dest, folder, folder_dest, mark in JOBS:
    if os.path.exists(mark):
        print(name, ": 復元済みなので飛ばす")
        continue
    if os.path.exists(tar):
        os.makedirs(tar_dest, exist_ok=True)
        cmd, how = ["tar", "-xf", tar, "-C", tar_dest], "tar"
        print(name, ": tar %.1f GB" % (os.path.getsize(tar) / 1e9))
    elif os.path.isdir(folder):
        os.makedirs(folder_dest, exist_ok=True)
        cmd, how = ["rsync", "-a", folder, folder_dest], "rsync"
    else:
        print(name, ": キャッシュがないので飛ばす（[step1.3 storage] などで Hugging Face から取得する）")
        continue
    print(name, ":", how, "で復元を始めた")
    procs.append((name, how, mark, subprocess.Popen(cmd)))
def _stop_all(msg):
    for _, _, _, q in procs:      # 失敗したときは、残りの復元も止める（裏で動き続けないように）
        if q.poll() is None:
            q.terminate()
    raise SystemExit(msg)
for name, how, mark, p in procs:
    if p.wait() != 0:
        _stop_all(name + " の復元（" + how + "）に失敗した")
    if name == "rag_data" and not os.path.isfile("/content/rag_data/index/chunk_offsets.npy"):
        _stop_all("rag_data を戻したのに index/chunk_offsets.npy がない。tar が古いか壊れている。下のキャッシュ作成セルで作り直す")
    open(mark, "w").close()
    print(name, ": 復元した（%s・開始から %.0f 秒）" % (how, time.time() - t0))
print("復元: %.0f 秒" % (time.time() - t0))

rag_data : 復元済みなので飛ばす
モデル : 復元済みなので飛ばす
復元: 0 秒


In [3]:
# [step1.1 config]
# =====================================================================
# 1. 検索の組み合わせと実行名
# =====================================================================
# ここだけ変更すれば単体・任意の組み合わせで実行できる。
RETRIEVERS = ["bm25", "ruri", "bge"]            # 選択肢: "bm25", "ruri", "bge", "qwen"（独自の埋め込み検索を加えた場合はその名前）
# 例: ["qwen"] / ["ruri", "bge"] / ["bm25", "ruri", "bge", "qwen"]
RUN_ID = "val_2stage_E0"

# =====================================================================
# 2. インデックス
# =====================================================================
BM25_INDEX = "bm25-sudachi-c"     # index/ 直下のフォルダ名
EMB_INDICES = {"ruri": "ruri-v3-310m", "bge": "bge-m3", "qwen": "qwen3-embedding-0.6b"}

# =====================================================================
# 3. モデル
# =====================================================================
# 質問の埋め込みに使う設定。デフォルト値は配布インデックスの index_config.json と同じ。
# 注意: model・revision・pooling・normalize・trust_remote_code は、文書ベクトルを作ったときと同じにすること。
#       違ってもエラーにはならず、検索精度だけが大きく下がる。
#       文書ベクトルを別の設定で作り直した場合は、EMB_INDICES をそのインデックスに変え、ここも合わせる。
#       query_prefix・max_length は質問側だけの設定なので、変えて試せる。
# index_config.json と食い違う項目は [step3.2 embedding] で表示する。
EMBED_CONFIGS = {
    "ruri": {
        "model": "cl-nagoya/ruri-v3-310m",
        "revision": "18b60fb8c2b9df296fb4212bb7d23ef94e579cd3",
        "query_prefix": "検索クエリ: ",
        "pooling": "mean",            # "mean" | "cls" | "last"
        "normalize": True,            # True: L2 正規化（内積 = コサイン類似度）
        "max_length": 8192,
        "trust_remote_code": False,
    },
    "bge": {
        "model": "BAAI/bge-m3",
        "revision": "5617a9f61b028005a4858fdac845db406aefb181",
        "query_prefix": "",
        "pooling": "cls",
        "normalize": True,
        "max_length": 8192,
        "trust_remote_code": False,
    },
    "qwen": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
        "query_prefix": "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
        "pooling": "last",
        "normalize": True,
        "max_length": 8192,
        "trust_remote_code": False,
    },
}
EMBED_BATCH_SIZE = 1              # 質問埋め込みのバッチサイズ（T4の長い質問にも配慮）
DENSE_QUERY_BATCH_SIZE = 32       # 行列検索時にまとめて処理する質問数

# リランカーの設定。kind は計算方法: "cross_encoder"（Ruri・BGE）| "qwen3"（Qwen3-Reranker の yes / no 判定）
RERANKER_CONFIGS = {
    "ruri": {
        "model": "cl-nagoya/ruri-v3-reranker-310m",
        "revision": "2c23c61a90583f0348880a2ffb2d7fd29e2d14f1",
        "kind": "cross_encoder",
        "max_length": 512,
        "batch_size": 4,
    },
    "bge": {
        "model": "BAAI/bge-reranker-v2-m3",
        "revision": "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
        "kind": "cross_encoder",
        "max_length": 512,
        "batch_size": 4,
    },
    "qwen": {
        "model": "Qwen/Qwen3-Reranker-0.6B",
        "revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",
        "kind": "qwen3",
        "instruction": "Given a web search query, retrieve relevant passages that answer the query",
        "max_length": 1024,           # 指示文と書式の分だけ長くなるため、Ruri・BGE より長くする
        "batch_size": 4,
    },
}
USE_RERANKER = True              # False: 統合結果の上位をそのまま使う
RERANKER_NAME = "ruri"            # USE_RERANKER = True のときに使うリランカー: "ruri" | "bge" | "qwen"

DEVICE = "auto"                   # 検索側のモデルの置き場所: "auto" | "cuda" | "cpu"

# =====================================================================
# 4. 検索と生成のパラメータ
# =====================================================================
FUSION = "rrf"                    # 複数の検索結果の統合方法（Step 5 の fuse を参照）
BM25_K1 = 1.2
BM25_B = 0.75
BM25_TOP_K = 100                  # BM25 の候補数
RURI_TOP_K = 100                  # Ruri の候補数
BGE_TOP_K = 100                   # BGE-M3 の候補数
QWEN_TOP_K = 100                  # Qwen の候補数
# 検索名ごとの候補数。独自の埋め込み検索を加えた場合は、ここにも同じ名前で候補数を加える
RETRIEVER_TOP_K = {"bm25": BM25_TOP_K, "ruri": RURI_TOP_K, "bge": BGE_TOP_K, "qwen": QWEN_TOP_K}
RRF_K = 60
FUSED_TOP_K = 100                # 統合後に残す件数
RERANK_CANDIDATES = 100          # リランクにかける件数（統合結果の上位から）
CONTEXT_TOP_K = 10                # 生成モデルに渡すチャンク数

# =====================================================================
# 5. データの取得元・結果の保存先と実行範囲
# =====================================================================
HF_REPO = "bumbuboon/llm-course-final-2026"   # データとインデックスを置いた Hugging Face のデータセット
HF_REVISION = "main"              # データセットのブランチ
LOCAL_DATA = "/content/rag_data"  # データとインデックスの取得先（Colab のローカルディスク）
DRIVE_ROOT = "/content/drive/MyDrive"
WORK_DIR = DRIVE_ROOT + "/LLM講座最終コンペ_eval/dev_work"       # 自分の作業フォルダ（Drive）
QUESTION_LIMIT = None             # 例: 10 にすると先頭10問だけ実行する（動作確認用）

# =====================================================================
# ここから下は派生値。編集不要。
# =====================================================================
CHUNKS_DIR = LOCAL_DATA + "/data/chunks"
QUESTIONS_PATH = WORK_DIR + "/questions.json"
INDEX_ROOT = LOCAL_DATA + "/index"   # 配布済みインデックスの取得先
DENSE_NAMES = [name for name in RETRIEVERS if name != "bm25"]
RUN_DIR = WORK_DIR + "/" + RUN_ID
if USE_RERANKER and RERANKER_NAME not in RERANKER_CONFIGS:
    raise ValueError("RERANKER_NAME の %r の設定が RERANKER_CONFIGS にない" % RERANKER_NAME)
RERANKER = RERANKER_CONFIGS[RERANKER_NAME] if USE_RERANKER else None

# RETRIEVERS の確認: 空・重複は禁止。埋め込み検索は、同じ名前の設定が3つの辞書にそろっている必要がある
if not RETRIEVERS:
    raise ValueError("RETRIEVERS が空である。使う検索を1つ以上指定すること")
_duplicates = sorted({name for name in RETRIEVERS if RETRIEVERS.count(name) > 1})
if _duplicates:
    raise ValueError("RETRIEVERS に同じ名前が重複している: %s" % _duplicates)
for name in RETRIEVERS:
    _tables = {"RETRIEVER_TOP_K": RETRIEVER_TOP_K} if name == "bm25" else {
        "EMB_INDICES": EMB_INDICES, "EMBED_CONFIGS": EMBED_CONFIGS, "RETRIEVER_TOP_K": RETRIEVER_TOP_K}
    _missing = [table for table, values in _tables.items() if name not in values]
    if _missing:
        raise ValueError("RETRIEVERS の %r の設定が %s にない。同じ名前で設定を加えること"
                         % (name, "・".join(_missing)))


print("RUN_ID     :", RUN_ID, "->", RUN_DIR)
print("RETRIEVERS :", RETRIEVERS, "/ FUSION:", FUSION if len(RETRIEVERS) > 1 else "なし")
print("TOP_K      :", {name: RETRIEVER_TOP_K[name] for name in RETRIEVERS})
print("BM25_INDEX :", BM25_INDEX if "bm25" in RETRIEVERS else "使わない")
print("EMB_INDICES:", {name: EMB_INDICES[name] for name in DENSE_NAMES})
print("RERANKER   :", "%s（%s）" % (RERANKER_NAME, RERANKER["model"]) if USE_RERANKER else "使わない")

RUN_ID     : val_2stage_E0 -> /content/drive/MyDrive/LLM講座最終コンペ_eval/dev_work/val_2stage_E0
RETRIEVERS : ['bm25', 'ruri', 'bge'] / FUSION: rrf
TOP_K      : {'bm25': 100, 'ruri': 100, 'bge': 100}
BM25_INDEX : bm25-sudachi-c
EMB_INDICES: {'ruri': 'ruri-v3-310m', 'bge': 'bge-m3'}
RERANKER   : ruri（cl-nagoya/ruri-v3-reranker-310m）


In [4]:
# [step1.2 env-check]
# ライブラリの版と GPU を確認する。想定と違う場合は警告を出す（結果は retrieval_config.json に記録する）
import importlib.metadata as _md
import platform
import torch
EXPECTED_VERSIONS = {"torch": "2.11.0", "vllm": "0.26.0"}
ENV = {"python": platform.python_version()}
for _pkg in ("torch", "transformers", "vllm", "sudachipy"):
    try:
        ENV[_pkg] = _md.version(_pkg)
    except _md.PackageNotFoundError:
        ENV[_pkg] = None
ENV["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
for _k, _v in ENV.items():
    print("  %-13s %s" % (_k, _v))
for _pkg, _want in EXPECTED_VERSIONS.items():
    if not (ENV.get(_pkg) or "").startswith(_want):
        print("[警告] %s の版が想定（%s）と違う: %s。ランタイムのバージョン 2026.07 を選び、[step0.1 setup-install] からやり直すこと"
              % (_pkg, _want, ENV.get(_pkg)))
if not ENV["gpu"]:
    print("[警告] GPU が使えない。ランタイムのタイプで T4 GPU を選ぶこと")

  python        3.12.13
  torch         2.11.0+cu128
  transformers  5.13.1
  vllm          0.26.0+cu129
  sudachipy     0.6.11
  gpu           Tesla T4


In [5]:
# # [step1.2 env-check]
# # ライブラリの版と GPU を確認する。想定と違う場合は警告を出す（結果は retrieval_config.json に記録する）
# import importlib.metadata as _md
# import platform
# import torch
# from huggingface_hub import login
# login()   # Hugging Face


# EXPECTED_VERSIONS = {"torch": "2.11.0", "vllm": "0.26.0"}

# ENV = {"python": platform.python_version()}
# for _pkg in ("torch", "transformers", "vllm", "sudachipy"):
#     try:
#         ENV[_pkg] = _md.version(_pkg)
#     except _md.PackageNotFoundError:
#         ENV[_pkg] = None
# ENV["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
# for _k, _v in ENV.items():
#     print("  %-13s %s" % (_k, _v))
# for _pkg, _want in EXPECTED_VERSIONS.items():
#     if not (ENV.get(_pkg) or "").startswith(_want):
#         print("[警告] %s の版が想定（%s）と違う: %s。ランタイムのバージョン 2026.07 を選び、[step0.1 setup-install] からやり直すこと"
#               % (_pkg, _want, ENV.get(_pkg)))
# if not ENV["gpu"]:
#     print("[警告] GPU が使えない。ランタイムのタイプで T4 GPU を選ぶこと")

In [6]:
# [step1.3 storage]
# 1) 配布済みデータと選択した検索インデックスをReadトークンで取得する
# 2) Drive をマウントし、作業フォルダ（実行結果の保存先）を作る
import json, os, time, getpass
from huggingface_hub import snapshot_download, hf_hub_download, HfApi

def write_json(path, value):
    # 一時ファイルに書いてから置き換える（途中で止まっても壊れたファイルを残さない）
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def get_secret(name):
    # ① Colab のシークレット → ② 環境変数 → ③ 手入力（画面に表示しない）の順に探す。トークンは表示も保存もしない
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = os.environ.get(name)
    if not value:
        value = getpass.getpass("%s を入力（Hugging Face の Read トークン。入力は画面に表示されない）: " % name)
    value = (value or "").strip()
    if not value:
        raise RuntimeError("%s が空である。Colab の鍵アイコン（シークレット）に追加するか、入力欄に貼ること" % name)
    if not value.startswith("hf_"):
        print("[注意] %s が hf_ で始まっていない。Read トークンか確認すること" % name)
    return value

HF_READ_TOKEN = get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_READ_TOKEN   # モデルの取得にも使う（トークンなしだと遅く、警告も出る）

# トークンが使えるかを先に確認する（ユーザー名だけ表示。トークンは表示しない）
try:
    print("[storage] Hugging Face:", HfApi().whoami(token=HF_READ_TOKEN).get("name"), "としてログインできた")
except Exception as e:
    raise RuntimeError("Hugging Face のトークンが使えない（%s）。Read 権限か、期限切れでないかを確認すること" % type(e).__name__) from e

# 選択した検索のインデックスのフォルダ名
SELECTED_INDEXES = ([BM25_INDEX] if "bm25" in RETRIEVERS else []) + [EMB_INDICES[name] for name in DENSE_NAMES]

t0 = time.time()
try:
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION, local_dir=LOCAL_DATA,
                      allow_patterns=["data/chunks/chunks-*.jsonl", "data/ATTRIBUTION.jsonl", "data/LICENSE.md", "index/chunk_offsets.npy"] + ["index/%s/*" % folder for folder in SELECTED_INDEXES],
                      token=HF_READ_TOKEN)
except Exception as e:
    if type(e).__name__ in ("GatedRepoError", "RepositoryNotFoundError") or "401" in str(e) or "403" in str(e):
        raise RuntimeError("データセットに入れない（%s）。データセットのページで利用条件に同意したアカウントの Read トークンか確認すること" % type(e).__name__) from e
    raise
print("[storage] 取得した（%.0f s）" % (time.time() - t0))

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("[storage] Colab 以外の環境である。DRIVE_ROOT 以下をそのまま使う")
os.makedirs(WORK_DIR, exist_ok=True)

# 選択したインデックスがそろっていて、作成が完了しているか確認する
INDEX_CONFIGS = {}
for folder in SELECTED_INDEXES:
    cfg_path = os.path.join(INDEX_ROOT, folder, "index_config.json")
    if not os.path.exists(cfg_path):
        raise FileNotFoundError("index/%s/index_config.json がない。BM25_INDEX・EMB_INDICES を確認すること" % folder)
    with open(cfg_path) as f:
        INDEX_CONFIGS[folder] = json.load(f)
    if INDEX_CONFIGS[folder].get("status") != "complete":
        raise RuntimeError("index/%s は作成が完了していない（status=%s）" % (folder, INDEX_CONFIGS[folder].get("status")))
    print("[storage] index/%s: type=%s status=complete" % (folder, INDEX_CONFIGS[folder].get("type")))

[storage] Hugging Face: YAMAHARZ250 としてログインできた


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1236 files:   0%|          | 0/1236 [00:00<?, ?it/s]

[storage] 取得した（2 s）
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[storage] index/bm25-sudachi-c: type=bm25 status=complete
[storage] index/ruri-v3-310m: type=embedding status=complete
[storage] index/bge-m3: type=embedding status=complete


In [7]:
# # [step1.3 storage]
# # 1) 配布済みデータと選択した検索インデックスをReadトークンで取得する
# # 2) Drive をマウントし、作業フォルダ（実行結果の保存先）を作る
# import json, os, time

# #from huggingface_hub import login
# #login()   # Hugging Face

# #os.environ["HF_TOKEN"] = ""   # 取得時の警告を消す
# from huggingface_hub import snapshot_download, hf_hub_download

# def write_json(path, value):
#     # 一時ファイルに書いてから置き換える（途中で止まっても壊れたファイルを残さない）
#     tmp = path + ".tmp"
#     with open(tmp, "w", encoding="utf-8") as f:
#         json.dump(value, f, ensure_ascii=False, indent=2)
#     os.replace(tmp, path)


# def get_secret(name):
#     try:
#         from google.colab import userdata
#         return userdata.get(name)
#     except ImportError:
#         value = os.environ.get(name)   # Colab 以外の環境
#     except Exception:
#         value = None
#     if value is None:
#         raise RuntimeError("Colab のシークレットに %s がない。左の鍵アイコンから追加し、"
#                            "「ノートブックからのアクセス」を有効にすること" % name)
#     return value


# HF_READ_TOKEN = ""
# os.environ["HF_TOKEN"] = ""   # モデルの取得にも使う（トークンなしだと遅く、警告も出る）
# # 選択した検索のインデックスのフォルダ名
# SELECTED_INDEXES = ([BM25_INDEX] if "bm25" in RETRIEVERS else []) + [EMB_INDICES[name] for name in DENSE_NAMES]

# t0 = time.time()
# snapshot_download(repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION, local_dir=LOCAL_DATA,
#                   allow_patterns=["data/chunks/chunks-*.jsonl", "data/ATTRIBUTION.jsonl", "data/LICENSE.md", "index/chunk_offsets.npy"] + ["index/%s/*" % folder for folder in SELECTED_INDEXES],
#                   token=HF_READ_TOKEN)
# print("[storage] 取得した（%.0f s）" % (time.time() - t0))


# try:
#     from google.colab import drive
#     drive.mount("/content/drive", force_remount=False)
# except ImportError:
#     print("[storage] Colab 以外の環境である。DRIVE_ROOT 以下をそのまま使う")
# os.makedirs(WORK_DIR, exist_ok=True)

# # 選択したインデックスがそろっていて、作成が完了しているか確認する
# INDEX_CONFIGS = {}
# for folder in SELECTED_INDEXES:
#     cfg_path = os.path.join(INDEX_ROOT, folder, "index_config.json")
#     if not os.path.exists(cfg_path):
#         raise FileNotFoundError("index/%s/index_config.json がない。BM25_INDEX・EMB_INDICES を確認すること" % folder)
#     with open(cfg_path) as f:
#         INDEX_CONFIGS[folder] = json.load(f)
#     if INDEX_CONFIGS[folder].get("status") != "complete":
#         raise RuntimeError("index/%s は作成が完了していない（status=%s）" % (folder, INDEX_CONFIGS[folder].get("status")))
#     print("[storage] index/%s: type=%s status=complete" % (folder, INDEX_CONFIGS[folder].get("type")))


## Step 2: データの読み込み

質問と、チャンクの位置の表（`chunk_offsets.npy`）を読み込む。`QUESTION_LIMIT` を指定した場合は、先頭の質問だけを実行対象（`TARGET_QUESTIONS`）にする。`get_chunks(chunk_ids)` で、指定した chunk_id のチャンクをローカルディスクのファイルから取り出せる。

In [8]:

# [step2.1 data]  ← 開発用セット専用。本番の質問を Hugging Face から自動取得しないよう、取得処理を外してある
import numpy as np
if not os.path.isfile(QUESTIONS_PATH):
    raise SystemExit("開発用の質問ファイルがない: %s\n開発用の質問（320件・dev0001 形式）を、questions.json の名前でこの場所に置く。"
                     "（このノートブックは、本番の質問を自動取得しない）" % QUESTIONS_PATH)
with open(QUESTIONS_PATH, encoding="utf-8") as f:
    questions = json.load(f)
if not isinstance(questions, list) or len(questions) != 320 or not all(str(q.get("q_id", "")).startswith("dev") for q in questions):
    raise SystemExit("開発用の質問ファイルではない（件数 %s・q_id が dev で始まらない）。本番の質問の可能性があるので止める: %s"
                     % (len(questions) if isinstance(questions, list) else type(questions).__name__, QUESTIONS_PATH))
print("[data] Drive の開発用質問を使う:", QUESTIONS_PATH)
TARGET_QUESTIONS = questions[:QUESTION_LIMIT]   # 実行対象の質問（QUESTION_LIMIT で先頭だけに絞れる）
CHUNK_FILES = sorted(f for f in os.listdir(CHUNKS_DIR) if f.startswith("chunks-") and f.endswith(".jsonl"))
OFFSETS_PATH = os.path.join(INDEX_ROOT, "chunk_offsets.npy")
CHUNK_OFFSETS = np.load(OFFSETS_PATH)
N_CHUNKS = len(CHUNK_OFFSETS)
_handles = {}
def get_chunks(chunk_ids):
    out = []
    for cid in chunk_ids:
        file_no, offset, length = CHUNK_OFFSETS[int(cid)]
        name = CHUNK_FILES[int(file_no)]
        if name not in _handles:
            _handles[name] = open(os.path.join(CHUNKS_DIR, name), "rb")
        fh = _handles[name]
        fh.seek(int(offset))
        out.append(json.loads(fh.read(int(length))))
    return out
print("[data] 質問 %d（実行対象 %d）/ チャンク %d（%d ファイル）"
      % (len(questions), len(TARGET_QUESTIONS), N_CHUNKS, len(CHUNK_FILES)))

[data] Drive の開発用質問を使う: /content/drive/MyDrive/LLM講座最終コンペ_eval/dev_work/questions.json
[data] 質問 320（実行対象 320）/ チャンク 1003139（491 ファイル）


In [9]:
# # [step2.1 data]
# import numpy as np


# _questions_missing = not os.path.isfile(QUESTIONS_PATH)
# _questions_source = QUESTIONS_PATH
# if _questions_missing:
#     _questions_source = hf_hub_download(
#         repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION,
#         filename="data/questions.json", token=HF_READ_TOKEN)
# with open(_questions_source, encoding="utf-8") as f:
#     questions = json.load(f)
# if _questions_missing:
#     write_json(QUESTIONS_PATH, questions)
#     print("[data] Hugging Face から質問を取得して保存した:", QUESTIONS_PATH)
# else:
#     print("[data] Drive の質問ファイルを使う:", QUESTIONS_PATH)

# TARGET_QUESTIONS = questions[:QUESTION_LIMIT]   # 実行対象の質問（QUESTION_LIMIT で先頭だけに絞れる）

# CHUNK_FILES = sorted(f for f in os.listdir(CHUNKS_DIR) if f.startswith("chunks-") and f.endswith(".jsonl"))


# OFFSETS_PATH = os.path.join(INDEX_ROOT, "chunk_offsets.npy")
# CHUNK_OFFSETS = np.load(OFFSETS_PATH)
# N_CHUNKS = len(CHUNK_OFFSETS)
# _handles = {}


# def get_chunks(chunk_ids):
#     out = []
#     for cid in chunk_ids:
#         file_no, offset, length = CHUNK_OFFSETS[int(cid)]
#         name = CHUNK_FILES[int(file_no)]
#         if name not in _handles:
#             _handles[name] = open(os.path.join(CHUNKS_DIR, name), "rb")
#         fh = _handles[name]
#         fh.seek(int(offset))
#         out.append(json.loads(fh.read(int(length))))
#     return out


# print("[data] 質問 %d（実行対象 %d）/ チャンク %d（%d ファイル）"
#       % (len(questions), len(TARGET_QUESTIONS), N_CHUNKS, len(CHUNK_FILES)))


## Step 3: 配布済みインデックスの読み込み

### 3.1 BM25

配布済みの転置インデックスを読み込む。`BM25_K1`・`BM25_B` は、読み込み時に各語の重みへ反映する。`RETRIEVERS` に `bm25` がなければ読み込まない。

In [10]:
# [step3.1 bm25]

_SUDACHI = None


def sudachi_tokenize(text):
    # SplitMode C で分割して辞書形にする。空白と補助記号（句読点など）は除く
    global _SUDACHI
    from sudachipy import Dictionary, Tokenizer
    if _SUDACHI is None:
        _SUDACHI = Dictionary().create()
    return [m.dictionary_form() for m in _SUDACHI.tokenize(text, Tokenizer.SplitMode.C)
            if m.part_of_speech()[0] not in {"空白", "補助記号"}]


class BM25:
    def __init__(self, path, k1=BM25_K1, b=BM25_B):
        z = np.load(os.path.join(path, "bm25_tf.npz"))
        self.indptr, self.indices = z["indptr"], z["indices"]
        tf = z["data"].astype(np.float32)
        doclen = np.load(os.path.join(path, "bm25_doclen.npy")).astype(np.float32)
        with open(os.path.join(path, "bm25_vocab.txt")) as f:
            self.vocab = {t: i for i, t in enumerate(f.read().split("\n")[:-1])}
        n = len(doclen)
        df = np.diff(self.indptr)
        idf = np.maximum(np.log((n - df + 0.5) / (df + 0.5)), 1e-6).astype(np.float32)
        norm = k1 * (1 - b + b * doclen / doclen.mean())
        term = np.repeat(np.arange(len(df), dtype=np.int32), df)   # 各要素がどの語か
        self.weights = idf[term] * tf * (k1 + 1) / (tf + norm[self.indices])
        self.n = n

    def search(self, query, k=BM25_TOP_K):
        # 戻り値: [(chunk_id, スコア), ...] をスコアの高い順に
        terms = {self.vocab[t] for t in sudachi_tokenize(query) if t in self.vocab}
        if not terms:
            return []
        scores = np.zeros(self.n, dtype=np.float32)
        for j in terms:
            s, e = self.indptr[j], self.indptr[j + 1]
            scores[self.indices[s:e]] += self.weights[s:e]
        k = min(k, int((scores > 0).sum()))
        if k == 0:
            return []
        top = np.argpartition(-scores, k - 1)[:k]
        return [(int(d), float(scores[d])) for d in sorted(top.tolist(), key=lambda d: (-scores[d], d))]


bm25 = None
if "bm25" in RETRIEVERS:
    t0 = time.time()
    bm25 = BM25(os.path.join(INDEX_ROOT, BM25_INDEX))
    print("[bm25] %s: %dチャンク / %d語（%.0f s）" % (BM25_INDEX, bm25.n, len(bm25.vocab), time.time() - t0))
else:
    print("[bm25] 使わない（RETRIEVERS に bm25 がない）")

[bm25] bm25-sudachi-c: 1003139チャンク / 1153164語（4 s）


### 3.2 選択した埋め込みモデルの設定を照合する

Ruri は平均、BGE-M3 は CLS、Qwen は最後の有効トークンでプーリングする。
質問の埋め込みには設定セルの `EMBED_CONFIGS` を使い、各インデックスの設定 JSON（`index_config.json`）と項目ごとに照合する。
文書ベクトル作成時と違う項目は `[警告]`、質問側だけの項目（`query_prefix`・`max_length`）の変更は `[変更]` として表示する。

In [11]:

# raise SystemExit("val_2stage では S3-2 を実行しない（v2 の保存ファイルを上書きするため）")

# [step3.2 embedding]

import gc


def pick_device():
    if DEVICE != "auto":
        return DEVICE
    return "cuda" if torch.cuda.is_available() else "cpu"


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class Encoder:
    def __init__(self, spec):
        from transformers import AutoModel, AutoTokenizer
        self.spec, self.prefix, self.device = spec, spec["query_prefix"], pick_device()
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        kw = dict(revision=spec["revision"], trust_remote_code=spec["trust_remote_code"])
        self.tokenizer = AutoTokenizer.from_pretrained(spec["model"], **kw)
        self.tokenizer.padding_side = "right"  # CLS/mean/lastすべてで文書作成時の方式と揃える
        if self.tokenizer.pad_token_id is None:
            if self.tokenizer.eos_token_id is None:
                raise ValueError("padding 用のトークンがない")
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModel.from_pretrained(spec["model"], dtype=dtype, attn_implementation="sdpa", **kw).to(self.device).eval()

    @torch.inference_mode()
    def encode(self, texts, batch_size=EMBED_BATCH_SIZE):
        out = []
        for i in range(0, len(texts), batch_size):
            batch = [self.prefix + t for t in texts[i:i + batch_size]]
            tok = self.tokenizer(batch, padding=True, truncation=True,
                                 max_length=self.spec["max_length"], return_tensors="pt").to(self.device)
            hidden = self.model(**tok).last_hidden_state
            mask = tok["attention_mask"]
            if self.spec["pooling"] == "mean":
                m = mask.unsqueeze(-1).to(hidden.dtype)
                vec = (hidden * m).sum(1) / m.sum(1).clamp_min(1)
            elif self.spec["pooling"] == "cls":
                vec = hidden[:, 0]
            else:   # "last"（Qwen）: 右詰めのpaddingなので、最後の有効トークンは有効トークン数 - 1 の位置
                last = mask.sum(1) - 1
                if (last < 0).any():
                    raise ValueError("有効トークンがない入力がある")
                vec = hidden[torch.arange(len(batch), device=hidden.device), last]
            vec = vec.float()
            if self.spec["normalize"]:
                vec = torch.nn.functional.normalize(vec, p=2, dim=1)
            out.append(vec.cpu().numpy())
        return np.concatenate(out)


QUERY_ONLY_KEYS = {"query_prefix", "max_length"}   # 文書ベクトルと揃えなくてよい項目

# 選択した全モデルについて、EMBED_CONFIGS とインデックス作成時の設定を照合する。実際のモデル読み込みは次の検索セル。
DENSE_INDEXES = {}
for name in DENSE_NAMES:
    folder = os.path.join(INDEX_ROOT, EMB_INDICES[name])
    spec, built = EMBED_CONFIGS[name], INDEX_CONFIGS[EMB_INDICES[name]]["embed"]   # [step1.3 storage] で読み込み済み
    if spec["pooling"] not in ("mean", "cls", "last"):
        raise ValueError('EMBED_CONFIGS["%s"]["pooling"] は "mean" | "cls" | "last" のいずれかにすること: %r'
                         % (name, spec["pooling"]))
    DENSE_INDEXES[name] = {"folder": folder, "embed": spec, "index_embed": built}
    print("[embedding]", name, folder, spec["model"], spec["pooling"])
    for key in spec:
        if spec[key] == built.get(key):
            continue
        if key in QUERY_ONLY_KEYS:
            print("  [変更] %s: 質問側の %s を変えている（インデックス %r / この実行 %r）" % (name, key, built.get(key), spec[key]))
        else:
            print("  [警告] %s: %s が文書ベクトル作成時と違う（インデックス %r / この実行 %r）。検索精度が大きく下がる"
                  % (name, key, built.get(key), spec[key]))


[embedding] ruri /content/rag_data/index/ruri-v3-310m cl-nagoya/ruri-v3-310m mean
[embedding] bge /content/rag_data/index/bge-m3 BAAI/bge-m3 cls


## Step 4: 埋め込み検索とリランカーの準備

実行対象の質問を各モデルで埋め込み、文書ベクトルをシャード単位で検索して上位候補を保持する。
モデル間でベクトルは混ぜず、各モデルの検索順位は Step 5 で定義する `fuse` で統合する。
リランカーを使う設定なら、続くセルでリランカーを読み込む。

In [12]:
# [step4.1 dense-search]
import re


def document_shards(folder):
    # 数字部分で並べるだけ。配列は検索時に各ファイルを1回ずつ読み込む。
    found = []
    for name in os.listdir(folder):
        match = re.fullmatch(r"emb_(\d+)\.npy", name)
        if match:
            found.append((int(match.group(1)), name))
    return [name for _, name in sorted(found)]


def search_dense_index(name, queries):
    # 検索モデルは1つずつ、文書ベクトルは1シャードずつ読み込む。
    if not queries:
        return {}
    entry = DENSE_INDEXES[name]
    folder, spec = entry["folder"], entry["embed"]
    files = document_shards(folder)
    device = pick_device()
    k = min(RETRIEVER_TOP_K[name], N_CHUNKS)
    encoder = q_gpu = block = sim = best_scores = best_ids = None
    try:
        encoder = Encoder(spec)
        q_array = encoder.encode(queries)
        # 質問ベクトルを作った後はモデルを解放し、文書検索のメモリを空ける。
        del encoder
        free_gpu()
        q_gpu = torch.from_numpy(q_array).to(device)
        best_scores = torch.full((len(queries), k), -torch.inf, device=device)
        best_ids = torch.full((len(queries), k), -1, device=device, dtype=torch.long)
        offset = 0
        for file in files:
            arr = np.load(os.path.join(folder, file), allow_pickle=False)
            block = torch.from_numpy(arr).to(device, dtype=torch.float32)
            for start in range(0, len(queries), DENSE_QUERY_BATCH_SIZE):
                stop = min(start + DENSE_QUERY_BATCH_SIZE, len(queries))
                sim = q_gpu[start:stop] @ block.T
                vals, ids = torch.topk(sim, min(k, len(arr)), dim=1)
                ids += offset
                candidates = torch.cat((best_scores[start:stop], vals), dim=1)
                candidate_ids = torch.cat((best_ids[start:stop], ids), dim=1)
                selected, order = torch.topk(candidates, k, dim=1)
                best_scores[start:stop] = selected
                best_ids[start:stop] = candidate_ids.gather(1, order)
                del sim, vals, ids, candidates, candidate_ids, selected, order
            offset += len(arr)
            del arr, block
        ids_list = best_ids.cpu().tolist()
        scores_list = best_scores.cpu().tolist()
        return {query: list(zip(ids_list[i], scores_list[i])) for i, query in enumerate(queries)}
    finally:
        encoder = q_gpu = block = sim = best_scores = best_ids = None
        free_gpu()


# 実行対象の質問だけ検索し、上位k件の結果を保持する。
# 同じ質問文は一度だけ処理する。
_query_texts = list(dict.fromkeys(q["question"] for q in TARGET_QUESTIONS))
DENSE_RESULTS = {}
for name in DENSE_NAMES:
    t0 = time.time()
    DENSE_RESULTS[name] = search_dense_index(name, _query_texts)
    print("[dense]", name, len(_query_texts), "問 / top %d / %.0f s" % (RETRIEVER_TOP_K[name], time.time() - t0))


Loading weights:   0%|          | 0/152 [00:00<?, ?it/s]

[dense] ruri 320 問 / top 100 / 22 s


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[dense] bge 320 問 / top 100 / 13 s


In [13]:
# [step4.2 reranker]
class Reranker:
    # Qwen3-Reranker の公式の書式。質問と文書を渡し、次の語が "yes" になる確率をスコアにする
    QWEN3_PREFIX = ("<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and "
                    'the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
    QWEN3_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

    def __init__(self, spec):
        from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
        self.spec, self.device = spec, pick_device()
        self.fp32 = self.device == "cpu"
        dtype = torch.float32 if self.fp32 else torch.float16
        if spec["kind"] == "qwen3":
            self.tokenizer = AutoTokenizer.from_pretrained(spec["model"], revision=spec["revision"], padding_side="left")
            self.model = AutoModelForCausalLM.from_pretrained(
                spec["model"], revision=spec["revision"], dtype=dtype).to(self.device).eval()
            self.no_yes = [self.tokenizer.convert_tokens_to_ids(t) for t in ("no", "yes")]
            self.prefix = self.tokenizer.encode(self.QWEN3_PREFIX, add_special_tokens=False)
            self.suffix = self.tokenizer.encode(self.QWEN3_SUFFIX, add_special_tokens=False)
        elif spec["kind"] == "cross_encoder":
            self.tokenizer = AutoTokenizer.from_pretrained(spec["model"], revision=spec["revision"])
            self.model = AutoModelForSequenceClassification.from_pretrained(
                spec["model"], revision=spec["revision"], dtype=dtype).to(self.device).eval()
        else:
            raise ValueError("RERANKER の kind が不明である: %r" % spec["kind"])

    def _tokenize(self, query, batch):
        if self.spec["kind"] == "qwen3":
            texts = ["<Instruct>: %s\n<Query>: %s\n<Document>: %s" % (self.spec["instruction"], query, p) for p in batch]
            enc = self.tokenizer(texts, padding=False, truncation="longest_first", return_attention_mask=False,
                                 max_length=self.spec["max_length"] - len(self.prefix) - len(self.suffix))
            enc["input_ids"] = [self.prefix + ids + self.suffix for ids in enc["input_ids"]]
            return self.tokenizer.pad(enc, padding=True, return_tensors="pt").to(self.device)
        return self.tokenizer([query] * len(batch), batch, padding=True, truncation=True,
                              max_length=self.spec["max_length"], return_tensors="pt").to(self.device)

    def _forward(self, tok):
        if self.spec["kind"] == "qwen3":
            # 最後の位置の "no" と "yes" の logits だけを使う（全位置の語彙 logits は計算しない）
            logits = self.model(**tok, logits_to_keep=1).logits[:, -1, self.no_yes].float()
            return logits, torch.softmax(logits, dim=1)[:, 1]
        logits = self.model(**tok).logits.float()
        return logits, logits[:, 0] if logits.shape[-1] == 1 else logits[:, -1]

    @torch.inference_mode()
    def score(self, query, passages):
        scores = []
        for i in range(0, len(passages), self.spec["batch_size"]):
            tok = self._tokenize(query, passages[i:i + self.spec["batch_size"]])
            logits, col = self._forward(tok)
            if not torch.isfinite(logits).all() and not self.fp32:
                # float16 で NaN / inf が出たときだけ float32 で計算し直す
                self.model.float()
                self.fp32 = True
                logits, col = self._forward(tok)
            scores.extend(col.cpu().tolist())
        return scores


reranker = Reranker(RERANKER) if USE_RERANKER else None
print("[reranker]", "%s（%s）" % (RERANKER_NAME, RERANKER["model"]) if USE_RERANKER else "使わない")

Loading weights:   0%|          | 0/156 [00:00<?, ?it/s]

[reranker] ruri（cl-nagoya/ruri-v3-reranker-310m）


## Step 5: 検索・統合・リランクの定義

**検索**（`retrieve`）: 1問ごとに次の順で処理し、生成モデルに渡す資料の dict のリストを返す。

1. **各検索**: `RETRIEVERS` に入っている検索を実行する。どの検索も `[(chunk_id, スコア), ...]` をスコアの高い順に返す
   - `bm25`（`bm25.search`）: 上位 `BM25_TOP_K` 件
   - `ruri` / `bge` / `qwen`: Step 4 で事前に検索した上位 `RURI_TOP_K` / `BGE_TOP_K` / `QWEN_TOP_K` 件を使う
2. **統合**（`fuse`）: 検索が1つだけならその結果をそのまま使う。2つ以上なら `FUSION` の方法でまとめ、上位 `FUSED_TOP_K` 件を残す。ベースラインは RRF（`score = Σ 1 / (RRF_K + 順位)`）
3. **リランク**: リランカーを使う場合は、統合結果の上位 `RERANK_CANDIDATES` 件について質問と `タイトル\n本文` の関連度を計算して並べ替え、上位 `CONTEXT_TOP_K` 件を残す。使わない場合は、統合結果の上位 `CONTEXT_TOP_K` 件をそのまま使う

検索を足すときは、同じ形の戻り値を返す関数を書いて `SEARCHERS` に登録する。統合方法を足すときは、`FUSIONS` に関数を登録して `FUSION` で名前を指定する。各検索のスコアも渡しているので、順位ではなくスコアを使う方法も書ける。

`retrieve()` の戻り値は、最終順位順の `[{"chunk_id": ..., "title": ..., "text": ...}, ...]` である。リランカーの有無にかかわらず同じ形になる。

In [14]:
# [step5.1 pipeline]

SEARCHERS = {}
if "bm25" in RETRIEVERS:
    SEARCHERS["bm25"] = lambda query: bm25.search(query)
for name in DENSE_NAMES:
    SEARCHERS[name] = DENSE_RESULTS[name].__getitem__


def rrf(results, k=RRF_K):
    # results: {検索名: [(chunk_id, スコア), ...]}。順位だけを使う
    scores = {}
    for ranked in results.values():
        for rank, (cid, _) in enumerate(ranked, 1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=lambda cid: (-scores[cid], cid))


# 統合方法の一覧。FUSION で名前を指定する。戻り値は chunk_id のリスト（関連が高い順）
FUSIONS = {
    "rrf": rrf,
}


def fuse(results):
    if len(results) == 1:   # 検索が1つなら統合しない
        ranked = [cid for cid, _ in next(iter(results.values()))]
    else:
        ranked = FUSIONS[FUSION](results)
    return ranked[:FUSED_TOP_K]


def retrieve(query):
    # 検索側の出口は、最終順位順のchunk_id・title・textを持つdictのリスト。
    fused = fuse({name: SEARCHERS[name](query) for name in RETRIEVERS})
    candidate_ids = fused[:RERANK_CANDIDATES] if reranker is not None else fused[:CONTEXT_TOP_K]
    contexts = get_chunks(candidate_ids)
    if reranker is not None and contexts:
        scores = reranker.score(query, [c["title"] + "\n" + c["text"] for c in contexts])
        order = sorted(range(len(contexts)), key=lambda i: (-scores[i], contexts[i]["chunk_id"]))
        contexts = [contexts[i] for i in order[:CONTEXT_TOP_K]]
    return [{"chunk_id": c["chunk_id"], "title": c["title"], "text": c["text"]} for c in contexts]


In [18]:
# 【S3-1 二段検索の定義】[step5.1 pipeline] の直後に置く。retrieve() は書き換えない
import json, os, re, unicodedata, collections
TITLE_INDEX = "/content/drive/MyDrive/LLM講座最終コンペ_eval/title_index/title_to_chunks.json"
A_MINLEN, A_MAX = 2, 3          # 手がかり a: 質問文の中の記事名（文字数の下限・最大の記事数）
B_MINLEN, B_MAX, B_SRC = 3, 5, 5  # 手がかり b: 1段目の上位 B_SRC 件の本文の中の記事名（見直し: 3→5）
PER_TITLE = 60                  # 1記事あたり、先頭から何チャンクまで候補にするか（見直し: 20→60）
KEEP = 10                       # 各段で保存する件数
assert reranker is not None, "リランカーが読み込まれていない（USE_RERANKER = True のまま実行する）"

TITLE2IDS = json.load(open(TITLE_INDEX, encoding="utf-8"))
assert len(TITLE2IDS) == 49043, len(TITLE2IDS)
nf = lambda s: unicodedata.normalize("NFKC", s or "")
_paren = re.compile(r"\s*[\(（][^\(\)（）]*[\)）]\s*$")
KEY2TITLES = collections.defaultdict(list)
for t in TITLE2IDS:                      # 記事名そのもの
    KEY2TITLES[nf(t)].append(t)
n_alias = 0
for t in TITLE2IDS:                      # 括弧を外した名前（同じ名前の記事がなければ）
    base = nf(_paren.sub("", t))
    if base and base != nf(t) and not any(nf(x) == base for x in KEY2TITLES.get(base, [])):
        KEY2TITLES[base].append(t)
        n_alias += 1
KEY_MAXLEN = min(40, max(len(k) for k in KEY2TITLES))
print("記事名のキー: %d（うち括弧を外した名前 %d）/ 照合する最大文字数 %d" % (len(KEY2TITLES), n_alias, KEY_MAXLEN))

def find_keys(text, minlen):
    # text の中に現れる記事名のキー → 出現回数。短いキーが長いキーの一部として現れただけなら数えない
    s = nf(text)
    spans = []
    for i in range(len(s)):
        for L in range(min(KEY_MAXLEN, len(s) - i), minlen - 1, -1):   # 長い方から
            if s[i:i + L] in KEY2TITLES:
                spans.append((i, i + L, s[i:i + L]))
                break
    cnt, cover_end = collections.Counter(), -1
    for a, b, k in spans:                # 先に始まって長いものを優先し、内側の短い一致は捨てる
        if b <= cover_end:
            continue
        cnt[k] += 1
        cover_end = max(cover_end, b)
    return cnt

def _score_ids(query, ids, known):
    # まだスコアのない chunk_id だけリランカーにかける
    new = [c for c in ids if c not in known]
    if new:
        ch = get_chunks(new)
        sc = reranker.score(query, [c["title"] + "\n" + c["text"] for c in ch])
        for c, s in zip(ch, sc):
            known[c["chunk_id"]] = (float(s), c)
    return known

def retrieve_2stage(query):
    # 1段目（retrieve() と同じ処理。スコアも残す）
    fused = fuse({name: SEARCHERS[name](query) for name in RETRIEVERS})
    known = _score_ids(query, fused[:RERANK_CANDIDATES], {})
    order = sorted(known, key=lambda c: (-known[c][0], c))
    s1 = order[:KEEP]
    s1_titles = {known[c][1]["title"] for c in s1}
    # 手がかり a
    ka = sorted(find_keys(query, A_MINLEN), key=lambda k: (-len(k), k))[:A_MAX]
    ta = [(t, len(k)) for k in ka for t in KEY2TITLES[k][:2]]
    # 手がかり b
    cnt = collections.Counter()
    for c in s1[:B_SRC]:
        cnt.update(find_keys(known[c][1]["text"], B_MINLEN))
    a_titles = {t for t, _ in ta}
    kb = [k for k in sorted(cnt, key=lambda k: (-cnt[k], -len(k), k))
          if not any(t in s1_titles or t in a_titles for t in KEY2TITLES[k])][:B_MAX]
    tb = [(t, len(k)) for k in kb for t in KEY2TITLES[k][:1]]
    out = {"s1": [(c, None) for c in s1]}
    for tag, tl in (("s2a", ta), ("s2b", tb)):
        ids, klen = [], {}
        for t, L in tl:
            for c in TITLE2IDS[t][:PER_TITLE]:
                if c not in klen:
                    ids.append(c)
                    klen[c] = L
        _score_ids(query, ids, known)
        out[tag] = [(c, klen[c]) for c in sorted(ids, key=lambda c: (-known[c][0], c))[:KEEP]]
    def pack(lst):
        return [{"chunk_id": c, "title": known[c][1]["title"], "text": known[c][1]["text"],
                 "score": round(known[c][0], 6), **({"keylen": L} if L is not None else {})} for c, L in lst]
    return {k: pack(v) for k, v in out.items()}, len(ka), len(kb)

# 動作確認（1問だけ。中身は表示しない）
_t = time.time()
_r, _na, _nb = retrieve_2stage(TARGET_QUESTIONS[0]["question"])
print("動作確認: s1 %d件 / s2a %d件（記事名 %d）/ s2b %d件（記事名 %d）/ %.1f 秒"
      % (len(_r["s1"]), len(_r["s2a"]), _na, len(_r["s2b"]), _nb, time.time() - _t))

記事名のキー: 53831（うち括弧を外した名前 5403）/ 照合する最大文字数 40
動作確認: s1 10件 / s2a 0件（記事名 0）/ s2b 10件（記事名 5）/ 4.1 秒


In [19]:
# [V1 インタフェース調査] 読むだけ。関数名・引数・データの形だけを表示する（質問文・正解・本文は表示しない）
import inspect, json

def _sig(f):
    try:
        return str(inspect.signature(f))
    except Exception as e:
        return "（取得できない: %s）" % type(e).__name__

KEYS = ("retrieve", "rerank", "stage", "bridge", "title", "clue", "fuse", "rrf", "s1", "s2")
G = dict(globals())

print("== 1. 関数（名前に retrieve/rerank/stage/bridge/title/clue/fuse/rrf/s1/s2 を含むもの）==")
for name, obj in sorted(G.items()):
    if not name.startswith("_") and inspect.isfunction(obj) and any(k in name.lower() for k in KEYS):
        print("  %-30s %s" % (name, _sig(obj)))

print("\n== 2. score メソッドを持つオブジェクト（リランカーの候補）==")
for name, obj in sorted(G.items()):
    if name.startswith("_") or inspect.isclass(obj) or inspect.ismodule(obj):
        continue
    try:
        sc = getattr(obj, "score", None)
    except Exception:
        continue
    if callable(sc):
        print("  %-22s 型 %-22s score%s" % (name, type(obj).__name__, _sig(sc)))

print("\n== 3. 大きな dict（記事名一覧の候補。キーは表示しない）==")
for name, obj in sorted(G.items()):
    if not name.startswith("_") and isinstance(obj, dict) and len(obj) > 40000:
        k = next(iter(obj)); v = obj[k]
        print("  %-22s 件数 %d / キーの型 %s / 値の型 %s / 値の例 %s"
              % (name, len(obj), type(k).__name__, type(v).__name__, str(v)[:60]))

print("\n== 4. 二段検索の保存ファイル（v1）の形（キーだけ）==")
p = "/content/drive/MyDrive/LLM講座最終コンペ_eval/dev_contexts_2stage_v1.jsonl"
with open(p, encoding="utf-8") as f:
    o = json.loads(f.readline())
for k, v in o.items():
    if k == "question":
        print("  %-14s （表示しない）" % k)
    elif isinstance(v, list) and v and isinstance(v[0], dict):
        print("  %-14s list[%d] 要素のキー %s" % (k, len(v), sorted(v[0].keys())))
    else:
        print("  %-14s %s" % (k, type(v).__name__))
print("\nV1 終わり（何も保存していない）")

== 1. 関数（名前に retrieve/rerank/stage/bridge/title/clue/fuse/rrf/s1/s2 を含むもの）==
  fuse                           (results)
  retrieve                       (query)
  retrieve_2stage                (query)
  rrf                            (results, k=60)

== 2. score メソッドを持つオブジェクト（リランカーの候補）==
  reranker               型 Reranker               score(query, passages)

== 3. 大きな dict（記事名一覧の候補。キーは表示しない）==
  KEY2TITLES             件数 53831 / キーの型 str / 値の型 list / 値の例 ['パリ']
  TITLE2IDS              件数 49043 / キーの型 str / 値の型 list / 値の例 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 1

== 4. 二段検索の保存ファイル（v1）の形（キーだけ）==
  q_id           str
  question       （表示しない）
  s1             list[10] 要素のキー ['chunk_id', 'score', 'text', 'title']
  s2a            list
  s2b            list[10] 要素のキー ['chunk_id', 'keylen', 'score', 'text', 'title']

V1 終わり（何も保存していない）


In [20]:
# [E1-1 定義] Q2 採点の二段検索（A_diag）。S3-1 の後に実行する（S3-1 の find_keys・KEY2TITLES・TITLE2IDS・nf を使う）
# 何も保存しない。最後に3問だけ試して、時間と Q2 を作れたかを表示する（質問文・本文は表示しない）
import time, collections
E1_RUN_ID = "V2_phase1a_Q2_K10_Adiag"
E1 = dict(A_MINLEN=2, A_MAX=3, B_MINLEN=3, B_MAX=5, B_SRC=3, PER_TITLE=20, KEEP=10)   # v1 の値（S3-1 の定数は使わない）

for _name in ("find_keys", "KEY2TITLES", "TITLE2IDS", "nf", "fuse", "get_chunks", "SEARCHERS"):
    if _name not in globals():
        raise SystemExit(_name + " がない。S3-1 まで実行してから、このセルを実行する")
if reranker is None:
    raise SystemExit("リランカーがない（USE_RERANKER = True で step4.2 から実行する）")

def _score_new(query, ids, cache):
    # cache: chunk_id → (スコア, 記事名)。まだスコアのないものだけリランカーにかける
    new = [c for c in ids if c not in cache]
    if new:
        ch = get_chunks(new)
        sc = reranker.score(query, [c["title"] + "\n" + c["text"] for c in ch])
        for c, s in zip(ch, sc):
            cache[c["chunk_id"]] = (float(s), c["title"])
    return cache

def retrieve_e1(query):
    # --- 1段目（retrieve() と同じ処理）---
    fused = fuse({name: SEARCHERS[name](query) for name in RETRIEVERS})
    s_orig = _score_new(query, fused[:RERANK_CANDIDATES], {})
    s1 = sorted(s_orig, key=lambda c: (-s_orig[c][0], c))[:E1["KEEP"]]
    s1_titles = {s_orig[c][1] for c in s1}
    # --- 手がかり a（質問の中の記事名）---
    ka = sorted(find_keys(query, E1["A_MINLEN"]), key=lambda k: (-len(k), k))[:E1["A_MAX"]]
    ta = [t for k in ka for t in KEY2TITLES[k][:2]]
    # --- 手がかり b（1段目の上位 B_SRC 件の本文の中の記事名）---
    cnt = collections.Counter()
    for c in get_chunks(s1[:E1["B_SRC"]]):
        cnt.update(find_keys(c["text"], E1["B_MINLEN"]))
    kb = [k for k in sorted(cnt, key=lambda k: (-cnt[k], -len(k), k))
          if not any(t in s1_titles or t in ta for t in KEY2TITLES[k])][:E1["B_MAX"]]
    tb = [KEY2TITLES[k][0] for k in kb]
    # --- 2段目の候補（a・b をまとめる。どちらから来たかを記録）---
    pool, src = [], {}
    for tag, titles in (("a", ta), ("b", tb)):
        for t in titles:
            for c in TITLE2IDS[t][:E1["PER_TITLE"]]:
                if c not in src:
                    src[c] = tag
                    pool.append(c)
    # --- Q2：質問の中の記事名 X（a の1番目）を、たどった記事名 Y（b の1番目）に置き換える ---
    if ka and kb:
        kind, q2 = "Q2", nf(query).replace(ka[0], kb[0], 1)
    else:
        kind, q2 = "ORIG", query                  # 作れなければ元の質問（v1 と同じ）
    _score_new(query, pool, s_orig)               # 元の質問で採点（v1 と同じ採点）
    s_q2 = s_orig if kind == "ORIG" else _score_new(q2, pool, {})   # Q2 で採点（元の質問は渡さない）
    top = lambda sc: sorted(pool, key=lambda c: (-sc[c][0], c))[:E1["KEEP"]]
    pack = lambda ids, sc: [{"chunk_id": c, "title": sc[c][1], "score": round(sc[c][0], 6), "src": src.get(c, "s1")} for c in ids]
    return {"kind": kind, "q2": q2, "x_key": ka[0] if ka else None, "y_key": kb[0] if kb else None,
            "s1": pack(s1, s_orig), "s2_orig": pack(top(s_orig), s_orig), "s2_q2": pack(top(s_q2), s_q2),
            "pool": [[c, src[c]] for c in pool]}

# --- 3問だけ試す（保存しない）---
for q in TARGET_QUESTIONS[:3]:
    _t = time.time()
    _r = retrieve_e1(q["question"])
    print("試し %s: %s / 候補 %d 件（a %d・b %d）/ %.1f 秒" % (
        q["q_id"], _r["kind"], len(_r["pool"]),
        sum(1 for _, s in _r["pool"] if s == "a"), sum(1 for _, s in _r["pool"] if s == "b"), time.time() - _t))
print("E1-1 OK（何も保存していない）")

試し dev0001: ORIG / 候補 91 件（a 0・b 91）/ 3.1 秒
試し dev0002: ORIG / 候補 100 件（a 0・b 100）/ 3.2 秒
試し dev0003: Q2 / 候補 119 件（a 40・b 79）/ 5.0 秒
E1-1 OK（何も保存していない）


In [21]:
# [E1-2 実行・保存] 320問を実行して val_2stage_out/trace_<E1_RUN_ID>.jsonl に保存する（上書きしない）
# 1段目が dev_contexts.jsonl と同じかも確かめる（E0 と同じ確認）
import json, os, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT_DIR = EVAL_ROOT + "/val_2stage_out"
os.makedirs(OUT_DIR, exist_ok=True)
TRACE = OUT_DIR + "/trace_" + E1_RUN_ID + ".jsonl"
if os.path.exists(TRACE):
    raise SystemExit("すでにある（上書き防止）: " + TRACE + "\nE1-3 だけを実行するか、E1_RUN_ID を変える")
assert len(TARGET_QUESTIONS) == 320 and all(q["q_id"].startswith("dev") for q in TARGET_QUESTIONS), "開発用の320問ではない"

base = {}
with open(EVAL_ROOT + "/dev_contexts.jsonl", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            o = json.loads(line)
            base[o["q_id"]] = [int(c["chunk_id"]) for c in o["contexts"]]

t0 = time.time()
rows = []
for i, q in enumerate(TARGET_QUESTIONS, 1):
    rows.append({"q_id": q["q_id"], **retrieve_e1(q["question"])})
    if i % 40 == 0:
        print("  %d / 320（%.1f 分）" % (i, (time.time() - t0) / 60))

tmp = TRACE + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    for o in rows:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")
os.replace(tmp, TRACE)

same = sum(1 for o in rows if [c["chunk_id"] for c in o["s1"]] == base.get(o["q_id"]))
print("保存: %s / %d 問 / %.1f 分" % (TRACE, len(rows), (time.time() - t0) / 60))
print("Q2 を作れた問題: %s" % dict(collections.Counter(o["kind"] for o in rows)))
print("1段目が dev_contexts.jsonl と同じ: %d / 320" % same)
if same != 320:
    raise SystemExit("1段目が既存と違う。E1-3 に進まず、この出力を貼る")
print("E1-2 OK")

  40 / 320（2.5 分）
  80 / 320（5.1 分）
  120 / 320（7.5 分）
  160 / 320（9.9 分）
  200 / 320（12.1 分）
  240 / 320（14.4 分）
  280 / 320（16.8 分）
  320 / 320（19.4 分）
保存: /content/drive/MyDrive/LLM講座最終コンペ_eval/val_2stage_out/trace_V2_phase1a_Q2_K10_Adiag.jsonl / 320 問 / 19.4 分
Q2 を作れた問題: {'ORIG': 157, 'Q2': 163}
1段目が dev_contexts.jsonl と同じ: 320 / 320
E1-2 OK


In [22]:
# [E1-3 評価] 正解の文字列が資料にあるかで数える（質問・正解・本文は表示しない）
# E1-2 の保存ファイルを読むだけ。セッションが切れても、step2.1（get_chunks）と E1-1 の E1_RUN_ID があれば再実行できる
import json, os, re, time, unicodedata, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT_DIR = EVAL_ROOT + "/val_2stage_out"
TRACE = OUT_DIR + "/trace_" + E1_RUN_ID + ".jsonl"
ANS_PATH = EVAL_ROOT + "/dev_data/dev_answers.json"

raw = json.load(open(ANS_PATH, encoding="utf-8"))
recs = {k: (v if isinstance(v, dict) else {"answers": v}) for k, v in raw.items()} if isinstance(raw, dict) else {r["q_id"]: r for r in raw}
first = next(iter(recs.values()))
AKEY = next(k for k in ("answers", "answer", "gold", "gold_answers") if k in first)
DKEY = next((k for k in ("dataset", "source", "origin", "task") if k in first), None)
_norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())
YESNO = {"はい", "いいえ", "yes", "no"}
def golds(q):
    a = recs[q][AKEY]
    return [a] if isinstance(a, str) else list(a)
def is_yn(q):
    g = [_norm(x) for x in golds(q)]
    return bool(g) and all(x in YESNO for x in g)
def ds(q):
    return str(recs[q].get(DKEY, "?")) if DKEY else "?"

rows = [json.loads(l) for l in open(TRACE, encoding="utf-8") if l.strip()]
assert len(rows) == 320 and all(r["q_id"] in recs for r in rows)
NY = [r for r in rows if not is_yn(r["q_id"])]

_text = {}
def first_hit(q, ids):
    # ids の中で、正解の文字列がある最初の順位（なければ None）
    need = [c for c in ids if c not in _text]
    if need:
        for c in get_chunks(need):
            _text[c["chunk_id"]] = _norm(c["title"] + c["text"])
    g = [_norm(x) for x in golds(q) if _norm(x)]
    for i, c in enumerate(ids, 1):
        if any(x in _text[c] for x in g):
            return i
    return None

def slot(s1, s2, K, S=1):
    # 1段目の上位 K-S 件 ＋ 2段目の上位 S 件（1段目と同じ記事・同じチャンクは飛ばす）
    head = s1[:K - S]
    used_t = {c["title"] for c in head}
    used_c = {c["chunk_id"] for c in head}
    pick = [c for c in s2 if c["title"] not in used_t and c["chunk_id"] not in used_c][:S]
    return [c["chunk_id"] for c in head + pick]

ids = lambda lst: [c["chunk_id"] for c in lst]
GROUPS = sorted({ds(r["q_id"]) for r in NY})
st = collections.Counter()
for r in NY:
    q = r["q_id"]
    for g in (ds(q), "計"):
        st[(g, "問題数")] += 1
        st[(g, "Q2 を作れた")] += r["kind"] == "Q2"
        h5, h10 = first_hit(q, ids(r["s1"])[:5]), first_hit(q, ids(r["s1"]))
        hp = first_hit(q, [c for c, _ in r["pool"]])
        st[(g, "1段目5件")] += bool(h5)
        st[(g, "1段目10件")] += bool(h10)
        st[(g, "2段目の候補のどこか")] += bool(hp)
        for tag in ("orig", "q2"):
            rk = first_hit(q, ids(r["s2_" + tag]))
            st[(g, "2段目1位_" + tag)] += rk == 1
            st[(g, "2段目5位以内_" + tag)] += bool(rk and rk <= 5)
            if not h10 and hp:
                st[(g, "だけ_1位_" + tag)] += rk == 1
                st[(g, "だけ_5位以内_" + tag)] += bool(rk and rk <= 5)
            for K in (5, 7, 10):
                st[(g, "枠K%d_" % K + tag)] += bool(first_hit(q, slot(r["s1"], r["s2_" + tag], K)))
        if not h10 and hp:
            st[(g, "だけ_数")] += 1
        for K in (5, 7, 10):
            st[(g, "1段目のみK%d" % K)] += bool(first_hit(q, ids(r["s1"])[:K]))

cols = GROUPS + ["計"]
pct = lambda g, k: 100.0 * st[(g, k)] / max(1, st[(g, "問題数")])
print("非yn %d 問（%s）/ trace: %s\n" % (len(NY), " / ".join("%s %d" % (g, st[(g, "問題数")]) for g in GROUPS), os.path.basename(TRACE)))
print("%-30s" % "項目" + "".join("%22s" % g for g in cols))
def line(label, k):
    print("%-30s" % label + "".join("%22s" % ("%d（%.1f%%）" % (st[(g, k)], pct(g, k))) for g in cols))
def pair(label, k):
    print("%-30s" % label + "".join("%22s" % ("%d → %d（%+d）" % (st[(g, k + "_orig")], st[(g, k + "_q2")], st[(g, k + "_q2")] - st[(g, k + "_orig")])) for g in cols))
line("Q2 を作れた問題", "Q2 を作れた")
line("1段目5件に正解", "1段目5件")
line("1段目10件に正解", "1段目10件")
line("2段目の候補のどこかに正解", "2段目の候補のどこか")
print("--- 2段目（元の質問 → Q2）---")
pair("2段目 recall@1", "2段目1位")
pair("2段目 recall@5", "2段目5位以内")
print("%-30s" % "「2段目にだけある」問題" + "".join("%22d" % st[(g, "だけ_数")] for g in cols))
pair("  そのうち2段目1位", "だけ_1位")
pair("  そのうち2段目5位以内", "だけ_5位以内")
print("--- 枠のシミュレーション（1段目のみ ／ 元の質問 → Q2）---")
for K in (5, 7, 10):
    print("%-30s" % ("K=%d S=1" % K) + "".join("%22s" % ("%d ／ %d → %d" % (st[(g, "1段目のみK%d" % K)], st[(g, "枠K%d_orig" % K)], st[(g, "枠K%d_q2" % K)])) for g in cols))

# --- 判定（設計書 §11：JEMHopQA の 2段目 recall@5 が +10pt 以上、かつ「2段目にだけある」の5位以内が増える）---
JG = next((g for g in GROUPS if "jemhop" in g.lower()), None)
if JG:
    d_pt = pct(JG, "2段目5位以内_q2") - pct(JG, "2段目5位以内_orig")
    d_only = st[(JG, "だけ_5位以内_q2")] - st[(JG, "だけ_5位以内_orig")]
    ok = d_pt >= 10 and d_only > 0
    print("\n判定（%s）: recall@5 %+.1f pt / 「2段目にだけある」5位以内 %+d → %s" % (JG, d_pt, d_only, "合格（E2 へ）" if ok else "未達（出力を貼って次を決める）"))

rep = OUT_DIR + "/E1_report_" + E1_RUN_ID + ".json"
if os.path.exists(rep):
    rep = rep.replace(".json", "_" + time.strftime("%Y%m%d_%H%M%S") + ".json")
json.dump({"run_id": E1_RUN_ID, "time": time.strftime("%Y-%m-%d %H:%M:%S"), "settings": E1,
           "stats": {"%s|%s" % k: v for k, v in st.items()}}, open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("レポート保存:", rep)

非yn 275 問（jemhopqa 75 / niilc 200）/ trace: trace_V2_phase1a_Q2_K10_Adiag.jsonl

項目                                          jemhopqa                 niilc                     計
Q2 を作れた問題                                  49（65.3%）             80（40.0%）            129（46.9%）
1段目5件に正解                                   46（61.3%）            142（71.0%）            188（68.4%）
1段目10件に正解                                  52（69.3%）            148（74.0%）            200（72.7%）
2段目の候補のどこかに正解                              35（46.7%）             83（41.5%）            118（42.9%）
--- 2段目（元の質問 → Q2）---
2段目 recall@1                             18 → 20（+2）           49 → 42（-7）           67 → 62（-5）
2段目 recall@5                             26 → 27（+1）           68 → 59（-9）           94 → 86（-8）
「2段目にだけある」問題                                       2                     4                     6
  そのうち2段目1位                                0 → 0（+0）             3 → 2（-1）             3 → 2（-1）
  そのうち2段目5位以内            

In [24]:
# [D1 取りこぼしの場所] 1段目の上位10件に正解がない非yn 問題で、正解がどこで落ちたかを数える
# E1-3 と同じセッションで実行する（E1-3 の recs・golds・is_yn・ds・first_hit・rows を使う）。読むだけ・保存はレポートだけ
# 生成はしない。リランカーも使わない（統合までをやり直すだけ）。質問・正解・本文は表示しない
import json, os, time, collections
for _name in ("rows", "golds", "is_yn", "ds", "first_hit", "SEARCHERS", "fuse", "RETRIEVERS", "TARGET_QUESTIONS"):
    if _name not in globals():
        raise SystemExit(_name + " がない。E1-3 まで（セッションが切れていれば step5.1 → S3-1 → E1-1 → E1-3）を実行してから、このセルを実行する")

QTEXT = {q["q_id"]: q["question"] for q in TARGET_QUESTIONS}
MISS = [r for r in rows if not is_yn(r["q_id"]) and not first_hit(r["q_id"], [c["chunk_id"] for c in r["s1"]])]
print("1段目の上位10件に正解がない非yn: %d 問（%s）" % (
    len(MISS), dict(collections.Counter(ds(r["q_id"]) for r in MISS))))

def bucket_rank(rk):
    if rk is None: return "なし"
    if rk <= 10: return "1-10"
    if rk <= 30: return "11-30"
    if rk <= 50: return "31-50"
    return "51-100"

t0 = time.time()
st = collections.Counter()
for r in MISS:
    q, g = r["q_id"], ds(r["q_id"])
    res = {name: SEARCHERS[name](QTEXT[q]) for name in RETRIEVERS}
    fused = fuse(res)                                    # 統合の上位 FUSED_TOP_K 件（リランク前）
    rk_f = first_hit(q, fused[:RERANK_CANDIDATES])        # リランカーにかけた範囲での順位
    per = {name: first_hit(q, [c for c, _ in res[name]]) for name in RETRIEVERS}
    any_ret = any(per.values())
    if rk_f:
        cat = "A 統合100件にある → リランカーで10位より下"
    elif any_ret:
        cat = "B どれかの検索100件にある → 統合で落ちた"
    else:
        cat = "C どの検索の100件にもない"
    for gg in (g, "計"):
        st[(gg, cat)] += 1
        st[(gg, "統合順位 " + bucket_rank(rk_f))] += 1
        for name in RETRIEVERS:
            st[(gg, "検索 %s の100件にある" % name)] += bool(per[name])
print("（%.1f 分）\n" % ((time.time() - t0) / 60))

cols = sorted({ds(r["q_id"]) for r in MISS}) + ["計"]
keys = ["A 統合100件にある → リランカーで10位より下", "B どれかの検索100件にある → 統合で落ちた", "C どの検索の100件にもない"]
keys += ["統合順位 " + b for b in ("1-10", "11-30", "31-50", "51-100", "なし")]
keys += ["検索 %s の100件にある" % n for n in RETRIEVERS]
print("%-40s" % "項目" + "".join("%12s" % c for c in cols))
for k in keys:
    print("%-40s" % k + "".join("%12d" % st[(c, k)] for c in cols))

OUT_DIR = "/content/drive/MyDrive/LLM講座最終コンペ_eval/val_2stage_out"
rep = OUT_DIR + "/D1_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump({"n_miss": len(MISS), "stats": {"%s|%s" % k: v for k, v in st.items()}},
          open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nレポート保存:", rep)

1段目の上位10件に正解がない非yn: 75 問（{'jemhopqa': 23, 'niilc': 52}）
（0.1 分）

項目                                          jemhopqa       niilc           計
A 統合100件にある → リランカーで10位より下                         6          14          20
B どれかの検索100件にある → 統合で落ちた                           4           4           8
C どの検索の100件にもない                                   13          34          47
統合順位 1-10                                          1           2           3
統合順位 11-30                                         2           6           8
統合順位 31-50                                         0           1           1
統合順位 51-100                                        3           5           8
統合順位 なし                                           17          38          55
検索 bm25 の100件にある                                   5          11          16
検索 ruri の100件にある                                   6          14          20
検索 bge の100件にある                                    7          10          17

レポート保存: /c

In [31]:
# [D2 reader の取りこぼし（改）] 開発用320問の予測ファイルを中身で探し、上位5件に正解がある問題での正解・不正解を数える
# E1-3 と同じセッションで実行する（rows・golds・is_yn・ds・first_hit・_norm を使う）
# 読むだけ。本番用フォルダ（LLM講座最終コンペ_work）は探さない。質問・正解・回答は表示しない
import json, glob, os, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
PRED = EVAL_ROOT + "/runs/C_L10_final_ctx5_320_20261115_0643/submission.json"
ANS_KEYS = ("answer", "prediction", "pred", "output", "response", "generated")
SKIP = ("dev_data", "val_2stage_out", "dev_contexts")   # 正解ファイル・検索結果・E1 の記録は除く
MAX_MB = 20

for _name in ("rows", "golds", "is_yn", "ds", "first_hit", "_norm"):
    if _name not in globals():
        raise SystemExit(_name + " がない。E1-3 を実行してから、このセルを実行する")

def load_pred(p):
    # 開発用320問の予測なら {q_id: 回答} と使ったキーを返す。違えば None
    try:
        if p.endswith(".jsonl"):
            raw = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
        else:
            raw = json.load(open(p, encoding="utf-8"))
    except Exception:
        return None, None
    if isinstance(raw, dict):
        raw = [dict(v, q_id=k) if isinstance(v, dict) else {"q_id": k, "answer": v} for k, v in raw.items()]
    if not isinstance(raw, list) or not raw or not all(isinstance(o, dict) and "q_id" in o for o in raw):
        return None, None
    key = next((k for k in ANS_KEYS if k in raw[0]), None)
    if key is None:
        return None, None
    pred = {str(o["q_id"]): str(o.get(key) or "") for o in raw}
    if len(pred) != 320 or not all(q.startswith("dev") for q in pred):
        return None, None
    return pred, key

def correct(pred, q):
    # 正規化（NFKC・小文字・空白除去）した完全一致
    p = _norm(pred.get(q, ""))
    return bool(p) and any(p == _norm(x) for x in golds(q))

if not PRED:
    files = [p for ext in ("json", "jsonl") for p in glob.glob(EVAL_ROOT + "/**/*." + ext, recursive=True)]
    files = [p for p in files if not any(s in p for s in SKIP) and os.path.getsize(p) < MAX_MB * 2**20]
    files.sort(key=os.path.getmtime, reverse=True)
    t0, found = time.time(), []
    for p in files:
        pred, key = load_pred(p)
        if pred is None:
            continue
        tot = sum(correct(pred, r["q_id"]) for r in rows)
        ny = sum(correct(pred, r["q_id"]) for r in rows if not is_yn(r["q_id"]))
        found.append((p, key, tot, ny))
    print("開発用320問の予測ファイル: %d 件（調べたファイル %d 件 / %.1f 秒）\n" % (len(found), len(files), time.time() - t0))
    print("%-6s %-6s %-12s %-12s %s" % ("全体", "非yn", "回答のキー", "更新日時", "パス"))
    for p, key, tot, ny in found[:40]:
        print("%-6d %-6d %-12s %-12s %s" % (tot, ny, key, time.strftime("%m-%d %H:%M", time.localtime(os.path.getmtime(p))),
                                           p.replace(EVAL_ROOT + "/", "")))
    raise SystemExit("「全体 127」で lora-final・資料5件の行のパスを PRED に入れて、もう一度実行する（見つからなければ、この出力を貼る）")

pred, key = load_pred(PRED)
if pred is None:
    raise SystemExit("開発用320問の予測ではない（本番用・形の違うファイル）: " + PRED)

st = collections.Counter()
for r in rows:
    q = r["q_id"]
    ok = correct(pred, q)
    for gg in (ds(q), "計"):
        st[(gg, "全体 正解")] += ok
        st[(gg, "全体 問題数")] += 1
        if is_yn(q):
            st[(gg, "yn 正解")] += ok
            st[(gg, "yn 問題数")] += 1
            continue
        ids = [c["chunk_id"] for c in r["s1"]]
        h5 = bool(first_hit(q, ids[:5]))
        h10 = bool(first_hit(q, ids))
        k5 = "上位5件にある" if h5 else ("6〜10位にだけある" if h10 else "上位10件にない")
        st[(gg, k5 + " ・正解" if ok else k5 + " ・不正解")] += 1

cols = sorted({ds(r["q_id"]) for r in rows}) + ["計"]
keys = ["全体 正解", "全体 問題数", "yn 正解", "yn 問題数"]
for k5 in ("上位5件にある", "6〜10位にだけある", "上位10件にない"):
    keys += [k5 + " ・正解", k5 + " ・不正解"]
print("予測:", PRED.replace(EVAL_ROOT + "/", ""), "/ 回答のキー:", key)
print("%-26s" % "項目" + "".join("%12s" % c for c in cols))
for k in keys:
    print("%-26s" % k + "".join("%12d" % st[(c, k)] for c in cols))
print("\n確認：全体の正解（計）が 127 なら、数え方はこれまでの評価と同じ")

予測: runs/C_L10_final_ctx5_320_20261115_0643/submission.json / 回答のキー: answer
項目                            jemhopqa       niilc           計
全体 正解                               24          78         102
全体 問題数                             120         200         320
yn 正解                                0           0           0
yn 問題数                              45           0          45
上位5件にある ・正解                         22          77          99
上位5件にある ・不正解                        24          65          89
6〜10位にだけある ・正解                       1           0           1
6〜10位にだけある ・不正解                      5           6          11
上位10件にない ・正解                         1           1           2
上位10件にない ・不正解                       22          51          73

確認：全体の正解（計）が 127 なら、数え方はこれまでの評価と同じ


In [32]:
# [D3 reader の外し方] 上位5件に正解があるのに外した非yn 問題を、外し方で分類する
# D2 と同じセッションで実行する（rows・golds・is_yn・ds・first_hit・_norm・get_chunks・load_pred・correct・pred を使う）
# 読むだけ・保存はレポートだけ。質問文・正解・回答の全文は表示しない（余分な文字の部分だけ表示する）
import json, os, glob, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
for _name in ("rows", "golds", "is_yn", "ds", "first_hit", "_norm", "get_chunks", "load_pred", "correct", "pred"):
    if _name not in globals():
        raise SystemExit(_name + " がない。D2 を2回目まで実行してから、このセルを実行する")

top5 = lambda r: [c["chunk_id"] for c in r["s1"]][:5]
TARGET = [r for r in rows if not is_yn(r["q_id"]) and first_hit(r["q_id"], top5(r)) and not correct(pred, r["q_id"])]
print("対象（上位5件にある・不正解）: %d 問（%s）\n" % (len(TARGET), dict(collections.Counter(ds(r["q_id"]) for r in TARGET))))

def ctx_text(r):
    return "".join(_norm(c["title"] + c["text"]) for c in get_chunks(top5(r)))

def classify(q, ctx):
    p = _norm(pred[q])
    gs = [_norm(x) for x in golds(q) if _norm(x)]
    if not p:
        return "1 空の回答", None
    g_in = [g for g in gs if g in p]
    if g_in:
        g = max(g_in, key=len)
        return "2 正解＋余分な文字", p.replace(g, "□", 1)
    if any(p in g for g in gs):
        return "3 正解の一部だけ", None
    if p in ctx:
        return "4 資料の別の語", None
    return "5 資料にない語", None

st = collections.Counter()
extra = collections.Counter()
plen, glen = collections.defaultdict(list), collections.defaultdict(list)
cat_of = {}
for r in TARGET:
    q, g = r["q_id"], ds(r["q_id"])
    cat, ex = classify(q, ctx_text(r))
    cat_of[q] = cat
    for gg in (g, "計"):
        st[(gg, cat)] += 1
        plen[gg].append(len(_norm(pred[q])))
        glen[gg].append(min(len(_norm(x)) for x in golds(q) if _norm(x)))
    if ex is not None:
        extra[ex if len(ex) <= 12 else ex[:12] + "…"] += 1

cols = sorted({ds(r["q_id"]) for r in TARGET}) + ["計"]
CATS = ["1 空の回答", "2 正解＋余分な文字", "3 正解の一部だけ", "4 資料の別の語", "5 資料にない語"]
print("%-24s" % "外し方" + "".join("%12s" % c for c in cols))
for k in CATS:
    print("%-24s" % k + "".join("%12d" % st[(c, k)] for c in cols))
avg = lambda v: sum(v) / max(1, len(v))
print("%-24s" % "回答の平均文字数" + "".join("%12.1f" % avg(plen[c]) for c in cols))
print("%-24s" % "正解の平均文字数" + "".join("%12.1f" % avg(glen[c]) for c in cols))

print("\n「2 正解＋余分な文字」の形（□ が正解の部分。多い順・最大20）:")
for k, v in extra.most_common(20):
    print("  %3d  %s" % (v, k))

# --- これまでの評価（14回）のどれかで正解しているか ---
runs = sorted(glob.glob(EVAL_ROOT + "/runs/*/submission.json"))
others = {}
for p in runs:
    pr, _ = load_pred(p)
    if pr is not None:
        others[p.split("/runs/")[1].split("/")[0]] = pr
GOLDCTX = [n for n in others if n.startswith("D_")]   # 正解の資料を渡した診断用の評価（実際には使えない設定）
st2 = collections.Counter()
for r in TARGET:
    q, g = r["q_id"], ds(r["q_id"])
    any_c = any(correct(pr, q) for n, pr in others.items() if n not in GOLDCTX)
    any_d = any(correct(others[n], q) for n in GOLDCTX)
    for gg in (g, "計"):
        st2[(gg, "どれかの評価で正解（D_ を除く）")] += any_c
        st2[(gg, "D_（正解の資料）で正解")] += any_d
print("\nこれまでの評価 %d 回（うち D_ は %d 回: %s）で、対象の問題を正解しているか:" % (len(others), len(GOLDCTX), ", ".join(GOLDCTX)))
for k in ("どれかの評価で正解（D_ を除く）", "D_（正解の資料）で正解"):
    print("%-34s" % k + "".join("%12d" % st2[(c, k)] for c in cols))

OUT_DIR = EVAL_ROOT + "/val_2stage_out"
rep = OUT_DIR + "/D3_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump({"n_target": len(TARGET), "stats": {"%s|%s" % k: v for k, v in st.items()},
           "stats_runs": {"%s|%s" % k: v for k, v in st2.items()},
           "extra_top": extra.most_common(50), "cat_of": cat_of},
          open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nレポート保存:", rep)

対象（上位5件にある・不正解）: 89 問（{'niilc': 65, 'jemhopqa': 24}）

外し方                         jemhopqa       niilc           計
1 空の回答                             0           0           0
2 正解＋余分な文字                         1           4           5
3 正解の一部だけ                          2          12          14
4 資料の別の語                          20          48          68
5 資料にない語                           1           1           2
回答の平均文字数                         5.3         4.7         4.8
正解の平均文字数                         5.0         4.4         4.6

「2 正解＋余分な文字」の形（□ が正解の部分。多い順・最大20）:
    1  □10月11日
    1  フローズン・□
    1  1928年□
    1  □、二級建築士、木造建築…
    1  □像

これまでの評価 14 回（うち D_ は 2 回: D_V22_s2_gold5_320_20261115_0830, D_V22_s2_rev5_320_20261115_0826）で、対象の問題を正解しているか:
どれかの評価で正解（D_ を除く）                           12          13          25
D_（正解の資料）で正解                                10          11          21

レポート保存: /content/drive/MyDrive/LLM講座最終コンペ_eval/val_2stage_out/D3_report_20261122_153015.json


In [33]:
# [D4 正解の資料の位置] 上位5件に正解がある非yn 問題で、正解を含む最初の資料の順位ごとに正解率を数える
# D3 と同じセッションで実行する。保存済みの予測を読むだけ（生成なし）。質問・正解・回答は表示しない
import json, glob, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
for _name in ("rows", "golds", "is_yn", "ds", "first_hit", "_norm", "get_chunks", "load_pred", "correct", "pred"):
    if _name not in globals():
        raise SystemExit(_name + " がない。D2・D3 を実行してから、このセルを実行する")

top5 = lambda r: [c["chunk_id"] for c in r["s1"]][:5]
_txt = {}
def n_hit(q, ids):
    # 正解の文字列を含む資料の数
    need = [c for c in ids if c not in _txt]
    for c in get_chunks(need):
        _txt[c["chunk_id"]] = _norm(c["title"] + c["text"])
    gs = [_norm(x) for x in golds(q) if _norm(x)]
    return sum(1 for c in ids if any(g in _txt[c] for g in gs))

# 比べる予測：基準（lora-final ctx5）と、v2s2 ctx5（D_gold5 と同じモデル）
RUNS = {"final_ctx5": pred}
for p in glob.glob(EVAL_ROOT + "/runs/C_V22_s2_ctx5_320_*/submission.json"):
    pr, _ = load_pred(p)
    if pr is not None:
        RUNS["v2s2_ctx5"] = pr
        break

st = collections.Counter()
for r in rows:
    q = r["q_id"]
    if is_yn(q):
        continue
    rk = first_hit(q, top5(r))
    if not rk:
        continue
    nh = n_hit(q, top5(r))
    nh_k = "1件" if nh == 1 else ("2件" if nh == 2 else "3件以上")
    for g in (ds(q), "計"):
        for name, pr in RUNS.items():
            ok = correct(pr, q)
            st[(g, name, "順位%d" % rk, "数")] += 1
            st[(g, name, "順位%d" % rk, "正解")] += ok
            st[(g, name, "含む資料" + nh_k, "数")] += 1
            st[(g, name, "含む資料" + nh_k, "正解")] += ok

GROUPS = sorted({ds(r["q_id"]) for r in rows if not is_yn(r["q_id"])}) + ["計"]
ROWS_K = ["順位%d" % i for i in range(1, 6)] + ["含む資料1件", "含む資料2件", "含む資料3件以上"]
for name in RUNS:
    print("== %s ==" % name)
    print("%-18s" % "区分" + "".join("%18s" % g for g in GROUPS))
    for k in ROWS_K:
        cells = []
        for g in GROUPS:
            n, c = st[(g, name, k, "数")], st[(g, name, k, "正解")]
            cells.append("%d/%d（%.0f%%）" % (c, n, 100.0 * c / n) if n else "-")
        print("%-18s" % k + "".join("%18s" % s for s in cells))
    print()

rep = EVAL_ROOT + "/val_2stage_out/D4_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump({"runs": list(RUNS), "stats": {"|".join(k): v for k, v in st.items()}},
          open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("レポート保存:", rep)





== final_ctx5 ==
区分                          jemhopqa             niilc                 計
順位1                       15/25（60%）       72/108（67%）       87/133（65%）
順位2                        4/12（33%）         4/14（29%）         8/26（31%）
順位3                         1/5（20%）          1/7（14%）         2/12（17%）
順位4                        1/1（100%）           0/6（0%）          1/7（14%）
順位5                         1/3（33%）           0/7（0%）         1/10（10%）
含む資料1件                     5/15（33%）         4/34（12%）         9/49（18%）
含む資料2件                      5/9（56%）         8/25（32%）        13/34（38%）
含む資料3件以上                  12/22（55%）        65/83（78%）       77/105（73%）

== v2s2_ctx5 ==
区分                          jemhopqa             niilc                 計
順位1                       18/25（72%）       70/108（65%）       88/133（66%）
順位2                        8/12（67%）         4/14（29%）        12/26（46%）
順位3                          0/5（0%）           0/7（0%）          0/12（0%）
順位4              

In [34]:

# [D5 誤答がどの資料から来たか] 上位5件に正解がある非yn の誤答について、誤答の語を含む最初の資料の順位を数える
# D4 と同じセッションで実行する。保存済みの予測を読むだけ（生成なし）。質問・正解・回答は表示しない
import json, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
for _name in ("rows", "golds", "is_yn", "ds", "first_hit", "_norm", "get_chunks", "correct", "RUNS"):
    if _name not in globals():
        raise SystemExit(_name + " がない。D2〜D4 を実行してから、このセルを実行する")

top5 = lambda r: [c["chunk_id"] for c in r["s1"]][:5]
_t5 = {}
def texts5(r):
    ids = top5(r)
    need = [c for c in ids if c not in _t5]
    for c in get_chunks(need):
        _t5[c["chunk_id"]] = _norm(c["title"] + c["text"])
    return [_t5[c] for c in ids]

def gold_ranks(q, tx):
    gs = [_norm(x) for x in golds(q) if _norm(x)]
    return [i for i, t in enumerate(tx, 1) if any(g in t for g in gs)]

def pred_rank(p, tx):
    # 誤答の語を含む最初の資料の順位（どこにもなければ None）
    p = _norm(p)
    if not p:
        return None
    for i, t in enumerate(tx, 1):
        if p in t:
            return i
    return None

def lab_pred(pr_rank, g_ranks):
    if pr_rank is None:
        return "誤答はどの資料にもない"
    if pr_rank == 1:
        return "誤答は1位の資料にある"
    if pr_rank in g_ranks:
        return "誤答は正解と同じ資料（2位以下）"
    return "誤答は別の資料（2位以下）"
LABS = ["誤答は1位の資料にある", "誤答は正解と同じ資料（2位以下）", "誤答は別の資料（2位以下）", "誤答はどの資料にもない"]

st = collections.Counter()
for r in rows:
    q = r["q_id"]
    if is_yn(q):
        continue
    tx = texts5(r)
    gr = gold_ranks(q, tx)
    if not gr:
        continue
    grp = "正解1位" if gr[0] == 1 else "正解2位以下"
    one = "（正解を含む資料1件）" if len(gr) == 1 else "（2件以上）"
    for name, pr in RUNS.items():
        if correct(pr, q):
            continue
        lab = lab_pred(pred_rank(pr.get(q, ""), tx), gr)
        for g in (ds(q), "計"):
            st[(name, g, grp, "誤答数")] += 1
            st[(name, g, grp, lab)] += 1
            st[(name, g, grp + one, "誤答数")] += 1
            st[(name, g, grp + one, lab)] += 1

GROUPS = sorted({ds(r["q_id"]) for r in rows if not is_yn(r["q_id"])}) + ["計"]
BLOCKS = ["正解1位", "正解2位以下", "正解2位以下（正解を含む資料1件）", "正解2位以下（2件以上）"]
for name in RUNS:
    print("== %s ==" % name)
    for b in BLOCKS:
        print("[%s] 誤答 %s" % (b, " / ".join("%s %d" % (g, st[(name, g, b, "誤答数")]) for g in GROUPS)))
        for lab in LABS:
            if b == "正解1位" and lab == "誤答は正解と同じ資料（2位以下）":
                continue
            print("    %-30s" % lab + "".join("%10d" % st[(name, g, b, lab)] for g in GROUPS))
    print()

rep = EVAL_ROOT + "/val_2stage_out/D5_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump({"runs": list(RUNS), "stats": {"|".join(k): v for k, v in st.items()}},
          open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("レポート保存:", rep)

== final_ctx5 ==
[正解1位] 誤答 jemhopqa 10 / niilc 36 / 計 46
    誤答は1位の資料にある                            5        26        31
    誤答は別の資料（2位以下）                          3         7        10
    誤答はどの資料にもない                            1         1         2
[正解2位以下] 誤答 jemhopqa 14 / niilc 29 / 計 43
    誤答は1位の資料にある                           11        24        35
    誤答は正解と同じ資料（2位以下）                       1         1         2
    誤答は別の資料（2位以下）                          2         4         6
    誤答はどの資料にもない                            0         0         0
[正解2位以下（正解を含む資料1件）] 誤答 jemhopqa 8 / niilc 20 / 計 28
    誤答は1位の資料にある                            6        16        22
    誤答は正解と同じ資料（2位以下）                       1         0         1
    誤答は別の資料（2位以下）                          1         4         5
    誤答はどの資料にもない                            0         0         0
[正解2位以下（2件以上）] 誤答 jemhopqa 6 / niilc 9 / 計 15
    誤答は1位の資料にある                            5         8        13
    誤答は正解と同じ資料（2位以下）   

In [35]:
# [D6 資料の件数による得失] 同じモデルで ctx5 と ctx7・ctx10 を比べ、問題ごとの得（×→○）と損（○→×）を
# 正解を含む最初の資料の順位（上位10件の中）で分けて数える。保存済みの予測を読むだけ（生成なし）
# D5 と同じセッションで実行する。質問・正解・回答は表示しない
import json, glob, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
for _name in ("rows", "golds", "is_yn", "ds", "first_hit", "_norm", "get_chunks", "load_pred", "correct"):
    if _name not in globals():
        raise SystemExit(_name + " がない。D2〜D5 を実行してから、このセルを実行する")

def find(pattern):
    # runs の下で名前が pattern に合う、開発用320問の予測（新しいものから）
    for p in sorted(glob.glob(EVAL_ROOT + "/runs/" + pattern + "/submission.json"), reverse=True):
        pr, _ = load_pred(p)
        if pr is not None:
            return p.split("/runs/")[1].split("/")[0], pr
    return None, None

PAIRS = [("C_L10_final_ctx5_320_2026111*", "C_L11_final_ctx7_320_*"),
         ("C_L10_final_ctx5_320_2026111*", "C_L12_final_ctx10_320_*"),
         ("C_V22_s2_ctx5_320_*",           "C_V22_s2_ctx10_320_*")]

def rank_lab(r):
    rk = first_hit(r["q_id"], [c["chunk_id"] for c in r["s1"]])   # 上位10件の中の順位
    if not rk: return "正解なし（10件）"
    if rk == 1: return "1位"
    if rk == 2: return "2位"
    if rk <= 5: return "3〜5位"
    return "6〜10位"
LABS = ["1位", "2位", "3〜5位", "6〜10位", "正解なし（10件）"]
NY = [r for r in rows if not is_yn(r["q_id"])]
lab_of = {r["q_id"]: rank_lab(r) for r in NY}

out = {}
for pa, pb in PAIRS:
    na, A = find(pa)
    nb, B = find(pb)
    if A is None or B is None:
        print("見つからない: %s / %s\n" % (pa, pb))
        continue
    st = collections.Counter()
    for r in NY:
        q = r["q_id"]
        a, b = correct(A, q), correct(B, q)
        k = "得" if (not a and b) else ("損" if (a and not b) else ("両方○" if a else "両方×"))
        for lab in (lab_of[q], "計"):
            st[(lab, k)] += 1
            st[(lab, "A正解")] += a
            st[(lab, "B正解")] += b
    print("== %s → %s ==" % (na, nb))
    print("%-18s%8s%8s%8s%8s%10s%10s" % ("正解の順位", "問題数", "得", "損", "差", "A 正解", "B 正解"))
    for lab in LABS + ["計"]:
        n = sum(st[(lab, k)] for k in ("得", "損", "両方○", "両方×"))
        print("%-18s%8d%8d%8d%+8d%10d%10d" % (lab, n, st[(lab, "得")], st[(lab, "損")],
              st[(lab, "得")] - st[(lab, "損")], st[(lab, "A正解")], st[(lab, "B正解")]))
    print()
    out[na + "→" + nb] = {"|".join(k): v for k, v in st.items()}

rep = EVAL_ROOT + "/val_2stage_out/D6_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump(out, open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("レポート保存:", rep)

== C_L10_final_ctx5_320_20261115_0643 → C_L11_final_ctx7_320_20261115_0648 ==
正解の順位                  問題数       得       損       差      A 正解      B 正解
1位                     133       1       2      -1        87        86
2位                      26       2       2      +0         8         8
3〜5位                    29       2       0      +2         4         6
6〜10位                   12       0       0      +0         1         1
正解なし（10件）               75       0       0      +0         2         2
計                      275       5       4      +1       102       103

== C_L10_final_ctx5_320_20261115_0643 → C_L12_final_ctx10_320_20261115_0654 ==
正解の順位                  問題数       得       損       差      A 正解      B 正解
1位                     133       4       2      +2        87        89
2位                      26       2       2      +0         8         8
3〜5位                    29       3       0      +3         4         7
6〜10位                   12       1       1      +0         1 

In [36]:
# [E2r-1 後段の切り出し] 質問を、助詞の直後の最後の「、」で前段と後段に分ける。分けられた問題の数を見る
# 検索しない・保存しない。質問文は表示しない（文字数だけ）
import re, collections
for _name in ("rows", "is_yn", "ds", "nf", "TARGET_QUESTIONS"):
    if _name not in globals():
        raise SystemExit(_name + " がない。S3-1・E1-1・E1-3 まで実行してから、このセルを実行する")

SPLIT_RE = re.compile(r"(?:が|は|の|で|に|を)、")
MIN_PART = 6
def split_residual(q):
    # (前段, 後段) を返す。分けられなければ None
    qn = nf(q)
    ms = list(SPLIT_RE.finditer(qn))
    if not ms:
        return None
    m = ms[-1]
    head, tail = qn[:m.start() + 1], qn[m.end():]
    if len(head) < MIN_PART or len(tail) < MIN_PART:
        return None
    return head, tail

QTEXT = {q["q_id"]: q["question"] for q in TARGET_QUESTIONS}
st = collections.Counter()
lens = collections.defaultdict(list)
for r in rows:
    q = r["q_id"]
    if is_yn(q):
        continue
    sp = split_residual(QTEXT[q])
    for g in (ds(q), "計"):
        st[(g, "問題数")] += 1
        st[(g, "分けられた")] += sp is not None
        if sp:
            lens[g].append((len(sp[0]), len(sp[1])))
cols = sorted({ds(r["q_id"]) for r in rows if not is_yn(r["q_id"])}) + ["計"]
print("%-16s" % "" + "".join("%14s" % c for c in cols))
for k in ("問題数", "分けられた"):
    print("%-16s" % k + "".join("%14d" % st[(c, k)] for c in cols))
avg = lambda v, i: sum(x[i] for x in v) / max(1, len(v))
print("%-16s" % "前段の平均文字数" + "".join("%14.1f" % avg(lens[c], 0) for c in cols))
print("%-16s" % "後段の平均文字数" + "".join("%14.1f" % avg(lens[c], 1) for c in cols))
print("E2r-1 OK（何も保存していない）")

                      jemhopqa         niilc             計
問題数                         75           200           275
分けられた                        4             4             8
前段の平均文字数                  16.2          15.8          16.0
後段の平均文字数                  17.5          21.2          19.4
E2r-1 OK（何も保存していない）


In [37]:
# [E2r-2 後段で検索] 分けられた非yn 問題で、RES（後段だけ）・YRES（y_key＋後段）を1段目と同じ流れで検索し、
# ORIG（E1 の s1）と比べる。生成なし。結果は val_2stage_out/trace_E2r_*.jsonl に保存（上書きしない）
import json, os, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT_DIR = EVAL_ROOT + "/val_2stage_out"
E2R_RUN_ID = "E2r_residual_K10"
TRACE2 = OUT_DIR + "/trace_" + E2R_RUN_ID + ".jsonl"
for _name in ("split_residual", "SEARCHERS", "RETRIEVERS", "fuse", "reranker", "get_chunks", "first_hit", "slot", "RERANK_CANDIDATES"):
    if _name not in globals():
        raise SystemExit(_name + " がない。E2r-1 と E1-3 を実行してから、このセルを実行する")

def search_top10(query):
    # 配布の1段目と同じ流れ（検索3種 → RRF → リランカーで上位 RERANK_CANDIDATES 件を並べ替え → 上位10件）
    fused = fuse({name: SEARCHERS[name](query) for name in RETRIEVERS})[:RERANK_CANDIDATES]
    ch = get_chunks(fused)
    sc = reranker.score(query, [c["title"] + "\n" + c["text"] for c in ch])
    order = sorted(range(len(ch)), key=lambda i: (-float(sc[i]), ch[i]["chunk_id"]))[:10]
    return [{"chunk_id": ch[i]["chunk_id"], "title": ch[i]["title"], "score": round(float(sc[i]), 6)} for i in order]

if os.path.exists(TRACE2):
    print("保存済みを読む（検索しない）:", TRACE2)
    out = [json.loads(l) for l in open(TRACE2, encoding="utf-8") if l.strip()]
else:
    out, t0 = [], time.time()
    targets = [r for r in rows if not is_yn(r["q_id"]) and split_residual(QTEXT[r["q_id"]])]
    print("対象: %d 問" % len(targets))
    for i, r in enumerate(targets, 1):
        head, tail = split_residual(QTEXT[r["q_id"]])
        res = search_top10(tail)
        y = r.get("y_key")
        yres = search_top10(y + " " + tail) if y else res
        out.append({"q_id": r["q_id"], "has_y": bool(y), "res": res, "yres": yres})
        if i % 25 == 0:
            print("  %d / %d（%.1f 分）" % (i, len(targets), (time.time() - t0) / 60))
    tmp = TRACE2 + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for o in out:
            f.write(json.dumps(o, ensure_ascii=False) + "\n")
    os.replace(tmp, TRACE2)
    print("保存: %s（%.1f 分）" % (TRACE2, (time.time() - t0) / 60))

# --- 比べる ---
S1 = {r["q_id"]: r["s1"] for r in rows}
ids = lambda lst: [c["chunk_id"] for c in lst]
st = collections.Counter()
for o in out:
    q = o["q_id"]
    s1 = S1[q]
    miss10 = not first_hit(q, ids(s1))
    for g in (ds(q), "計"):
        st[(g, "対象")] += 1
        st[(g, "y_key あり")] += o["has_y"]
        st[(g, "ORIG 10件に正解なし")] += miss10
        for name, lst in (("ORIG", s1), ("RES", o["res"]), ("YRES", o["yres"])):
            rk = first_hit(q, ids(lst))
            st[(g, name, "@1")] += rk == 1
            st[(g, name, "@5")] += bool(rk and rk <= 5)
            st[(g, name, "@10")] += bool(rk)
            if name != "ORIG":
                st[(g, name, "救えた")] += bool(miss10 and rk)
                for K in (5, 10):
                    st[(g, name, "枠K%d" % K)] += bool(first_hit(q, slot(s1, lst, K)))
        for K in (5, 10):
            st[(g, "ORIG", "枠K%d" % K)] += bool(first_hit(q, ids(s1)[:K]))

cols = sorted({ds(o["q_id"]) for o in out}) + ["計"]
print("\n%-28s" % "項目" + "".join("%14s" % c for c in cols))
for k in ("対象", "y_key あり", "ORIG 10件に正解なし"):
    print("%-28s" % k + "".join("%14d" % st[(c, k)] for c in cols))
for m in ("@1", "@5", "@10"):
    print("%-28s" % ("recall" + m + "  ORIG / RES / YRES") + "".join("%14s" % ("%d/%d/%d" % (st[(c, "ORIG", m)], st[(c, "RES", m)], st[(c, "YRES", m)])) for c in cols))
print("%-28s" % "救えた  RES / YRES" + "".join("%14s" % ("%d/%d" % (st[(c, "RES", "救えた")], st[(c, "YRES", "救えた")])) for c in cols))
for K in (5, 10):
    print("%-28s" % ("枠K%d S=1  ORIG / +RES / +YRES" % K) + "".join("%14s" % ("%d/%d/%d" % (st[(c, "ORIG", "枠K%d" % K)], st[(c, "RES", "枠K%d" % K)], st[(c, "YRES", "枠K%d" % K)])) for c in cols))

rep = OUT_DIR + "/E2r_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump({"run_id": E2R_RUN_ID, "split_re": SPLIT_RE.pattern, "min_part": MIN_PART,
           "stats": {"|".join(k) if isinstance(k, tuple) else k: v for k, v in st.items()}},
          open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nレポート保存:", rep)

対象: 8 問


KeyError: 'どちらが先に生まれましたか?'

In [38]:
# [S1 検索関数の中身] SEARCHERS の各関数のソースと、使っている大きな表を表示する（読むだけ）
import inspect
print("RETRIEVERS:", RETRIEVERS)
for name in RETRIEVERS:
    f = SEARCHERS[name]
    print("\n==== %s : %s ====" % (name, getattr(f, "__name__", type(f).__name__)))
    try:
        print(inspect.getsource(f))
    except Exception as e:
        print("（ソースを取得できない: %s）" % type(e).__name__)
    cv = inspect.getclosurevars(f) if inspect.isfunction(f) else None
    if cv:
        for k, v in {**cv.nonlocals, **cv.globals}.items():
            if isinstance(v, dict) and len(v) > 100:
                kk = next(iter(v))
                print("  表 %-20s 件数 %d / キーの型 %s / 値の型 %s" % (k, len(v), type(kk).__name__, type(v[kk]).__name__))
            elif callable(v) and not inspect.isclass(v) and not inspect.ismodule(v):
                print("  呼んでいる関数 %s" % k)

print("\n==== 埋め込みを作る関数の候補（名前に encode / embed / query を含む）====")
for k, v in sorted(globals().items()):
    if not k.startswith("_") and callable(v) and not inspect.isclass(v) and any(s in k.lower() for s in ("encode", "embed", "query")):
        try:
            print("  %-28s %s" % (k, inspect.signature(v)))
        except Exception:
            print("  %-28s" % k)
print("\nS1 終わり（何も保存していない）")

RETRIEVERS: ['bm25', 'ruri', 'bge']

==== bm25 : <lambda> ====
    SEARCHERS["bm25"] = lambda query: bm25.search(query)


==== ruri : __getitem__ ====
（ソースを取得できない: TypeError）

==== bge : __getitem__ ====
（ソースを取得できない: TypeError）

==== 埋め込みを作る関数の候補（名前に encode / embed / query を含む）====

S1 終わり（何も保存していない）


In [39]:
# [S2 質問の形] 非yn の質問を、JEMHopQA 全部（75）と NIILC 先頭30 だけ表示する（開発用の質問。読むだけ）
# 正解・資料は表示しない。1段目10件に正解があるかの印だけつける（○ある / ×ない）
QTEXT = {q["q_id"]: q["question"] for q in TARGET_QUESTIONS}
NY = [r for r in rows if not is_yn(r["q_id"])]
for g, lim in (("jemhopqa", 75), ("niilc", 30)):
    print("==== %s ====" % g)
    k = 0
    for r in NY:
        if ds(r["q_id"]) != g:
            continue
        mark = "○" if first_hit(r["q_id"], [c["chunk_id"] for c in r["s1"]]) else "×"
        print("%s %s %s" % (r["q_id"], mark, QTEXT[r["q_id"]]))
        k += 1
        if k >= lim:
            break
    print()
print("S2 終わり（何も保存していない）")

==== jemhopqa ====
dev0003 × フランシスコ・ザビエルと新島襄は、どちらが先に生まれましたか?
dev0014 ○ 映画『コクリコ坂から』を配給した会社の本社が入っているビルの名前は何ですか？
dev0017 ○ 杉咲花の父の職業は？
dev0019 ○ 長瀬智也の所属していたアイドルグループの活動開始年は？
dev0025 ○ 飛騨市と関市で市町村合併を実施した時期の早いのはどちらですか？
dev0031 × トヨタ・マークXの先代車の発売年は？
dev0032 ○ 小泉進次郎と小泉孝太郎のうち、先に出生したのはどっち？
dev0033 ○ 香取神宮と日光二荒山神社、千葉県にあるのはどちらですか？
dev0038 × タレントである辺見えみりの兄の生年月日は？
dev0041 × 1968年に販売されたトヨタ・マークIIの後継の販売開始年は？
dev0046 ○ LiSAと松本人志はどちらが先に生まれましたか？
dev0047 ○ 橋本龍太郎と徳川家広のうち、先に出生したのはどっち？
dev0048 ○ 永山瑛太の実弟の職業は？
dev0049 ○ 東條英機が死没した施設は何という戦争の後に設置されましたか？
dev0052 ○ 福山雅治と小山田圭吾、ミュージシャンとしてデビューしたのが早いのはどちらでしょう？
dev0057 ○ 大阪市の西成区と中央区ではどちらが大きいですか？
dev0058 × 生田斗真の弟は何大学何学部を卒業しましたか？
dev0071 × 太陽の塔の設計者の死亡した年月日はいつですか？
dev0078 ○ 『しあわせの保護色』の楽曲でセンターポジションを務めた人物が所属する事務所は何ですか？
dev0081 ○ 『パプリカ』の曲を作詞・作曲・プロデュースしたアーティストが音楽活動を始めた年はいつですか？
dev0086 × 犬飼貴丈が所属する事務所の代表者は誰ですか？
dev0099 ○ 西安市と武漢市、面積が広いのはどちらですか？
dev0100 ○ 長嶋茂雄と小林旭、どちらが年上ですか？
dev0108 × 漫画『チェンソーマン』作者の出身地の都道府県はどこですか？
dev0114 ○ サム・ライミとジョディ・フォスターのうち、先に出生したのはどっち？
dev0117 ○ 月亭方正と松尾諭、どちらが立正大学出

In [40]:
# [S3 ruri・bge の表を作ったコード] 表の形と、表を作ったセルのコードを実行履歴から表示する（読むだけ）
import inspect
qs = {q["question"] for q in TARGET_QUESTIONS}
for name in ("ruri", "bge"):
    f = SEARCHERS[name]
    d = getattr(f, "__self__", None)
    print("==== %s ====" % name)
    if not isinstance(d, dict):
        print("  dict ではない:", type(d).__name__)
        continue
    k = next(iter(d)); v = d[k]
    print("  表の件数 %d / うち開発用の質問 %d" % (len(d), sum(1 for x in d if x in qs)))
    print("  値の型 %s / 値の長さ %s / 先頭3つ %s" % (type(v).__name__, len(v) if hasattr(v, "__len__") else "-",
          str(list(v)[:3]) if hasattr(v, "__iter__") else str(v)[:80]))
    print("  この表を指している変数名:", [n for n, o in globals().items() if o is d and not n.startswith("_")])

if "DENSE_INDEXES" in globals():
    print("\n==== DENSE_INDEXES ====")
    di = DENSE_INDEXES
    print("  型 %s" % type(di).__name__)
    if isinstance(di, dict):
        for k, v in di.items():
            attrs = [a for a in dir(v) if not a.startswith("_")][:25]
            print("  %-8s 型 %-20s 属性 %s" % (k, type(v).__name__, attrs))

print("\n==== 表を作ったセル（実行履歴から。同じ中身は1回だけ）====")
seen = set()
for i, src in enumerate(globals().get("In", [])):
    if src in seen or "[S1" in src or "[S3" in src:
        continue
    if any(s in src for s in ('SEARCHERS["ruri"]', "SEARCHERS['ruri']", "SEARCHERS[name]", "DENSE_INDEXES", ".encode(", "query_prefix", "検索クエリ")):
        seen.add(src)
        lines = src.splitlines()
        print("---- In[%d]（%d 行）----" % (i, len(lines)))
        print("\n".join(lines[:150]))
        if len(lines) > 150:
            print("……（150 行で省略）")
print("\nS3 終わり（何も保存していない）")

==== ruri ====
  表の件数 320 / うち開発用の質問 320
  値の型 list / 値の長さ 100 / 先頭3つ [(253595, 0.897648274898529), (99898, 0.8747216463088989), (43645, 0.8722313046455383)]
  この表を指している変数名: ['d']
==== bge ====
  表の件数 320 / うち開発用の質問 320
  値の型 list / 値の長さ 100 / 先頭3つ [(253595, 0.5548064708709717), (657392, 0.5491494536399841), (874648, 0.5439466238021851)]
  この表を指している変数名: ['d']

==== DENSE_INDEXES ====
  型 dict
  ruri     型 dict                 属性 ['clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']
  bge      型 dict                 属性 ['clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']

==== 表を作ったセル（実行履歴から。同じ中身は1回だけ）====
---- In[6]（146 行）----
# [step1.1 config]
# =====================================================================
# 1. 検索の組み合わせと実行名
# =====================================================================
# ここだけ変更すれば単体・任意の組み合わせで実行できる。
RETRIEVERS = ["bm25", "ruri", "bge"]            #

In [41]:
# [S4 正解データの項目] 開発用の正解データ・質問データに、途中の答え（橋渡し）が入っているかを見る（読むだけ）
import glob, json, os
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
jq = [q for q in recs if ds(q) == "jemhopqa"]
print("正解データ（recs）の項目:", sorted(recs[jq[0]].keys()))
print("質問データ（TARGET_QUESTIONS）の項目:", sorted(TARGET_QUESTIONS[0].keys()))

print("\n開発用フォルダのファイル:")
for p in sorted(glob.glob(EVAL_ROOT + "/dev_data/*")):
    print("  %-40s %8.1f KB" % (os.path.basename(p), os.path.getsize(p) / 1024))

# 橋渡しの候補になる項目があれば、3問だけ中身を表示する（開発用の正解データ）
CAND = [k for k in recs[jq[0]].keys() if any(s in k.lower() for s in ("deriv", "triple", "evidence", "support", "hop", "bridge", "path", "fact"))]
print("\n橋渡しの候補の項目:", CAND or "なし")
for q in ("dev0038", "dev0071", "dev0129"):
    if q in recs and CAND:
        print("  %s:" % q, {k: recs[q][k] for k in CAND})
print("\nS4 終わり（何も保存していない）")

正解データ（recs）の項目: ['answers', 'source']
質問データ（TARGET_QUESTIONS）の項目: ['q_id', 'question']

開発用フォルダのファイル:
  dev_answers.json                             28.0 KB
  questions.json                               28.0 KB

橋渡しの候補の項目: なし

S4 終わり（何も保存していない）


In [43]:
# [E3a-1 前段と後段に分ける] JEMHopQA の非yn を、前段 E と後段 A に分ける。検索しない・保存しない
import re, collections
for _name in ("rows", "is_yn", "ds", "first_hit", "nf", "TARGET_QUESTIONS"):
    if _name not in globals():
        raise SystemExit(_name + " がない。E1-3 まで実行してから、このセルを実行する")
QTEXT = {q["q_id"]: q["question"] for q in TARGET_QUESTIONS}
CMP_RE = re.compile(r"どちら|どっち")
VERB_RE = re.compile(r"[をが]")
TAIL_RE = re.compile(r"[？?\s]+$")

def split_bridge(q):
    # (E, A) を返す。分けられなければ None
    s = TAIL_RE.sub("", nf(q))
    if CMP_RE.search(s):
        return None
    i = s.rfind("は")
    if i <= 0:
        return None
    T, P = s[:i], s[i + 1:]
    if VERB_RE.search(P) or "の" not in T:
        E, A = T, "は" + P
    else:
        j = T.rfind("の")
        E, A = T[:j], T[j + 1:] + "は" + P
    if len(E) < 3 or len(A) < 2 or not re.search(r"[のがを]", E):
        return None
    return E, A

E3A = []
for r in rows:
    q = r["q_id"]
    if is_yn(q) or ds(q) != "jemhopqa":
        continue
    sp = split_bridge(QTEXT[q])
    if sp:
        mark = "○" if first_hit(q, [c["chunk_id"] for c in r["s1"]]) else "×"
        E3A.append({"q_id": q, "mark": mark, "E": sp[0], "A": sp[1], "subq": sp[0] + "は？"})
cnt = collections.Counter(o["mark"] for o in E3A)
print("JEMHopQA 非yn 75 問のうち、分けられた: %d 問（1段目10件に正解 ○ %d / × %d）" % (len(E3A), cnt["○"], cnt["×"]))
for o in E3A:
    print("%s %s  E=%s ／ A=%s" % (o["q_id"], o["mark"], o["E"], o["A"]))
print("E3a-1 OK（何も保存していない）")

JEMHopQA 非yn 75 問のうち、分けられた: 42 問（1段目10件に正解 ○ 21 / × 21）
dev0014 ○  E=映画『コクリコ坂から』を配給した会社の本社が入っているビル ／ A=名前は何ですか
dev0017 ○  E=杉咲花の父 ／ A=職業は
dev0019 ○  E=長瀬智也の所属していたアイドルグループ ／ A=活動開始年は
dev0031 ×  E=トヨタ・マークXの先代車 ／ A=発売年は
dev0038 ×  E=タレントである辺見えみりの兄 ／ A=生年月日は
dev0041 ×  E=1968年に販売されたトヨタ・マークIIの後継 ／ A=販売開始年は
dev0048 ○  E=永山瑛太の実弟 ／ A=職業は
dev0049 ○  E=東條英機が死没した施設 ／ A=は何という戦争の後に設置されましたか
dev0058 ×  E=生田斗真の弟 ／ A=は何大学何学部を卒業しましたか
dev0071 ×  E=太陽の塔の設計者 ／ A=死亡した年月日はいつですか
dev0078 ○  E=『しあわせの保護色』 ／ A=楽曲でセンターポジションを務めた人物が所属する事務所は何ですか
dev0086 ×  E=犬飼貴丈が所属する事務所 ／ A=代表者は誰ですか
dev0108 ×  E=漫画『チェンソーマン』作者の出身地 ／ A=都道府県はどこですか
dev0118 ×  E=徳川家宣の父方の祖父 ／ A=死没年月日は西暦でいつですか
dev0119 ×  E=蓮佛美沙子が初出演した連続テレビ小説のオープニング曲 ／ A=アーティストは
dev0123 ○  E=島津斉興が藩主となった藩 ／ A=藩庁があった城は
dev0129 ×  E=SUBARUの筆頭株主である企業の初代社長 ／ A=名前は
dev0135 ○  E=『男はつらいよ』の主演俳優 ／ A=誕生日は何月何日
dev0145 ○  E=『おちょやん』の主演俳優 ／ A=父親は誰ですか
dev0148 ○  E=『HUNTER×HUNTER』の作者 ／ A=出身地は何県何市ですか
dev0151 ○  E=長岡京に遷都される前の都 ／ A=所在地は現在の何県にあたるでしょうか
dev0152 ○  E=松下政経塾の創立者が亡くなった ／ A=は何年ですか
dev

In [44]:
# [E3a-2 前段の問いで検索] 前段の問い（E＋「は？」）を、いつもの流れ（bm25・ruri・bge → RRF → リランカー）で検索し、上位5件を保存する
# ruri・bge は、新しい文をまとめて search_dense_index で検索し、DENSE_RESULTS に書き足す（元の320問の結果は変えない）
import json, os, time
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT_DIR = EVAL_ROOT + "/val_2stage_out"
E3A_SEARCH = OUT_DIR + "/E3a_subq_search.jsonl"
for _name in ("E3A", "search_dense_index", "DENSE_RESULTS", "DENSE_NAMES", "SEARCHERS", "fuse", "get_chunks", "reranker", "bm25"):
    if globals().get(_name) is None:
        raise SystemExit(_name + " がない。step4.2・step5.1・E3a-1 を実行してから、このセルを実行する（step7.2 を実行していたら、step3.1 からやり直す）")
for name in DENSE_NAMES:
    assert SEARCHERS[name].__self__ is DENSE_RESULTS[name], "SEARCHERS[%s] が DENSE_RESULTS を引いていない" % name

def ensure_dense(queries):
    # 表にない文だけを、まとめて検索して書き足す（同じ文は1回だけ）
    for name in DENSE_NAMES:
        add = [q for q in dict.fromkeys(queries) if q not in DENSE_RESULTS[name]]
        if add:
            t0 = time.time()
            DENSE_RESULTS[name].update(search_dense_index(name, add))
            print("  [dense] %s: %d 文を書き足した（%.0f 秒）" % (name, len(add), time.time() - t0))

def search_topk(query, k):
    # retrieve() と同じ流れ。上位 k 件の dict（chunk_id・title・text）
    fused = fuse({name: SEARCHERS[name](query) for name in RETRIEVERS})[:RERANK_CANDIDATES]
    ch = get_chunks(fused)
    sc = reranker.score(query, [c["title"] + "\n" + c["text"] for c in ch])
    order = sorted(range(len(ch)), key=lambda i: (-float(sc[i]), ch[i]["chunk_id"]))[:k]
    return [{"chunk_id": ch[i]["chunk_id"], "title": ch[i]["title"], "text": ch[i]["text"]} for i in order]

if os.path.exists(E3A_SEARCH):
    print("保存済みを読む（検索しない）:", E3A_SEARCH)
    SUBQ = {o["q_id"]: o for o in (json.loads(l) for l in open(E3A_SEARCH, encoding="utf-8") if l.strip())}
else:
    t0 = time.time()
    ensure_dense([o["subq"] for o in E3A])
    SUBQ = {}
    for o in E3A:
        SUBQ[o["q_id"]] = dict(o, contexts=search_topk(o["subq"], 5))
    tmp = E3A_SEARCH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for o in SUBQ.values():
            f.write(json.dumps(o, ensure_ascii=False) + "\n")
    os.replace(tmp, E3A_SEARCH)
    print("保存: %s / %d 問 / %.1f 分" % (E3A_SEARCH, len(SUBQ), (time.time() - t0) / 60))
print("E3a-2 OK")

Loading weights:   0%|          | 0/152 [00:00<?, ?it/s]

  [dense] ruri: 41 文を書き足した（6 秒）


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  [dense] bge: 41 文を書き足した（9 秒）
保存: /content/drive/MyDrive/LLM講座最終コンペ_eval/val_2stage_out/E3a_subq_search.jsonl / 42 問 / 1.4 分
E3a-2 OK


In [46]:
# [E3a-3 Y を出す] 前段の問いと上位5件を、lora-final（提出と同じプロンプト・同じ生成設定）に渡して答えさせる
# step7.2 は使わない（リランカーを残すため）。vLLM の起動に必要なところだけをここに持つ。結果を保存して表示する
import json, os, re, time, contextlib, importlib
E3A_Y = OUT_DIR + "/E3a_bridge_Y.jsonl"
for _name in ("SUBQ", "GENERATOR", "answer_prompt", "SEED", "free_gpu"):
    if _name not in globals():
        raise SystemExit(_name + " がない。E3a-2・[step7.1]・【提出前の照合セル】を実行してから、このセルを実行する")
assert GENERATOR["model"] == "YAMAHARZ250/qwen3.5-0.8b-rag-lora-final", GENERATOR["model"]

def e3_schema(contexts):
    return {"type": "object",
            "properties": {"answer": {"type": "string"}, "chunk_id": {"enum": [c["chunk_id"] for c in contexts] + [None]}},
            "required": ["answer", "chunk_id"], "additionalProperties": False}

def e3_parse(raw):
    try:
        return str(json.loads(raw).get("answer") or "")
    except Exception:
        m = re.match(r'\s*\{\s*"answer"\s*:\s*"((?:[^"\\]|\\.)*)', raw)
        return m.group(1) if m else ""

if globals().get("Y_LLM") is None:
    try:   # Colab 向けの回避策（step7.2 と同じ）
        _m = importlib.import_module("vllm.utils.system_utils")
        if hasattr(_m, "suppress_stdout"):
            _m.suppress_stdout = contextlib.nullcontext
    except ImportError:
        pass
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import StructuredOutputsParams
    free_gpu()
    t0 = time.time()
    G = GENERATOR
    Y_LLM = LLM(model=G["model"], revision=G["revision"], dtype=G["dtype"], max_model_len=G["max_model_len"],
                gpu_memory_utilization=0.45, enforce_eager=G["enforce_eager"], seed=SEED,
                **({"limit_mm_per_prompt": {"image": 0, "video": 0}} if G["text_only"] else {}))
    print("[vLLM] 起動: %.0f 秒（リランカーは残している）" % (time.time() - t0))
from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams

items = list(SUBQ.values())
params = [SamplingParams(max_tokens=GENERATOR["max_new_tokens"], seed=SEED,
                         **{k: GENERATOR[k] for k in ("temperature", "top_p", "top_k", "min_p", "presence_penalty", "repetition_penalty")},
                         structured_outputs=StructuredOutputsParams(json=e3_schema(o["contexts"])))
          for o in items]
convs = [[{"role": "user", "content": answer_prompt(o["subq"], o["contexts"])}] for o in items]
outs = Y_LLM.chat(convs, params, use_tqdm=False, chat_template_kwargs={"enable_thinking": GENERATOR["enable_thinking"]})
for o, out in zip(items, outs):
    o["Y"] = e3_parse(out.outputs[0].text)

if os.path.exists(E3A_Y):
    E3A_Y = E3A_Y.replace(".jsonl", "_" + time.strftime("%Y%m%d_%H%M%S") + ".jsonl")
with open(E3A_Y, "w", encoding="utf-8") as f:
    for o in items:
        f.write(json.dumps({k: o[k] for k in ("q_id", "mark", "E", "A", "subq", "Y")}, ensure_ascii=False) + "\n")
print("保存:", E3A_Y, "\n")
print("q_id    1段目  前段 E  →  Y")
for o in items:
    print("%s  %s  %s  →  %s" % (o["q_id"], o["mark"], o["E"], o["Y"]))
print("\nE3a-3 OK")

INFO 11-22 16:42:53 [api_utils.py:273] non-default args: {'dtype': 'float16', 'seed': 42, 'max_model_len': 8192, 'gpu_memory_utilization': 0.45, 'disable_log_stats': True, 'revision': '9a4d0723a493aa3c2b9570b065744ea16f4a9237', 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'YAMAHARZ250/qwen3.5-0.8b-rag-lora-final'}


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

INFO 11-22 16:43:35 [model.py:623] Resolved architecture: Qwen3_5ForConditionalGeneration
WARNING 11-22 16:43:35 [model.py:2123] Casting torch.bfloat16 to torch.float16.
INFO 11-22 16:43:35 [model.py:1788] Using max model len 8192
INFO 11-22 16:43:35 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-22 16:43:35 [vllm.py:1109] Asynchronous scheduling is enabled.
WARNING 11-22 16:43:35 [vllm.py:1163] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 11-22 16:43:35 [vllm.py:1213] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 11-22 16:43:35 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 11-22 16:43:35 [vllm.py:1392] Cudagraph is disabled under eager mode

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

... (3 outputs left out)

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

INFO 11-22 16:43:45 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
WARNING 11-22 16:43:50 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
[vLLM] 起動: 509 秒（リランカーは残している）


video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

INFO 11-22 16:51:29 [hf.py:540] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.
保存: /content/drive/MyDrive/LLM講座最終コンペ_eval/val_2stage_out/E3a_bridge_Y.jsonl 

q_id    1段目  前段 E  →  Y
dev0014  ○  映画『コクリコ坂から』を配給した会社の本社が入っているビル  →  東宝日比谷ビル
dev0017  ○  杉咲花の父  →  木暮武彦
dev0019  ○  長瀬智也の所属していたアイドルグループ  →  TOKIO
dev0031  ×  トヨタ・マークXの先代車  →  マークX
dev0038  ×  タレントである辺見えみりの兄  →  辺見鑑孝
dev0041  ×  1968年に販売されたトヨタ・マークIIの後継  →  セリカ
dev0048  ○  永山瑛太の実弟  →  永山絢斗
dev0049  ○  東條英機が死没した施設  →  巣鴨拘置所
dev0058  ×  生田斗真の弟  →  萩原健也
dev0071  ×  太陽の塔の設計者  →  丹下健三
dev0078  ○  『しあわせの保護色』  →  乃木坂46
dev0086  ×  犬飼貴丈が所属する事務所  →  青二プロダクション
dev0108  ×  漫画『チェンソーマン』作者の出身地  →  日本
dev0118  ×  徳川家宣の父方の祖父  →  徳川慶頼
dev0119  ×  蓮佛美沙子が初出演した連続テレビ小説のオープニング曲  →  おひさま
dev0123  ○  島津斉興が藩主となった藩  →  薩摩藩
dev0129  ×  SUBARUの筆頭株主である企業の初代社長  →  北謙治
dev0135  ○  『男はつらいよ』の主演俳優  →  渥美清
dev0145  ○  『おちょやん』の主演俳優  →  杉咲花
dev0148  ○  『HUNTER×HUNTER』の作者  →  冨樫義博
dev0151  ○  長岡京

In [15]:
# [E3b Y＋後段で検索し直す] × の問題で、MODEL（出た Y）と ORACLE（正しい Y）＋後段 A で検索し、ORIG と比べる
# 実行順：セッション再起動 → Drive マウント → step0.9〜step5.1 → このセル。生成なし・提出なし
# ORACLE_Y は開発用の上限の測定だけに使う（提出・本番には使わない）
import json, os, re, time, unicodedata, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT_DIR = EVAL_ROOT + "/val_2stage_out"
for _name in ("SEARCHERS", "DENSE_RESULTS", "DENSE_NAMES", "search_dense_index", "fuse", "get_chunks", "reranker", "RETRIEVERS", "RERANK_CANDIDATES"):
    if globals().get(_name) is None:
        raise SystemExit(_name + " がない。step0.9〜step5.1 を実行してから、このセルを実行する")

# ---- 正しい Y（要確認。分からない行は None）----
ORACLE_Y = {
    "dev0031": "マークII", "dev0038": None, "dev0041": "マークX", "dev0058": "生田竜聖",
    "dev0071": "岡本太郎", "dev0086": None, "dev0108": "藤本タツキ", "dev0118": "徳川家光",
    "dev0119": None, "dev0129": "トヨタ自動車", "dev0185": "愛子", "dev0186": "大人計画",
    "dev0210": "Apple", "dev0211": "ワイルド・スピード ICE BREAK", "dev0224": "アップル・インク",
    "dev0258": "日本プロ野球名球会", "dev0261": "徳川家光", "dev0263": "許斐剛",
    "dev0275": None, "dev0304": None, "dev0308": None,
}
# E に属性語まで入った問題は、後段 A を直す（E3a-1 の分け方のずれの手直し。ORACLE・MODEL 共通）
A_FIX = {"dev0108": "出身地の都道府県はどこですか", "dev0129": "初代社長の名前は",
         "dev0186": "代表者(社長)のお名前は何ですか", "dev0263": "出身地の都道府県は"}

# ---- 読み込み ----
_norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())
ans = json.load(open(EVAL_ROOT + "/dev_data/dev_answers.json", encoding="utf-8"))
ans = {k: (v if isinstance(v, dict) else {"answers": v}) for k, v in ans.items()} if isinstance(ans, dict) else {r["q_id"]: r for r in ans}
golds = lambda q: [g for g in ([ans[q]["answers"]] if isinstance(ans[q]["answers"], str) else ans[q]["answers"]) if _norm(g)]
S1 = {}
for l in open(OUT_DIR + "/trace_V2_phase1a_Q2_K10_Adiag.jsonl", encoding="utf-8"):
    if l.strip():
        o = json.loads(l)
        S1[o["q_id"]] = [c["chunk_id"] for c in o["s1"]]
YS = {}
for l in open(OUT_DIR + "/E3a_bridge_Y.jsonl", encoding="utf-8"):
    if l.strip():
        o = json.loads(l)
        YS[o["q_id"]] = o
TARGET = [q for q in ORACLE_Y if q in YS and YS[q]["mark"] == "×"]
print("対象（× の問題）: %d 問 / 正しい Y あり %d 問" % (len(TARGET), sum(1 for q in TARGET if ORACLE_Y[q])))

_txt = {}
def first_hit(q, ids):
    need = [c for c in ids if c not in _txt]
    for c in get_chunks(need):
        _txt[c["chunk_id"]] = _norm(c["title"] + c["text"])
    gs = [_norm(g) for g in golds(q)]
    for i, c in enumerate(ids, 1):
        if any(g in _txt[c] for g in gs):
            return i
    return None

def make_q(y, a):
    return y + (a if a.startswith("は") else "の" + a)

def ensure_dense(queries):
    for name in DENSE_NAMES:
        add = [q for q in dict.fromkeys(queries) if q not in DENSE_RESULTS[name]]
        if add:
            DENSE_RESULTS[name].update(search_dense_index(name, add))
            print("  [dense] %s: %d 文を書き足した" % (name, len(add)))

def search_ids(query, k=10):
    fused = fuse({name: SEARCHERS[name](query) for name in RETRIEVERS})[:RERANK_CANDIDATES]
    ch = get_chunks(fused)
    sc = reranker.score(query, [c["title"] + "\n" + c["text"] for c in ch])
    order = sorted(range(len(ch)), key=lambda i: (-float(sc[i]), ch[i]["chunk_id"]))[:k]
    return [ch[i]["chunk_id"] for i in order], {ch[i]["chunk_id"]: ch[i]["title"] for i in order}

def slot5(q, s1, s2, titles2):
    # ORIG の上位4件 ＋ 2段目の1件（ORIG の4件と同じ記事・同じチャンクは飛ばす）
    head = s1[:4]
    ht = {c["title"] for c in get_chunks(head)}
    pick = [c for c in s2 if c not in head and titles2[c] not in ht][:1]
    return head + pick

# ---- クエリを作る ----
Q = {}
for q in TARGET:
    a = A_FIX.get(q, YS[q]["A"])
    Q[(q, "MODEL")] = make_q(YS[q]["Y"], a) if YS[q]["Y"] else None
    Q[(q, "ORACLE")] = make_q(ORACLE_Y[q], a) if ORACLE_Y[q] else None
t0 = time.time()
ensure_dense([v for v in Q.values() if v])

# ---- 検索して比べる ----
rows_out, st = [], collections.Counter()
for q in TARGET:
    rec = {"q_id": q, "orig_rank10": first_hit(q, S1[q])}
    for kind in ("MODEL", "ORACLE"):
        if not Q[(q, kind)]:
            rec[kind] = None
            continue
        ids, titles = search_ids(Q[(q, kind)])
        rk = first_hit(q, ids)
        rk5 = first_hit(q, slot5(q, S1[q], ids, titles))
        rec[kind] = {"query": Q[(q, kind)], "rank10": rk, "slot5": rk5}
        st[(kind, "測った")] += 1
        st[(kind, "上位1件")] += rk == 1
        st[(kind, "上位5件")] += bool(rk and rk <= 5)
        st[(kind, "上位10件")] += bool(rk)
        st[(kind, "枠K5（ORIG4＋1）")] += bool(rk5)
    rows_out.append(rec)
print("（%.1f 分）\n" % ((time.time() - t0) / 60))

print("q_id     MODEL の順位   ORACLE の順位   クエリ（ORACLE）")
for r in rows_out:
    f = lambda k: "-" if r[k] is None else (str(r[k]["rank10"]) if r[k]["rank10"] else "なし")
    print("%s   %-12s   %-12s   %s" % (r["q_id"], f("MODEL"), f("ORACLE"), (r["ORACLE"] or {}).get("query", "")))
print("\n%-20s%10s%10s" % ("（× の問題）", "MODEL", "ORACLE"))
for k in ("測った", "上位1件", "上位5件", "上位10件", "枠K5（ORIG4＋1）"):
    print("%-20s%10d%10d" % (k, st[("MODEL", k)], st[("ORACLE", k)]))
print("参考：ORIG はこの問題すべてで上位10件に正解なし（0 件）")

rep = OUT_DIR + "/E3b_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump({"oracle_y": ORACLE_Y, "a_fix": A_FIX, "rows": rows_out, "stats": {"|".join(k): v for k, v in st.items()}},
          open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("レポート保存:", rep)

対象（× の問題）: 21 問 / 正しい Y あり 15 問


Loading weights:   0%|          | 0/152 [00:00<?, ?it/s]

  [dense] ruri: 33 文を書き足した


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  [dense] bge: 33 文を書き足した
（1.1 分）

q_id     MODEL の順位   ORACLE の順位   クエリ（ORACLE）
dev0031   なし             1              マークIIの発売年は
dev0038   なし             -              
dev0041   4              1              マークXの販売開始年は
dev0058   なし             1              生田竜聖は何大学何学部を卒業しましたか
dev0071   なし             なし             岡本太郎の死亡した年月日はいつですか
dev0086   なし             -              
dev0108   なし             なし             藤本タツキの出身地の都道府県はどこですか
dev0118   なし             なし             徳川家光の死没年月日は西暦でいつですか
dev0119   なし             -              
dev0129   なし             1              トヨタ自動車の初代社長の名前は
dev0185   なし             なし             愛子の生年月日は
dev0186   なし             なし             大人計画の代表者(社長)のお名前は何ですか
dev0210   なし             なし             Appleの本社所在地は
dev0211   なし             なし             ワイルド・スピード ICE BREAKの監督は誰
dev0224   なし             なし             アップル・インクのCOOは誰ですか
dev0258   なし             なし             日本プロ野球名球会の設立年月日はいつですか
dev0261   なし             1              徳川家光の墓所は

In [23]:
# [R3-0b 学習データで正解の資料が何位にあったか（探し方を直した版）] 読むだけ。学習しない・生成しない
# 1回目：CTX = ANS = "" のまま → q_id が tr のファイルの一覧。2回目：選んだパスを入れる
import json, glob, os, re, time, unicodedata, collections
DRIVE = "/content/drive/MyDrive"
CTX = DRIVE + "/LLM講座最終コンペ_lora/train_contexts_1500.jsonl"
ANS = DRIVE + "/LLM講座最終コンペ_lora/train_answers.json"
MAX_MB = 300

_norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())
CTX_RE = re.compile(r"\[chunk_id=(\d+)\]\s*([^\n]*)\n(.*?)(?=\n\n\[chunk_id=|\n\n質問|\Z)", re.S)
QID_KEYS = ("q_id", "qid", "id")

def load_any(p):
    # リスト・辞書・jsonl・parquet・csv を、[dict, ...] にそろえて返す（読めなければ None）
    try:
        if p.endswith(".jsonl"):
            return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
        if p.endswith(".json"):
            d = json.load(open(p, encoding="utf-8"))
        elif p.endswith(".parquet"):
            import pandas as pd
            d = pd.read_parquet(p).to_dict("records")
        elif p.endswith(".csv"):
            import pandas as pd
            d = pd.read_csv(p).to_dict("records")
        else:
            return None
    except Exception:
        return None
    if isinstance(d, dict):
        if d and all(isinstance(k, str) for k in list(d)[:5]):
            d = [dict(v, q_id=k) if isinstance(v, dict) else {"q_id": k, "answers": v} for k, v in d.items()]
        else:
            return None
    return d if isinstance(d, list) else None

def qid_of(o):
    return str(next((o[k] for k in QID_KEYS if isinstance(o, dict) and k in o), ""))

def ctx_of(o):
    # 資料の本文のリスト（なければ None）
    c = o.get("contexts") or o.get("context") or o.get("passages")
    if isinstance(c, list) and c:
        return [str(x.get("title", "")) + str(x.get("text", "")) if isinstance(x, dict) else str(x) for x in c]
    msgs = o.get("messages") or o.get("conversations")
    if isinstance(msgs, list):
        user = next((m.get("content", "") for m in msgs if isinstance(m, dict) and m.get("role") in ("user", "human")), "")
        t = [a + b for _, a, b in CTX_RE.findall(user)]
        return t or None
    p = o.get("prompt") or o.get("input") or o.get("text")
    if isinstance(p, str) and "[chunk_id=" in p:
        return [a + b for _, a, b in CTX_RE.findall(p)] or None
    return None

def ans_of(o):
    # 正解のリスト（なければ None）
    for k in ("answers", "answer", "gold", "gold_answers", "output", "completion"):
        if k in o and o[k] is not None:
            v = o[k]
            if isinstance(v, str) and v.strip().startswith("{"):
                try:
                    v = json.loads(v).get("answer")
                except Exception:
                    m = re.search(r'"answer"\s*:\s*"([^"]*)"', v)
                    v = m.group(1) if m else None
            if v is None:
                continue
            return [v] if isinstance(v, str) else [str(x) for x in v]
    msgs = o.get("messages") or o.get("conversations")
    if isinstance(msgs, list):
        asst = next((m.get("content", "") for m in msgs if isinstance(m, dict) and m.get("role") in ("assistant", "gpt")), "")
        m = re.search(r'"answer"\s*:\s*"([^"]*)"', asst)
        return [m.group(1)] if m else None
    return None

if not (CTX and ANS):
    files = [p for ext in ("json", "jsonl", "parquet", "csv") for p in glob.glob(DRIVE + "/**/*." + ext, recursive=True)]
    files = [p for p in files if os.path.getsize(p) < MAX_MB * 2**20]
    t0, found = time.time(), []
    for p in files:
        d = load_any(p)
        if not d or not isinstance(d[0], dict):
            continue
        ids = [qid_of(o) for o in d[:200]]
        if sum(i.startswith("tr") for i in ids) < len(ids) * 0.5:
            continue
        has_c = sum(ctx_of(o) is not None for o in d[:20])
        has_a = sum(ans_of(o) is not None for o in d[:20])
        found.append((p, len(d), has_c, has_a, sorted(d[0].keys())[:10]))
    found.sort(key=lambda x: os.path.getmtime(x[0]), reverse=True)
    print("q_id が tr のファイル: %d 件（調べたファイル %d 件 / %.0f 秒）\n" % (len(found), len(files), time.time() - t0))
    print("件数    資料  正解  更新日時      項目 / パス")
    for p, n, hc, ha, keys in found[:40]:
        print("%-6d  %-4s  %-4s  %-12s  %s\n        %s" % (
            n, "○" if hc >= 10 else "×", "○" if ha >= 10 else "×",
            time.strftime("%m-%d %H:%M", time.localtime(os.path.getmtime(p))), keys, p.replace(DRIVE + "/", "")))
    raise SystemExit("資料 ○ のファイルを CTX、正解 ○ のファイルを ANS に入れて、もう一度実行する（なければ、この出力を貼る）")

# ---- 数える ----
dc, da = load_any(CTX), load_any(ANS)
C = {qid_of(o): ctx_of(o) for o in dc if ctx_of(o)}
A = {qid_of(o): ans_of(o) for o in da if ans_of(o)}
common = [q for q in C if q in A]
print("資料: %s / %d 問" % (CTX.replace(DRIVE + "/", ""), len(C)))
print("正解: %s / %d 問" % (ANS.replace(DRIVE + "/", ""), len(A)))
print("両方そろった: %d 問 / q_id 先頭2文字 %s" % (len(common), dict(collections.Counter(q[:2] for q in common))))
assert common and not any(q.startswith(("dev", "te")) for q in common), "開発用・本番の質問が混ざっている"

YN = {"はい", "いいえ", "yes", "no"}
st, n_ctx = collections.Counter(), collections.Counter()
for q in common:
    gs = [_norm(g) for g in A[q] if _norm(g)]
    texts = C[q]
    n_ctx[len(texts)] += 1
    if gs and all(g in YN for g in gs):
        st["はい/いいえ"] += 1
        continue
    if any("特定できません" in g for g in gs):
        st["答え：資料からは特定できません"] += 1
        continue
    tx = [_norm(t) for t in texts]
    rk = next((i for i, t in enumerate(tx, 1) if any(g in t for g in gs)), None)
    nh = sum(1 for t in tx if any(g in t for g in gs))
    st["非yn"] += 1
    st["正解の最初の資料: " + ("%d位" % rk if rk and rk <= 5 else ("6位以下" if rk else "資料になし"))] += 1
    st["正解を含む資料の数: " + ("0件" if nh == 0 else ("1件" if nh == 1 else ("2件" if nh == 2 else "3件以上")))] += 1

print("\n1問あたりの資料の数:", dict(sorted(n_ctx.items())))
for k in ("はい/いいえ", "答え：資料からは特定できません", "非yn"):
    print("%-34s %5d" % (k, st[k]))
print("--- 非yn の内訳（参考：開発用 非yn 275 問・資料5件：1位 133（48%）/ 2位 26 / 3〜5位 29 / 5件になし 87）---")
for k in ["正解の最初の資料: %d位" % i for i in range(1, 6)] + ["正解の最初の資料: 6位以下", "正解の最初の資料: 資料になし",
          "正解を含む資料の数: 0件", "正解を含む資料の数: 1件", "正解を含む資料の数: 2件", "正解を含む資料の数: 3件以上"]:
    print("%-34s %5d（%.0f%%）" % (k, st[k], 100.0 * st[k] / max(1, st["非yn"])))

rep = DRIVE + "/LLM講座最終コンペ_eval/val_2stage_out/R3_0_report_" + time.strftime("%Y%m%d_%H%M%S") + ".json"
json.dump({"ctx": CTX, "ans": ANS, "n": len(common), "stats": dict(st), "n_ctx": dict(n_ctx)},
          open(rep, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nレポート保存:", rep)

資料: LLM講座最終コンペ_lora/train_contexts_1500.jsonl / 1500 問
正解: LLM講座最終コンペ_lora/train_answers.json / 1500 問
両方そろった: 1500 問 / q_id 先頭2文字 {'tr': 1500}

1問あたりの資料の数: {5: 1500}
はい/いいえ                               290
答え：資料からは特定できません                        0
非yn                                 1210
--- 非yn の内訳（参考：開発用 非yn 275 問・資料5件：1位 133（48%）/ 2位 26 / 3〜5位 29 / 5件になし 87）---
正解の最初の資料: 1位                         558（46%）
正解の最初の資料: 2位                          98（8%）
正解の最初の資料: 3位                          46（4%）
正解の最初の資料: 4位                          28（2%）
正解の最初の資料: 5位                          19（2%）
正解の最初の資料: 6位以下                         0（0%）
正解の最初の資料: 資料になし                      461（38%）
正解を含む資料の数: 0件                        461（38%）
正解を含む資料の数: 1件                        179（15%）
正解を含む資料の数: 2件                        130（11%）
正解を含む資料の数: 3件以上                      440（36%）

レポート保存: /content/drive/MyDrive/LLM講座最終コンペ_eval/val_2stage_out/R3_0_report_20261122_175254.json


In [24]:
# [R3-1 学習データの R3a・R3b を作る] 元のファイルは読むだけ。新しいフォルダに、元と同じファイル名・同じ形で書く
# R3a：2位以下に正解がある非yn を2回入れる ／ R3b：資料に正解がない非yn を外す。学習しない・GPU を使わない
import json, os, re, copy, unicodedata, collections
DRIVE = "/content/drive/MyDrive"
L = DRIVE + "/LLM講座最終コンペ_lora"
SRC = {"contexts": L + "/train_contexts_1500.jsonl",
       "answers": L + "/train_answers.json",
       "questions": L + "/train_freeze/questions.json"}
OUT = L + "/r3_data"
for k, p in SRC.items():
    if not os.path.exists(p):
        raise SystemExit("元のファイルがない（%s）: %s" % (k, p))

_norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())
YN = {"はい", "いいえ", "yes", "no"}

# ---- 読む（形を覚えておき、同じ形で書く）----
C = [json.loads(l) for l in open(SRC["contexts"], encoding="utf-8") if l.strip()]
A_raw = json.load(open(SRC["answers"], encoding="utf-8"))
Q_raw = json.load(open(SRC["questions"], encoding="utf-8"))

def as_map(raw, key):
    # {q_id: 元の1件分} にする（辞書の形・リストの形のどちらでも）
    if isinstance(raw, dict):
        return dict(raw)
    return {r["q_id"]: r for r in raw}
A_map, Q_map = as_map(A_raw, "answers"), as_map(Q_raw, "question")

def golds(q):
    v = A_map[q]
    v = v["answers"] if isinstance(v, dict) and "answers" in v else v
    return [v] if isinstance(v, str) else list(v)

ids_c = [o["q_id"] for o in C]
assert len(ids_c) == len(set(ids_c)) == 1500, "資料のファイルの q_id が 1500 件・重複なし、になっていない"
assert set(ids_c) == set(A_map) == set(Q_map), "資料・正解・質問の q_id がそろっていない"
assert all(q.startswith("tr") for q in ids_c), "q_id が tr で始まらないものがある"

# ---- 問題ごとの区分（R3-0 と同じ数え方）----
cat = {}
for o in C:
    q = o["q_id"]
    gs = [_norm(g) for g in golds(q) if _norm(g)]
    if gs and all(g in YN for g in gs):
        cat[q] = "yn"
        continue
    tx = [_norm(str(c.get("title", "")) + str(c.get("text", ""))) for c in o["contexts"]]
    rk = next((i for i, t in enumerate(tx, 1) if any(g in t for g in gs)), None)
    cat[q] = "なし" if rk is None else ("1位" if rk == 1 else "2位以下")
print("区分:", dict(collections.Counter(cat.values())))

# ---- 作る ----
def build(name, keep, dup):
    # keep: 残す q_id の集合 / dup: 2回入れる q_id の集合（keep の中から）
    d = OUT + "/" + name
    if os.path.exists(d):
        raise SystemExit("すでにある（上書き防止）: " + d + "（作り直すなら、フォルダの名前を変えるか消してから）")
    pairs = [(q, q) for q in ids_c if q in keep] + [(q + "_dup", q) for q in ids_c if q in dup]
    os.makedirs(d)
    # 資料（jsonl・元と同じ並び。2回目の分は最後にまとめる）
    with open(d + "/train_contexts_1500.jsonl", "w", encoding="utf-8") as f:
        byq = {o["q_id"]: o for o in C}
        for new, old in pairs:
            o = copy.deepcopy(byq[old]); o["q_id"] = new
            f.write(json.dumps(o, ensure_ascii=False) + "\n")
    # 正解・質問（元と同じ形）
    for raw, mp, fname in ((A_raw, A_map, "train_answers.json"), (Q_raw, Q_map, "questions.json")):
        if isinstance(raw, dict):
            out = {new: copy.deepcopy(mp[old]) for new, old in pairs}
        else:
            out = []
            for new, old in pairs:
                r = copy.deepcopy(mp[old]); r["q_id"] = new; out.append(r)
        json.dump(out, open(d + "/" + fname, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    cc = collections.Counter(cat[old] for _, old in pairs)
    r1, r2 = cc["1位"], cc["2位以下"]
    print("%s: %d 問 / 内訳 %s / 資料に正解がある非yn のうち 2位以下 %.0f%%  → %s"
          % (name, len(pairs), dict(cc), 100.0 * r2 / max(1, r1 + r2), d))

ALL = set(ids_c)
build("r3a", keep=ALL, dup={q for q in ALL if cat[q] == "2位以下"})
build("r3b", keep={q for q in ALL if cat[q] != "なし"}, dup=set())

json.dump({"src": SRC, "cat_count": dict(collections.Counter(cat.values())), "cat": cat},
          open(OUT + "/r3_categories.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("区分の記録:", OUT + "/r3_categories.json")
print("R3-1 OK")

区分: {'1位': 558, 'yn': 290, 'なし': 461, '2位以下': 191}
r3a: 1691 問 / 内訳 {'1位': 558, 'yn': 290, 'なし': 461, '2位以下': 382} / 資料に正解がある非yn のうち 2位以下 41%  → /content/drive/MyDrive/LLM講座最終コンペ_lora/r3_data/r3a
r3b: 1039 問 / 内訳 {'1位': 558, 'yn': 290, '2位以下': 191} / 資料に正解がある非yn のうち 2位以下 26%  → /content/drive/MyDrive/LLM講座最終コンペ_lora/r3_data/r3b
区分の記録: /content/drive/MyDrive/LLM講座最終コンペ_lora/r3_data/r3_categories.json
R3-1 OK


In [ ]:
raise SystemExit("val_2stage では S3-2 を実行しない（v2 の保存ファイルを上書きするため）")

# 【S3-2 二段検索の結果を保存】新しい名前で保存する。既存の dev_contexts.jsonl は読むだけ（1段目の再現確認）
import json, os, time, hashlib
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT = EVAL_ROOT + "/dev_contexts_2stage_v2.jsonl"
OLD = EVAL_ROOT + "/dev_contexts.jsonl"
assert "LLM講座最終コンペ_eval" in str(WORK_DIR) and "LLM講座最終コンペ_work" not in str(WORK_DIR), WORK_DIR
assert len(TARGET_QUESTIONS) == 320 and all(q["q_id"].startswith("dev") for q in TARGET_QUESTIONS), "開発用の320問ではない"
assert not os.path.exists(OUT), "すでにある（上書き防止）: " + OUT

t0 = time.time()
rows, na_list, nb_list = [], [], []
for i, q in enumerate(TARGET_QUESTIONS, 1):
    r, na, nb = retrieve_2stage(q["question"])
    rows.append({"q_id": q["q_id"], "question": q["question"], **r})
    na_list.append(na); nb_list.append(nb)
    if i % 40 == 0:
        print("  %d / 320（%.1f 分）" % (i, (time.time() - t0) / 60))
tmp = OUT + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    for o in rows:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")
os.replace(tmp, OUT)
sha = hashlib.sha256(open(OUT, "rb").read()).hexdigest()
print("保存: %s / %d 問 / %.1f 分 / sha256 先頭 %s" % (os.path.basename(OUT), len(rows), (time.time() - t0) / 60, sha[:12]))
print("手がかり a の記事名がある問題: %d / b: %d" % (sum(1 for x in na_list if x), sum(1 for x in nb_list if x)))

# 1段目の再現確認（既存の dev_contexts.jsonl の上位10件と、chunk_id の並びが同じか）
old = {}
for l in open(OLD, encoding="utf-8"):
    if l.strip():
        o = json.loads(l)
        old[o["q_id"]] = [c["chunk_id"] for c in o["contexts"]]
same = sum(1 for o in rows if [c["chunk_id"] for c in o["s1"]] == old.get(o["q_id"]))
same5 = sum(1 for o in rows if [c["chunk_id"] for c in o["s1"]][:5] == (old.get(o["q_id"]) or [])[:5])
print("1段目の再現: 上位10件が同じ %d / 320・上位5件が同じ %d / 320" % (same, same5))

  40 / 320（2.9 分）
  80 / 320（5.8 分）
  120 / 320（8.7 分）
  160 / 320（11.4 分）
  200 / 320（14.1 分）
  240 / 320（16.7 分）
  280 / 320（19.5 分）
  320 / 320（22.4 分）
保存: dev_contexts_2stage_v2.jsonl / 320 問 / 22.4 分 / sha256 先頭 b6397ccf4f98
手がかり a の記事名がある問題: 167 / b: 320
1段目の再現: 上位10件が同じ 320 / 320・上位5件が同じ 320 / 320


In [ ]:
# 【S3-3 二段検索を検索だけで評価】資料の中に正解の文字列がある問題の数を、組み合わせごとに数える（質問・正解は表示しない）
import json, re, glob, unicodedata, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
IN = EVAL_ROOT + "/dev_contexts_2stage_v2.jsonl"
ANS_PATH = "/content/drive/MyDrive/LLM講座最終コンペ_eval/dev_data/dev_answers.json"   # ← 開発用の正解ファイルのパス（評価ノートブックで読んでいるもの）
if not ANS_PATH:
    print("候補（ファイル名だけ）:")
    for p in sorted(glob.glob(EVAL_ROOT + "/**/*.json*", recursive=True)):
        if any(w in p.lower() for w in ("answer", "gold", "label", "dev")) and "contexts" not in p:
            print("  ", p.replace(EVAL_ROOT + "/", ""))
    raise SystemExit("ANS_PATH を入れて、もう一度実行する")

# --- 正解の読み込み（dict 形式・list 形式のどちらでも）---
raw = json.load(open(ANS_PATH, encoding="utf-8")) if ANS_PATH.endswith(".json") else [json.loads(l) for l in open(ANS_PATH, encoding="utf-8") if l.strip()]
if isinstance(raw, dict):
    recs = {k: (v if isinstance(v, dict) else {"answers": v}) for k, v in raw.items()}
else:
    recs = {r["q_id"]: r for r in raw}
first = next(iter(recs.values()))
print("正解ファイルの1件目のキー:", sorted(first.keys()))
AKEY = next(k for k in ("answers", "answer", "gold", "gold_answers") if k in first)
DKEY = next((k for k in ("dataset", "source", "origin", "task") if k in first), None)
_norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())
YESNO = {"はい", "いいえ", "yes", "no"}

rows = [json.loads(l) for l in open(IN, encoding="utf-8") if l.strip()]
assert len(rows) == 320 and all(r["q_id"] in recs for r in rows)
def golds(qid):
    a = recs[qid][AKEY]
    return [a] if isinstance(a, str) else list(a)
def is_yn(qid):
    g = [_norm(x) for x in golds(qid)]
    return bool(g) and all(x in YESNO for x in g)
def ds(qid):
    return str(recs[qid].get(DKEY, "?"))[:10] if DKEY else "?"
def hit(qid, ctx):
    g = [_norm(x) for x in golds(qid) if _norm(x)]
    return any(x in _norm(c["title"] + c["text"]) for c in ctx for x in g)

NY = [r for r in rows if not is_yn(r["q_id"])]
groups = collections.Counter(ds(r["q_id"]) for r in NY)
print("非yn: %d 問 / 内訳: %s" % (len(NY), dict(groups)))

def combine(r, K, S, pool_tags, mode):
    s1 = r["s1"]
    pool, seen = [], set()
    for c in sorted([c for t in pool_tags for c in r[t]], key=lambda c: -c["score"]):
        if c["chunk_id"] not in seen:
            seen.add(c["chunk_id"]); pool.append(c)
    if mode == "merge":            # 1段目10件と2段目を、リランカーのスコアだけで並べて上位 K 件
        allc, seen2 = [], set()
        for c in sorted(s1 + pool, key=lambda c: -c["score"]):
            if c["chunk_id"] not in seen2:
                seen2.add(c["chunk_id"]); allc.append(c)
        return allc[:K]
    head = s1[:K - S]              # "slot": 1段目の上位 K-S 件 ＋ 2段目の上位 S 件
    ids = {c["chunk_id"] for c in head}
    return head + [c for c in pool if c["chunk_id"] not in ids][:S]

def count(K, S, tags, mode):
    tot = sum(hit(r["q_id"], combine(r, K, S, tags, mode)) for r in NY)
    by = {g: sum(hit(r["q_id"], combine(r, K, S, tags, mode)) for r in NY if ds(r["q_id"]) == g) for g in groups}
    return tot, by

base5 = count(5, 0, [], "slot")
print("\n基準（1段目のみ・5件）: %d / %d  %s   ※E13 の数え方では 190" % (base5[0], len(NY), base5[1]))
print("\n%-6s %-4s %-3s %-6s %6s %6s  %s" % ("方式", "K", "S", "2段目", "当たり", "基準差", "内訳"))
for mode in ("slot", "merge"):
    for K in (5, 7, 10):
        for tags in (["s2a"], ["s2b"], ["s2a", "s2b"]):
            for S in ((1, 2, 3) if mode == "slot" else (0,)):
                if S >= K:
                    continue
                tot, by = count(K, S, tags, mode)
                print("%-6s %-4d %-3s %-6s %6d %+6d  %s" % (mode, K, S if mode == "slot" else "-", "+".join(t[-1] for t in tags), tot, tot - base5[0], by))
    for K in (7, 10):
        tot, by = count(K, 0, [], "slot")
        print("%-6s %-4d %-3s %-6s %6d %+6d  %s" % ("1段目", K, 0, "なし", tot, tot - base5[0], by))
    print()

正解ファイルの1件目のキー: ['answers', 'source']
非yn: 275 問 / 内訳: {'niilc': 200, 'jemhopqa': 75}

基準（1段目のみ・5件）: 188 / 275  {'niilc': 142, 'jemhopqa': 46}   ※E13 の数え方では 190

方式     K    S   2段目       当たり    基準差  内訳
slot   5    1   a         179     -9  {'niilc': 136, 'jemhopqa': 43}
slot   5    2   a         174    -14  {'niilc': 132, 'jemhopqa': 42}
slot   5    3   a         163    -25  {'niilc': 126, 'jemhopqa': 37}
slot   5    1   b         183     -5  {'niilc': 138, 'jemhopqa': 45}
slot   5    2   b         179     -9  {'niilc': 135, 'jemhopqa': 44}
slot   5    3   b         171    -17  {'niilc': 131, 'jemhopqa': 40}
slot   5    1   a+b       181     -7  {'niilc': 138, 'jemhopqa': 43}
slot   5    2   a+b       178    -10  {'niilc': 136, 'jemhopqa': 42}
slot   5    3   a+b       169    -19  {'niilc': 132, 'jemhopqa': 37}
slot   7    1   a         192     +4  {'niilc': 145, 'jemhopqa': 47}
slot   7    2   a         188     +0  {'niilc': 142, 'jemhopqa': 46}
slot   7    3   a         179     -9  {

In [ ]:
# 【S3-4 二段検索の取りこぼしの診断】S3-3 と同じセッションで実行する（質問・正解は表示しない）
import collections, statistics
def rank_of_hit(qid, ctx):
    g = [_norm(x) for x in golds(qid) if _norm(x)]
    for i, c in enumerate(ctx, 1):
        if any(x in _norm(c["title"] + c["text"]) for x in g):
            return i
    return None

cat = collections.Counter()
rank_b, rank_a = collections.Counter(), collections.Counter()
gap_b, keylen_b = [], collections.Counter()
for r in NY:
    q, g = r["q_id"], ds(r["q_id"])
    if rank_of_hit(q, r["s1"][:5]):
        cat[(g, "1 1段目の5件にある")] += 1
        continue
    in10 = rank_of_hit(q, r["s1"]) is not None
    ra, rb = rank_of_hit(q, r["s2a"]), rank_of_hit(q, r["s2b"])
    if in10 and (ra or rb):
        cat[(g, "2 1段目の6〜10位にも、2段目にもある")] += 1
    elif in10:
        cat[(g, "3 1段目の6〜10位にだけある")] += 1
    elif ra or rb:
        cat[(g, "4 2段目にだけある")] += 1
    else:
        cat[(g, "5 どこにもない")] += 1
    if not in10:                 # 2段目でしか取れない問題の、2段目の中での順位
        if ra: rank_a[min(ra, 4)] += 1
        if rb:
            rank_b[min(rb, 4)] += 1
            c = r["s2b"][rb - 1]
            gap_b.append(c["score"] - r["s1"][4]["score"])
            keylen_b[min(c.get("keylen", 0), 8)] += 1

print("1段目の5件に正解がない問題の内訳（非yn 275問）:")
for name in sorted({k[1] for k in cat}):
    print("  %-26s niilc %3d / jemhopqa %3d" % (name, cat[("niilc", name)], cat[("jemhopqa", name)]))
print("\n「2段目にだけある」問題の、2段目の中での順位（4 は4位以下）")
print("  手がかり a:", dict(sorted(rank_a.items())), "/ 手がかり b:", dict(sorted(rank_b.items())))
if gap_b:
    print("  手がかり b の正解の資料のスコア − 1段目5位のスコア: 中央値 %.3f / 正の数 %d / %d"
          % (statistics.median(gap_b), sum(1 for x in gap_b if x > 0), len(gap_b)))
    print("  そのときの記事名の文字数（8 は8文字以上）:", dict(sorted(keylen_b.items())))
up = sum(1 for r in NY if rank_of_hit(r["q_id"], r["s1"] + r["s2a"] + r["s2b"]))
print("\n上限（1段目10件＋2段目 a・b 各10件のどこかにある）: %d / 275（1段目5件の基準 188）" % up)

1段目の5件に正解がない問題の内訳（非yn 275問）:
  1 1段目の5件にある                niilc 142 / jemhopqa  46
  2 1段目の6〜10位にも、2段目にもある      niilc   1 / jemhopqa   2
  3 1段目の6〜10位にだけある           niilc   5 / jemhopqa   4
  4 2段目にだけある                 niilc   3 / jemhopqa   1
  5 どこにもない                   niilc  49 / jemhopqa  22

「2段目にだけある」問題の、2段目の中での順位（4 は4位以下）
  手がかり a: {} / 手がかり b: {1: 2, 4: 2}
  手がかり b の正解の資料のスコア − 1段目5位のスコア: 中央値 -3.441 / 正の数 1 / 4
  そのときの記事名の文字数（8 は8文字以上）: {3: 1, 4: 1, 5: 1, 6: 1}

上限（1段目10件＋2段目 a・b 各10件のどこかにある）: 204 / 275（1段目5件の基準 188）


In [ ]:
# 【S3-5】はい/いいえ問題で、質問に出てくる記事が資料にそろっているかを数える（CPU・表示は件数だけ）
from google.colab import drive
drive.mount("/content/drive")
import json, re, unicodedata, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
TITLE2IDS = json.load(open(EVAL_ROOT + "/title_index/title_to_chunks.json", encoding="utf-8"))
ANS = json.load(open(EVAL_ROOT + "/dev_data/dev_answers.json", encoding="utf-8"))
rows = [json.loads(l) for l in open(EVAL_ROOT + "/dev_contexts_2stage_v2.jsonl", encoding="utf-8") if l.strip()]
assert len(TITLE2IDS) == 49043 and len(rows) == 320

# --- S3-1 と同じ記事名の照合（括弧を外した名前も使う）---
nf = lambda s: unicodedata.normalize("NFKC", s or "")
_paren = re.compile(r"\s*[\(（][^\(\)（）]*[\)）]\s*$")
KEY2TITLES = collections.defaultdict(list)
for t in TITLE2IDS:
    KEY2TITLES[nf(t)].append(t)
for t in TITLE2IDS:
    base = nf(_paren.sub("", t))
    if base and base != nf(t) and not any(nf(x) == base for x in KEY2TITLES.get(base, [])):
        KEY2TITLES[base].append(t)
KEY_MAXLEN = min(40, max(len(k) for k in KEY2TITLES))
def find_keys(text, minlen):
    s = nf(text); spans = []
    for i in range(len(s)):
        for L in range(min(KEY_MAXLEN, len(s) - i), minlen - 1, -1):
            if s[i:i + L] in KEY2TITLES:
                spans.append((i, i + L, s[i:i + L])); break
    cnt, end = collections.Counter(), -1
    for a, b, k in spans:
        if b <= end: continue
        cnt[k] += 1; end = max(end, b)
    return cnt

_norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())
YESNO = {"はい", "いいえ", "yes", "no"}
def rec(q): return ANS[q] if isinstance(ANS[q], dict) else {"answers": ANS[q]}
def is_yn(q):
    a = rec(q)["answers"]; a = [a] if isinstance(a, str) else a
    return bool(a) and all(_norm(x) in YESNO for x in a)

stat = collections.Counter()
for r in rows:
    q = r["q_id"]
    if not is_yn(q):
        continue
    src = str(rec(q).get("source", "?"))
    stat[(src, "はい/いいえ")] += 1
    keys = [k for k in sorted(find_keys(r["question"], 2), key=lambda k: (-len(k), k)) if len(k) >= 2][:3]
    need = [KEY2TITLES[k][0] for k in keys]
    if len(need) < 2:
        stat[(src, "質問の記事名が2つ未満")] += 1
        continue
    stat[(src, "質問の記事名が2つ以上")] += 1
    t5 = {c["title"] for c in r["s1"][:5]}
    t10 = {c["title"] for c in r["s1"]}
    t5a = t5 | {c["title"] for c in r["s2a"][:2]}
    stat[(src, "  1段目5件に全部そろう")] += all(t in t5 for t in need)
    stat[(src, "  1段目5件に1つも無い記事がある")] += any(t not in t5 for t in need)
    stat[(src, "  1段目10件なら全部そろう")] += all(t in t10 for t in need)
    stat[(src, "  1段目5件＋手がかりaの上位2件で全部そろう")] += all(t in t5a for t in need)

for src in sorted({k[0] for k in stat}):
    print("■", src)
    for name in ["はい/いいえ", "質問の記事名が2つ未満", "質問の記事名が2つ以上",
                 "  1段目5件に全部そろう", "  1段目5件に1つも無い記事がある",
                 "  1段目10件なら全部そろう", "  1段目5件＋手がかりaの上位2件で全部そろう"]:
        print("  %-30s %3d" % (name, stat[(src, name)]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
■ jemhopqa
  はい/いいえ                          45
  質問の記事名が2つ未満                     20
  質問の記事名が2つ以上                     25
    1段目5件に全部そろう                   11
    1段目5件に1つも無い記事がある              14
    1段目10件なら全部そろう                 15
    1段目5件＋手がかりaの上位2件で全部そろう        12


In [ ]:
# # 【S3-6】比較問題の診断（S3-5 と同じセッションで実行。表示は件数だけ）
# import glob, os, json, collections
# RUN_NAME = "C_V22_s2_ctx5_320"   # 比べる評価の名前
# PRED = "/content/drive/MyDrive/LLM講座最終コンペ_work/run_LoRA_ctx5_T4/generation_config.json"                         # ← 予測ファイルのパス（空なら候補を表示して止まる）

# if not PRED:
#     print("候補（ファイル名だけ）:")
#     for p in sorted(glob.glob(EVAL_ROOT + "/**/*", recursive=True)):
#         if RUN_NAME in p and os.path.isfile(p) and p.endswith((".json", ".jsonl")):
#             print("  ", p.replace(EVAL_ROOT + "/", ""), "(%.0f KB)" % (os.path.getsize(p) / 1e3))
#     raise SystemExit("PRED に予測ファイル（q_id と answer を含むもの）のパスを入れて、もう一度実行する")

# # --- 予測の読み込み（list / dict / jsonl のどれでも）---
# if PRED.endswith(".jsonl"):
#     raw = [json.loads(l) for l in open(PRED, encoding="utf-8") if l.strip()]
# else:
#     raw = json.load(open(PRED, encoding="utf-8"))
# if isinstance(raw, dict) and "q_id" not in raw:
#     raw = [{"q_id": k, **(v if isinstance(v, dict) else {"answer": v})} for k, v in raw.items()]
# print("予測の1件目のキー:", sorted(raw[0].keys()))
# pred = {str(o["q_id"]): str(o.get("answer", o.get("pred", o.get("prediction", "")))) for o in raw}
# YES = {"はい", "yes"}

# def gold_yes(q):
#     a = rec(q)["answers"]; a = [a] if isinstance(a, str) else a
#     return _norm(a[0]) in YES

# stat = collections.Counter()
# for r in rows:
#     q = r["q_id"]
#     if not is_yn(q) or q not in pred:
#         continue
#     keys = sorted(find_keys(r["question"], 2), key=lambda k: (-len(k), k))[:3]
#     need = [KEY2TITLES[k][0] for k in keys]
#     if len(need) < 2:
#         grp = "A 記事名が2つ未満"
#     else:
#         t5 = {c["title"] for c in r["s1"][:5]}
#         miss = [t for t in need if t not in t5]
#         if not miss:
#             grp = "B 全部そろう"
#         else:
#             grp = "C 欠ける"
#             # 欠けている記事ごとに、保存済みの2段目の候補（a の10件）にその記事のチャンクがあるか
#             cand_titles = {c["title"] for c in r["s2a"]}
#             stat[("C 欠ける", "記事ごとに1件ずつ取れば全部埋まる")] += all(t in cand_titles for t in miss)
#             stat[("C 欠ける", "欠けている記事の数=1")] += len(miss) == 1
#     p_yes = _norm(pred[q]) in YES
#     ok = p_yes == gold_yes(q)
#     stat[(grp, "問題数")] += 1
#     stat[(grp, "正解")] += ok
#     stat[(grp, "「はい」と答えた")] += p_yes
#     stat[(grp, "正解が「はい」")] += gold_yes(q)

# for grp in ["A 記事名が2つ未満", "B 全部そろう", "C 欠ける"]:
#     n = stat[(grp, "問題数")]
#     print("■ %s：%d 問 / 正解 %d（%.0f%%）/「はい」と答えた %d / 正解が「はい」 %d"
#           % (grp, n, stat[(grp, "正解")], 100 * stat[(grp, "正解")] / max(1, n),
#              stat[(grp, "「はい」と答えた")], stat[(grp, "正解が「はい」")]))
# print("  C のうち、欠けている記事が1つだけ: %d / 記事ごとに1件ずつ取れば全部埋まる: %d"
#       % (stat[("C 欠ける", "欠けている記事の数=1")], stat[("C 欠ける", "記事ごとに1件ずつ取れば全部埋まる")]))


# # 【S3-6 確認】予測と検索結果の q_id が合っているかを見る（表示は開発用の q_id と件数だけ）
# PRED = EVAL_ROOT + "/runs/C_V22_s2_ctx5_320_20261115_0547/submission.json"
# raw = json.load(open(PRED, encoding="utf-8"))
# print("submission の型:", type(raw).__name__, "/ 件数:", len(raw))
# pid = [str(o["q_id"]) for o in raw]
# rid = [str(r["q_id"]) for r in rows]
# print("予測の q_id の例:", pid[:3], "/ 型:", type(raw[0]["q_id"]).__name__)
# print("検索結果の q_id の例:", rid[:3])
# print("一致する q_id:", len(set(pid) & set(rid)), "/ 320")
# print("はい/いいえ問題の数（rows で数える）:", sum(1 for r in rows if is_yn(r["q_id"])))

予測の1件目のキー: ['dtype', 'enable_thinking', 'enforce_eager', 'gpu_memory_utilization', 'max_model_len', 'max_new_tokens', 'min_p', 'model', 'presence_penalty', 'q_id', 'repetition_penalty', 'revision', 'temperature', 'text_only', 'top_k', 'top_p', 'use_structured_outputs']
■ A 記事名が2つ未満：0 問 / 正解 0（0%）/「はい」と答えた 0 / 正解が「はい」 0
■ B 全部そろう：0 問 / 正解 0（0%）/「はい」と答えた 0 / 正解が「はい」 0
■ C 欠ける：0 問 / 正解 0（0%）/「はい」と答えた 0 / 正解が「はい」 0
  C のうち、欠けている記事が1つだけ: 0 / 記事ごとに1件ずつ取れば全部埋まる: 0
submission の型: list / 件数: 320
予測の q_id の例: ['dev0001', 'dev0002', 'dev0003'] / 型: str
検索結果の q_id の例: ['dev0001', 'dev0002', 'dev0003']
一致する q_id: 320 / 320
はい/いいえ問題の数（rows で数える）: 45


In [ ]:
# 【S3-6 改】比較問題の診断（S3-5 と同じセッションで実行。表示は件数だけ）
import json, collections
PRED = EVAL_ROOT + "/runs/C_V22_s2_ctx5_320_20261115_0547/submission.json"
raw = json.load(open(PRED, encoding="utf-8"))
assert isinstance(raw, list) and "answer" in raw[0] and "chunk" in raw[0], "submission.json ではない: %s" % sorted(raw[0].keys())
pred = {str(o["q_id"]): str(o["answer"]) for o in raw}
YES = {"はい", "yes"}
def gold_yes(q):
    a = rec(q)["answers"]; a = [a] if isinstance(a, str) else a
    return _norm(a[0]) in YES

n_rows = n_yn = n_in = 0
stat = collections.Counter()
for r in rows:
    q = r["q_id"]; n_rows += 1
    if not is_yn(q):
        continue
    n_yn += 1
    if q not in pred:
        continue
    n_in += 1
    keys = sorted(find_keys(r["question"], 2), key=lambda k: (-len(k), k))[:3]
    need = [KEY2TITLES[k][0] for k in keys]
    if len(need) < 2:
        grp = "A 記事名が2つ未満"
    else:
        t5 = {c["title"] for c in r["s1"][:5]}
        miss = [t for t in need if t not in t5]
        if not miss:
            grp = "B 全部そろう"
        else:
            grp = "C 欠ける"
            cand_titles = {c["title"] for c in r["s2a"]}
            stat[("C", "1件ずつ取れば埋まる")] += all(t in cand_titles for t in miss)
            stat[("C", "欠けは1記事")] += len(miss) == 1
    p_yes = _norm(pred[q]) in YES
    stat[(grp, "問題数")] += 1
    stat[(grp, "正解")] += (p_yes == gold_yes(q))
    stat[(grp, "「はい」と答えた")] += p_yes
    stat[(grp, "正解が「はい」")] += gold_yes(q)

print("絞り込み: rows %d → はい/いいえ %d → 予測あり %d" % (n_rows, n_yn, n_in))
for grp in ["A 記事名が2つ未満", "B 全部そろう", "C 欠ける"]:
    n = stat[(grp, "問題数")]
    print("■ %s：%d 問 / 正解 %d（%.0f%%）/「はい」と答えた %d / 正解が「はい」 %d"
          % (grp, n, stat[(grp, "正解")], 100 * stat[(grp, "正解")] / max(1, n),
             stat[(grp, "「はい」と答えた")], stat[(grp, "正解が「はい」")]))
print("  C のうち、欠けは1記事: %d / 記事ごとに1件ずつ取れば埋まる: %d" % (stat[("C", "欠けは1記事")], stat[("C", "1件ずつ取れば埋まる")]))

絞り込み: rows 320 → はい/いいえ 45 → 予測あり 45
■ A 記事名が2つ未満：20 問 / 正解 18（90%）/「はい」と答えた 5 / 正解が「はい」 7
■ B 全部そろう：11 問 / 正解 6（55%）/「はい」と答えた 5 / 正解が「はい」 6
■ C 欠ける：14 問 / 正解 11（79%）/「はい」と答えた 12 / 正解が「はい」 9
  C のうち、欠けは1記事: 12 / 記事ごとに1件ずつ取れば埋まる: 5


## Step 6: 検索結果を共通形式にまとめる

検索設定を `retrieval_config.json` に保存してから、実行対象の質問（`TARGET_QUESTIONS`）について `retrieve()` を実行する。`retrieve()` が検索・統合・リランク・本文取得まで行い、最終順位順の資料を返すので、ここでは質問 ID と質問文を付けて次の `generation_inputs`（dict のリスト）にまとめる。`generation_inputs` 自体はファイルに保存しない。

```python
generation_inputs = [
    {
        "q_id": "q001",
        "question": "質問文",
        "contexts": [
            {"chunk_id": 456, "title": "記事タイトル", "text": "本文"}
        ],
    }
]
```

`contexts` は、回答生成モデルへ提示する順に並ぶ。リランカーの有無にかかわらず同じ形式である。
独自の検索処理を使う場合も、この形式のデータを用意すれば Step 7 以降を実行できる。形式は続くセル（`[step6.2 validate]`）で検証する。
質問数・資料数はこのインターフェースでは固定しない。資料が0件の場合も扱える。

In [16]:

# [step6.1 retrieval]

retrieval_config = {
    "bm25_index": BM25_INDEX if "bm25" in RETRIEVERS else None,
    "dense_indexes": {name: {"index": EMB_INDICES[name], "embed": DENSE_INDEXES[name]["embed"],
                             "index_embed": DENSE_INDEXES[name]["index_embed"]} for name in DENSE_NAMES},
    "reranker": RERANKER if USE_RERANKER else None,
    "params": {"retrievers": RETRIEVERS, "fusion": FUSION if len(RETRIEVERS) > 1 else None, "bm25_k1": BM25_K1, "bm25_b": BM25_B,
               "retriever_top_k": {name: RETRIEVER_TOP_K[name] for name in RETRIEVERS}, "rrf_k": RRF_K, "fused_top_k": FUSED_TOP_K,
               "rerank_candidates": RERANK_CANDIDATES, "context_top_k": CONTEXT_TOP_K},
}
os.makedirs(RUN_DIR, exist_ok=True)
write_json(os.path.join(RUN_DIR, "retrieval_config.json"), dict(retrieval_config, env=ENV))
# 検索側の最終出力。このリスト以降は検索方式・リランカーの有無に依存しない。
generation_inputs = [
    {"q_id": q["q_id"], "question": q["question"], "contexts": retrieve(q["question"])}
    for q in TARGET_QUESTIONS
]
print("検索完了:", len(generation_inputs))


検索完了: 320


In [17]:
# [V2 E0 基準再現] step6.1 の1段目の結果が、既存 dev_contexts.jsonl と完全一致するか確かめる
# 基準ファイルは読むだけ。結果は val_2stage_out/E0_report_<RUN_ID>.json に保存（上書きしない）
import json, os, time, hashlib
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
BASE_PATH = EVAL_ROOT + "/dev_contexts.jsonl"
OUT = EVAL_ROOT + "/val_2stage_out"

# --- 取り違え防止 ---
assert RUN_ID == "val_2stage_E0", "RUN_ID が val_2stage_E0 ではない（2-1 の書き換えを確認する）: %s" % RUN_ID
assert "LLM講座最終コンペ_eval" in str(WORK_DIR), "WORK_DIR が評価用フォルダではない: %s" % WORK_DIR
assert "LLM講座最終コンペ_work" not in str(WORK_DIR), "WORK_DIR が本番用フォルダ: %s" % WORK_DIR
assert CONTEXT_TOP_K == 10, "CONTEXT_TOP_K が 10 ではない: %s" % CONTEXT_TOP_K
assert QUESTION_LIMIT is None, "QUESTION_LIMIT が None ではない"
if "generation_inputs" not in globals():
    raise SystemExit("generation_inputs がまだない。[step6.2 validate] を実行してから、このセルをもう一度実行する")

rows = list(generation_inputs)
assert len(rows) == 320, "件数が 320 ではない: %d" % len(rows)
assert all(str(r["q_id"]).startswith("dev") for r in rows), "q_id が dev で始まらない（開発用セットではない）"

# --- 比べる（chunk_id だけ）---
base = {}
with open(BASE_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            o = json.loads(line)
            base[o["q_id"]] = [int(c["chunk_id"]) for c in o["contexts"]]
new = {r["q_id"]: [int(c["chunk_id"]) for c in r["contexts"]] for r in rows}

qid_diff = sorted(set(base) ^ set(new))
common = [q for q in base if q in new]
diff10 = [q for q in common if new[q][:10] != base[q][:10]]
diff5 = [q for q in common if new[q][:5] != base[q][:5]]
set10 = [q for q in common if set(new[q][:10]) != set(base[q][:10])]
n_ctx = sorted(set(len(v) for v in new.values()))

print("基準ファイル        :", BASE_PATH, "/", len(base), "問")
print("今回（step6.1）     :", len(new), "問 / 1問あたりの資料:", n_ctx)
print("q_id の不一致       :", len(qid_diff))
print("上位10件 順序の違い :", len(diff10))
print("上位10件 集合の違い :", len(set10))
print("上位5件  順序の違い :", len(diff5))
if diff10:
    print("違う問題の q_id（先頭10）:", diff10[:10])

# --- レポート保存（上書きしない）---
os.makedirs(OUT, exist_ok=True)
path = OUT + "/E0_report_" + RUN_ID + ".json"
if os.path.exists(path):
    path = path.replace(".json", "_" + time.strftime("%Y%m%d_%H%M%S") + ".json")
report = {
    "run_id": RUN_ID, "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "n_base": len(base), "n_new": len(new), "qid_mismatch": len(qid_diff),
    "top10_order_diff": len(diff10), "top10_set_diff": len(set10), "top5_order_diff": len(diff5),
    "diff_qids_head": diff10[:10],
    "base_sha256": hashlib.sha256(open(BASE_PATH, "rb").read()).hexdigest(),
}
with open(path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print("レポート保存:", path)

if qid_diff or diff10:
    raise SystemExit("E0 不合格：1段目が既存と一致しない。新しいロジックに進まない。この出力を貼る")
print("E0 合格: 320/320 一致（上位10件の順序まで同じ）")

基準ファイル        : /content/drive/MyDrive/LLM講座最終コンペ_eval/dev_contexts.jsonl / 320 問
今回（step6.1）     : 320 問 / 1問あたりの資料: [10]
q_id の不一致       : 0
上位10件 順序の違い : 0
上位10件 集合の違い : 0
上位5件  順序の違い : 0
レポート保存: /content/drive/MyDrive/LLM講座最終コンペ_eval/val_2stage_out/E0_report_val_2stage_E0.json
E0 合格: 320/320 一致（上位10件の順序まで同じ）


In [ ]:
# [step6.2 validate]

# 検索側と回答生成側の共通インターフェース。件数や検索方法は固定しない。
def validate_generation_inputs(items):
    if not isinstance(items, list):
        raise ValueError("generation_inputs は dict のリストにすること")
    seen_questions = set()
    for i, item in enumerate(items):
        label = "generation_inputs[%d]" % i
        if not isinstance(item, dict) or not {"q_id", "question", "contexts"} <= item.keys():
            raise ValueError(label + " に q_id・question・contexts が必要である")
        qid = item["q_id"]
        if not isinstance(qid, str) or not qid or qid in seen_questions:
            raise ValueError(label + ".q_id は重複のない空でない文字列にすること")
        seen_questions.add(qid)
        if not isinstance(item["question"], str):
            raise ValueError(label + ".question は文字列にすること")
        if not isinstance(item["contexts"], list):
            raise ValueError(label + ".contexts は dict のリストにすること")
        seen_chunks = set()
        for j, context in enumerate(item["contexts"]):
            path = label + ".contexts[%d]" % j
            if not isinstance(context, dict) or not {"chunk_id", "title", "text"} <= context.keys():
                raise ValueError(path + " に chunk_id・title・text が必要である")
            cid = context["chunk_id"]
            if type(cid) is not int or cid < 0 or cid in seen_chunks:
                raise ValueError(path + ".chunk_id は質問内で重複のない非負整数にすること")
            seen_chunks.add(cid)
            if not isinstance(context["title"], str) or not isinstance(context["text"], str):
                raise ValueError(path + " の title・text は文字列にすること")
    return items


validate_generation_inputs(generation_inputs)

In [ ]:
raise SystemExit("このノートブック（dev_ctx_2stage）では、このセルは使わない")


# 【開発用検索結果の保存セル】R01 のコピーの [step6.2 validate] の直後に1回だけ置く（生成が走らないよう、最後に止める）
import json, os, time, collections
EVAL_ROOT = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUT = EVAL_ROOT + "/dev_contexts.jsonl"

# --- 取り違え防止（本番の質問で保存しない）---
assert "LLM講座最終コンペ_eval" in str(WORK_DIR), "WORK_DIR が評価用フォルダではない: " + str(WORK_DIR)
assert "LLM講座最終コンペ_work" not in str(WORK_DIR), "WORK_DIR が本番用フォルダ: " + str(WORK_DIR)
assert CONTEXT_TOP_K == 10, "CONTEXT_TOP_K は 10 にする（評価ノートブックが 5・7・10 件に切り出す）: %s" % CONTEXT_TOP_K

gi = generation_inputs
rows = list(gi.values()) if isinstance(gi, dict) else list(gi)
r0 = rows[0]
print("件数:", len(rows), "/ 1件目のキー:", sorted(r0.keys()) if isinstance(r0, dict) else type(r0))

def pick(r, *names):
    for n in names:
        if n in r:
            return r[n]
    raise SystemExit("キー %s が見つからない。上の「1件目のキー」の行だけを貼る（中身は貼らない）" % (names,))

out = []
for r in rows:
    ctx = pick(r, "contexts", "chunks", "retrieved_chunks")
    out.append({"q_id": pick(r, "q_id", "id"), "question": pick(r, "question", "query"),
                "contexts": [{"chunk_id": c["chunk_id"], "title": c["title"], "text": c["text"]} for c in ctx]})

# --- 保存（すでにあるファイルは別名に退避。上書きしない）---
if os.path.exists(OUT):
    bak = OUT + ".bak_" + time.strftime("%Y%m%d_%H%M%S")
    os.rename(OUT, bak)
    print("すでにあった dev_contexts.jsonl を退避:", bak)
with open(OUT, "w", encoding="utf-8") as f:
    for o in out:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")
print("保存:", OUT, "/", len(out), "問 / 1問あたりの資料:", dict(collections.Counter(len(o["contexts"]) for o in out)))
raise SystemExit("開発用セットの検索結果を保存した。ここで止める（生成は動かさない）")

## Step 7: 推論設定と回答生成モデルの起動

次のセルで、回答生成のプロンプトとハイパーパラメータをまとめて指定する。`answer_prompt()` の文章や資料の並べ方、`GENERATOR` の生成設定、`SEED`、`GEN_BATCH_SIZE` を変更できる。モデル ID と revision は利用ルールにより固定である。

検索方法を変える場合は検索側を編集し、最終出力を `generation_inputs` の共通形式に合わせる。

In [45]:
# [step7.1 generation-config] 編集可: プロンプト・生成ハイパーパラメータ
# 回答生成モデルはコンペのルールで Qwen/Qwen3.5-0.8B に固定（model と revision は変更不可）
'''
GENERATOR = {
    "model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "max_new_tokens": 1024,
    # サンプリング: モデルカードの推奨値（non-thinking モード・テキスト）
    "temperature": 1.0,
    "top_p": 1.0,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 2.0,
    "repetition_penalty": 1.0,
    "max_model_len": 8192,        # 入力+出力の最大トークン数
    "gpu_memory_utilization": 0.5,
    "dtype": "float16",           # T4 は bfloat16 非対応
    "enforce_eager": True,        # True: CUDA グラフを使わない（Qwen3.5 は T4 で False にすると初期化に失敗する）
    "enable_thinking": False,     # True: thinkingを有効にする
    "use_structured_outputs": True, # False: 出力JSONスキーマの制約を使わない
    "text_only": True,            # True: 画像・動画の入力を無効にする（画像も扱えるモデルの起動が速くなる）
}
'''

GENERATOR = {
    "model": "YAMAHARZ250/qwen3.5-0.8b-rag-lora-final",
    "revision": "9a4d0723a493aa3c2b9570b065744ea16f4a9237",
    "max_new_tokens": 64,
    # サンプリング: 温度0（毎回いちばん確率の高い語を選ぶ）・繰り返しペナルティなし
    "temperature": 0.0,
    "top_p": 1.0,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 0.0,
    "repetition_penalty": 1.0,
    "max_model_len": 8192,        # 入力+出力の最大トークン数
    "gpu_memory_utilization": 0.5,
    "dtype": "float16",           # T4 は bfloat16 非対応
    "enforce_eager": True,        # True: CUDA グラフを使わない（Qwen3.5 は T4 で False にすると初期化に失敗する）
    "enable_thinking": False,     # True: thinkingを有効にする
    "use_structured_outputs": True, # False: 出力JSONスキーマの制約を使わない
    "text_only": True,            # True: 画像・動画の入力を無効にする（画像も扱えるモデルの起動が速くなる）
}



SEED = 42
GEN_BATCH_SIZE = 10               # 何問ずつまとめて生成するか（1バッチごとに提出ファイルへ途中結果を保存）



'''
def answer_prompt(question, contexts):
    context_text = "\n\n".join("[chunk_id=%d] %s\n%s" % (c["chunk_id"], c["title"], c["text"]) for c in contexts)
    return (
        "次の資料だけを根拠に質問に答えてください。資料から答えを確定できない場合は"
        "answer を「資料からは特定できません」、chunk_id を null としてください。\n"
        "回答の説明は含めず、JSONだけを返してください。キーは answer と chunk_id の2つです。"
        "answer は短い回答文字列、chunk_id は回答の根拠として最も直接的な資料の chunk_id(整数)を1つだけにしてください。\n"
        '形式: {"answer":"回答文字列","chunk_id":整数}\n\n'
        "資料:\n" + context_text + "\n\n質問: " + question
    )

'''


def answer_prompt(question, contexts):
    context_text = "\n\n".join("[chunk_id=%d] %s\n%s" % (c["chunk_id"], c["title"], c["text"]) for c in contexts)
    return (
        "質問: " + question + "\n\n"
        "次の資料を根拠に、上の質問に短く答えてください。\n"
        "答え方のきまり:\n"
        "- answer には、質問の答えになる語句だけを書く（人名・地名・作品名・数値・日付など）。\n"
        "- 文にしない。「〜です」「〜である」を付けない。説明・補足・括弧書き・読み仮名・英語表記を付けない。\n"
        "- 「どちら」「どっち」と聞かれたら、選んだほうの名前だけを書く。\n"
        "- 「はい」か「いいえ」で答える質問には、「はい」か「いいえ」だけを書く。\n"
        "- 資料のどこにも答えが書かれていない場合だけ、answer を「資料からは特定できません」、chunk_id を null にする。\n"
        "chunk_id には、答えが書かれている資料の chunk_id(整数)を1つだけ書く。\n"
        '形式: {"answer":"答えの語句","chunk_id":整数}\n\n'
        "資料:\n" + context_text + "\n\n質問（再掲）: " + question
    )


print("GENERATOR:", GENERATOR["model"], "/ batch:", GEN_BATCH_SIZE)



GENERATOR: YAMAHARZ250/qwen3.5-0.8b-rag-lora-final / batch: 10


In [16]:
# 【提出前の照合セル】[step7.1 generation-config] の直後に1回実行する
# ① 提出用のプロンプトが、学習時と同じか（md5）② GENERATOR の model・revision が、本番モデルの値か
# 一致しなければ、ここで止まる（「すべてを実行」でも、生成に進まない）
import hashlib
TRAIN_MD5 = "1e990717ef9ebb51535b233deec1503e"                      # 学習セルが表示した値（2026-11-08 の本番学習）
WANT_MODEL = "YAMAHARZ250/qwen3.5-0.8b-rag-lora-final"
WANT_REV = "9a4d0723a493aa3c2b9570b065744ea16f4a9237"
now = hashlib.md5(answer_prompt("Q", [{"chunk_id": 1, "title": "T", "text": "X"}]).encode("utf-8")).hexdigest()
print("提出用の md5:", now, "/ 学習時の md5:", TRAIN_MD5, "→", "一致" if now == TRAIN_MD5 else "【不一致】")
bad = []
if now != TRAIN_MD5:
    bad.append("プロンプトの md5 が学習時と違う")
g = globals().get("GENERATOR")
if isinstance(g, dict) and "model" in g:
    print("GENERATOR: model =", g.get("model"), "/ revision =", g.get("revision"))
    if g.get("model") != WANT_MODEL or g.get("revision") != WANT_REV:
        bad.append("GENERATOR の model・revision が本番モデルの値と違う")
else:
    print("[注意] GENERATOR が辞書として見つからない。[step7.1] の有効な方（三連引用符の外）の model・revision を、目で確認する")
if bad:
    raise SystemExit("提出しない: " + " / ".join(bad) + "。[step7.1] を直して、そこから実行し直す")
print("OK: プロンプトも GENERATOR も、本番モデルと学習時に一致している")

NameError: name 'answer_prompt' is not defined

## ここから下は編集不要

ここから下のセルは、そのまま実行する。

### モデル起動

生成設定を `generation_config.json` に保存し、リランカーと BM25 を解放してから回答生成モデルを起動する。プロンプトとハイパーパラメータは、上の推論設定セルの値を使う。

In [ ]:
# [step7.2 generator] 編集不要

# [step7.1 generation-config] の生成設定を、検索設定とは別に保存する。
write_json(os.path.join(RUN_DIR, "generation_config.json"), {
    "generator": GENERATOR, "seed": SEED, "batch_size": GEN_BATCH_SIZE,
})


def answer_schema(contexts):
    # 出力を {"answer": 文字列, "chunk_id": 渡した資料の chunk_id か null} の JSON に制約する
    return {
        "type": "object",
        "properties": {
            "answer": {"type": "string"},
            "chunk_id": {"enum": [c["chunk_id"] for c in contexts] + [None]},
        },
        "required": ["answer", "chunk_id"],
        "additionalProperties": False,
    }


def parse_answer(raw):
    # 生成モデルの出力から (answer, chunk_id) を取り出す。読み取れないものは None
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError:
        payload = None
    if isinstance(payload, dict):
        answer, cid = payload.get("answer"), payload.get("chunk_id")
    else:
        # max_new_tokens に達して JSON が途中で切れた場合: answer を書けたところまで取り出す
        m = re.match(r'\s*\{\s*"answer"\s*:\s*"((?:[^"\\]|\\.)*)', raw)
        answer, cid = None, None
        if m:
            try:
                answer = json.loads('"' + m.group(1) + '"')
            except json.JSONDecodeError:
                answer = m.group(1)
    if not isinstance(answer, str):
        answer = None
    if isinstance(cid, str) and cid.strip().isdigit():   # "123" のような数字だけの文字列は整数に直す
        cid = int(cid.strip())
    if not isinstance(cid, int) or isinstance(cid, bool) or cid < 0:
        cid = None
    return answer, cid


class VLLMGenerator:
    def __init__(self, spec):
        self.spec = spec
        import contextlib, importlib
        # Colab 向けの回避策: Colab の出力は fileno() に対応しないため、vLLM の標準出力の抑制を無効にする
        try:
            m = importlib.import_module("vllm.utils.system_utils")
            if hasattr(m, "suppress_stdout"):
                m.suppress_stdout = contextlib.nullcontext
        except ImportError:
            pass
        from vllm import LLM, SamplingParams
        from vllm.sampling_params import StructuredOutputsParams
        extra = {"limit_mm_per_prompt": {"image": 0, "video": 0}} if spec["text_only"] else {}
        self.llm = LLM(model=spec["model"], revision=spec["revision"], dtype=spec["dtype"],
                       max_model_len=spec["max_model_len"], gpu_memory_utilization=spec["gpu_memory_utilization"],
                       enforce_eager=spec["enforce_eager"], seed=SEED, **extra)
        self.sampling = dict(max_tokens=spec["max_new_tokens"], seed=SEED, **{k: spec[k] for k in (
            "temperature", "top_p", "top_k", "min_p", "presence_penalty", "repetition_penalty")})
        self.SamplingParams, self.StructuredOutputsParams = SamplingParams, StructuredOutputsParams

    def generate(self, prompts, schemas):
        # schemas: プロンプトごとの JSON スキーマ
        params = [self.SamplingParams(
            **self.sampling, structured_outputs=self.StructuredOutputsParams(json=s) if self.spec["use_structured_outputs"] else None)
            for s in schemas]
        conversations = [[{"role": "user", "content": p}] for p in prompts]
        outs = self.llm.chat(conversations, params, use_tqdm=True,
                             chat_template_kwargs={"enable_thinking": self.spec["enable_thinking"]})
        return [o.outputs[0].text for o in outs]


# 検索とリランクが終わったので、リランカーと BM25 を解放してから vLLM を起動する。
reranker = None
bm25 = None
free_gpu()
t0 = time.time()
generator = VLLMGenerator(GENERATOR)
print("[generator] %s (%.0f s)" % (GENERATOR["model"], time.time() - t0))


## Step 8: 共通入力から回答生成・提出ファイル保存

`generation_inputs` だけからプロンプトと出力 JSON スキーマを作り、回答を生成する。資料の追加取得や検索結果の再参照は行わない。
回答は提出形式で `submission.json` に保存する。実行のたびに最初から上書きし、採点は行わない。
`GEN_BATCH_SIZE` 問を生成するたびに保存するので、途中で中断しても生成済みの回答は残る。

In [ ]:
# [step8.1 generation] 編集不要
SUBMISSION_PATH = os.path.join(RUN_DIR, "submission.json")
# 1バッチ生成するごとに、それまでの回答で提出ファイルを上書きする。中断しても完了分が残る。
submission = []
write_json(SUBMISSION_PATH, submission)   # 前回の実行結果が残らないように最初に空にする
for start in range(0, len(generation_inputs), GEN_BATCH_SIZE):
    batch = generation_inputs[start:start + GEN_BATCH_SIZE]
    prompts = [answer_prompt(item["question"], item["contexts"]) for item in batch]
    outputs = generator.generate(prompts, [answer_schema(item["contexts"]) for item in batch])
    if len(outputs) != len(batch):
        raise ValueError("生成件数が質問数と一致しない")
    for item, raw in zip(batch, outputs):
        answer, cid = parse_answer(raw)
        submission.append({"q_id": item["q_id"], "answer": answer, "chunk": cid})
    write_json(SUBMISSION_PATH, submission)
    print("[generation] %d / %d 問を保存した" % (len(submission), len(generation_inputs)))
print("提出ファイル:", SUBMISSION_PATH)


In [ ]:
# 【件数確認セル】submission.json の中身（質問・回答）は表示しない。件数と文字数の集計だけを出す
import glob, json, os
p = globals().get("SUBMISSION_PATH")
if not p or not os.path.exists(str(p)):
    cands = sorted(set(glob.glob("/content/**/submission*.json", recursive=True) + glob.glob("/content/drive/MyDrive/**/submission*.json", recursive=True)), key=os.path.getmtime)
    print("SUBMISSION_PATH が使えないので、新しい順に探した（先頭が最新）:", cands[::-1][:3])
    p = cands[-1] if cands else None
if not p:
    raise SystemExit("submission.json が見つからない")
print("対象:", p)
s = json.load(open(p, encoding="utf-8"))
if not isinstance(s, list) or not s or not isinstance(s[0], dict):
    raise SystemExit("想定外の形式: %s" % type(s).__name__)
print("1件のキー:", sorted(s[0].keys()))
ans = [(x.get("answer") or "") for x in s]
lens = sorted(len(a) for a in ans)
print("件数:", len(s), "/ chunk=None:", sum(1 for x in s if x.get("chunk") is None), "/ answer が空:", sum(1 for a in ans if not a),
      "/ 20文字超:", sum(1 for n in lens if n > 20), "/ 最大文字数:", lens[-1], "/ 中央値:", lens[len(lens) // 2])

In [ ]:
import torch
print("GPU あり:", torch.cuda.is_available(), "/", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし")
print("リランカーの置き場所:", next(reranker.model.parameters()).device if hasattr(reranker, "model") else "（確認できない）")